# CRVSE Phase 2-11: Live-Compatible PhysFormer Training

## Research question

Can a PhysFormer model trained with live-compatible source-FPS tensors as the primary training distribution outperform shallow fine-tuning of the previous full-recording-style checkpoint?

## Why this notebook exists

`NB_P2_10` tested whether the existing multichannel PhysFormer checkpoint could adapt to live-compatible tensors through shallow fine-tuning.

The best `NB_P2_10` variant was balanced last-block/head fine-tuning, but the improvement was small and ECG-Fitness out-of-domain performance degraded.

This notebook tests a different hypothesis:

```text
The old checkpoint may not adapt enough because it was originally trained on a different preprocessing distribution.

## Imports and Setup 

In [1]:
import torch, json, math, os, random, sys, time, copy, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

LIVE_COMPATIBLE_DIR = Path("/kaggle/input/datasets/cezarytubacki/live-compatible-crvse/Live_Compatible_CRVSE")
ORIGINAL_CHECKPOINT_PATH = (LIVE_COMPATIBLE_DIR / "CRVSETransformer_Ensemble_physformer_multichannel_best.pt")
BASELINE_SUMMARY_PATH = (LIVE_COMPATIBLE_DIR / "live_finetune_frozen_baseline_summary.csv")
BASELINE_PREDICTIONS_PATH = (LIVE_COMPATIBLE_DIR / "live_finetune_frozen_baseline_predictions.csv")
MANIFEST_PATH = (LIVE_COMPATIBLE_DIR / "live_finetune_manifest.csv")

DRY_RUN = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")

if DEVICE.type != "cuda" and not DRY_RUN:
    raise RuntimeError(
        "CUDA is not available. Enable a Kaggle GPU accelerator before training, "
        "or set DRY_RUN=True for CPU-only artifact checks."
    )


def find_unique_input_file(filename: str) -> Path:
    """Find exactly one Kaggle input file by filename."""
    matches = sorted(KAGGLE_INPUT.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename!r} under /kaggle/input. "
            "Attach the saved NB_P2_10 Kaggle output as an input dataset."
        )

    if len(matches) > 1:
        print(f"Multiple matches found for {filename!r}:")
        for match in matches:
            print(f"  - {match}")
        raise RuntimeError(
            f"Expected exactly one match for {filename!r}. "
            "Use the printed paths to choose the correct NB10 output dataset."
        )

    return matches[0]


MATERIALIZED_TENSOR_PATHS = {
    "finetune_train": find_unique_input_file("finetune_train_sourcefps_tensors.npz"),
    "finetune_val": find_unique_input_file("finetune_val_sourcefps_tensors.npz"),
    "finetune_test": find_unique_input_file("finetune_test_sourcefps_tensors.npz"),
    "ood_eval": find_unique_input_file("ood_eval_sourcefps_tensors.npz"),
}

MATERIALIZED_METADATA_PATHS = {
    "finetune_train": find_unique_input_file("finetune_train_sourcefps_metadata.csv"),
    "finetune_val": find_unique_input_file("finetune_val_sourcefps_metadata.csv"),
    "finetune_test": find_unique_input_file("finetune_test_sourcefps_metadata.csv"),
    "ood_eval": find_unique_input_file("ood_eval_sourcefps_metadata.csv"),
}

try:
    NB10_CONCLUSION_PATH = find_unique_input_file("nb_p2_10_conclusion.json")
except FileNotFoundError:
    NB10_CONCLUSION_PATH = None
    print("NB10 conclusion JSON was not found. This is not fatal for setup.")


required_paths = {
    "manifest": MANIFEST_PATH,
    "baseline_summary": BASELINE_SUMMARY_PATH,
    "baseline_predictions": BASELINE_PREDICTIONS_PATH,
    "original_checkpoint": ORIGINAL_CHECKPOINT_PATH,
    **{f"{role}_tensor": path for role, path in MATERIALIZED_TENSOR_PATHS.items()},
    **{f"{role}_metadata": path for role, path in MATERIALIZED_METADATA_PATHS.items()},
}

path_report = pd.DataFrame(
    [
        {
            "artifact": name,
            "exists": path.exists(),
            "path": str(path),
        }
        for name, path in required_paths.items()
    ]
)

missing = path_report.loc[~path_report["exists"], "path"].tolist()

if missing:
    raise FileNotFoundError(
        "Missing required NB11 artifacts:\n" + "\n".join(missing)
    )


shape_rows = []

for role, tensor_path in MATERIALIZED_TENSOR_PATHS.items():
    loaded = np.load(tensor_path)
    metadata = pd.read_csv(MATERIALIZED_METADATA_PATHS[role])

    x = loaded["x"]
    y = loaded["y"]

    shape_rows.append(
        {
            "role": role,
            "x_shape": tuple(x.shape),
            "y_shape": tuple(y.shape),
            "metadata_rows": len(metadata),
            "x_dtype": str(x.dtype),
            "y_dtype": str(y.dtype),
            "x_finite": bool(np.isfinite(x).all()),
            "y_finite": bool(np.isfinite(y).all()),
            "target_hr_min": float(np.min(y)),
            "target_hr_max": float(np.max(y)),
        }
    )

shape_report = pd.DataFrame(shape_rows)

expected_rows = {
    "finetune_train": 12503,
    "finetune_val": 2839,
    "finetune_test": 2828,
    "ood_eval": 882,
}

for _, row in shape_report.iterrows():
    role = row["role"]
    expected = expected_rows[role]
    actual = int(row["y_shape"][0])

    if actual != expected:
        raise AssertionError(
            f"{role} row count mismatch: expected {expected}, found {actual}"
        )

    if tuple(row["x_shape"][1:]) != (3, 240):
        raise AssertionError(
            f"{role} tensor shape mismatch: expected tail (3, 240), found {row['x_shape']}"
        )

    if not row["x_finite"] or not row["y_finite"]:
        raise AssertionError(f"{role} contains non-finite tensor or label values.")


if NB10_CONCLUSION_PATH is not None:
    with NB10_CONCLUSION_PATH.open("r", encoding="utf-8") as file:
        nb10_conclusion = json.load(file)
else:
    nb10_conclusion = None


print("Artifact path report:")
display(path_report)
print("\nMaterialized tensor shape report:")
display(shape_report)
print("\nNB10 conclusion:")
print(json.dumps(nb10_conclusion, indent=2))
print("\nPASS: NB11 setup found the NB10 materialized tensors and required baseline artifacts.")

Python: 3.12.13
NumPy: 2.0.2
Pandas: 2.3.3
PyTorch: 2.10.0+cu128
Device: cuda
Artifact path report:


,artifact,exists,path
0,manifest,True,/kaggle/input/datasets/cezarytubacki/live-comp...
1,baseline_summary,True,/kaggle/input/datasets/cezarytubacki/live-comp...
2,baseline_predictions,True,/kaggle/input/datasets/cezarytubacki/live-comp...
3,original_checkpoint,True,/kaggle/input/datasets/cezarytubacki/live-comp...
4,finetune_train_tensor,True,/kaggle/input/notebooks/cezarytubacki/nb-p2-10...
5,finetune_val_tensor,True,/kaggle/input/notebooks/cezarytubacki/nb-p2-10...
6,finetune_test_tensor,True,/kaggle/input/notebooks/cezarytubacki/nb-p2-10...
7,ood_eval_tensor,True,/kaggle/input/notebooks/cezarytubacki/nb-p2-10...
8,finetune_train_metadata,True,/kaggle/input/notebooks/cezarytubacki/nb-p2-10...
9,finetune_val_metadata,True,/kaggle/input/notebooks/cezarytubacki/nb-p2-10...



Materialized tensor shape report:


,role,x_shape,y_shape,metadata_rows,x_dtype,y_dtype,x_finite,y_finite,target_hr_min,target_hr_max
0,finetune_train,"(12503, 3, 240)","(12503,)",12503,float32,float32,True,True,40.056313,127.000000
1,finetune_val,"(2839, 3, 240)","(2839,)",2839,float32,float32,True,True,40.296436,127.000000
2,finetune_test,"(2828, 3, 240)","(2828,)",2828,float32,float32,True,True,47.631721,129.071884
3,ood_eval,"(882, 3, 240)","(882,)",882,float32,float32,True,True,53.662189,167.968079



NB10 conclusion:
{
  "best_aggregate_experiment": "balanced_last_block_head",
  "best_val_delta_bpm": -0.0759,
  "best_test_delta_bpm": -0.0347,
  "best_ood_delta_bpm": 0.432,
  "adopt_checkpoint": false,
  "reason": "Fine-tuning produced only small aggregate improvements. Balanced last-block/head was the best variant, but the gain was below the practical threshold and ECG-Fitness OOD performance degraded. No fine-tuned checkpoint should replace the frozen model from this notebook.",
  "next_research_direction": "If improvement is still desired, the next notebook should test training or transfer learning on live-compatible tensors as the primary training distribution, rather than shallow adaptation of the old checkpoint."
}

PASS: NB11 setup found the NB10 materialized tensors and required baseline artifacts.


## Frozen Source-FPS Baseline Reproduction

### What is being tested

This section checks whether the materialized live-compatible source-FPS tensors reproduce the frozen CRVSE PhysFormer baseline.

### Why this matters

Before training a new model, the notebook needs a trustworthy evaluation path.

The cached tensors should represent the same live-compatible preprocessing distribution used in `NB_P2_10`. If the frozen checkpoint evaluated here does not match the saved baseline CSV, then later training results could be compared against the wrong reference.


In [2]:
EVAL_BATCH_SIZE = 256
ROLE_ORDER = ["finetune_train", "finetune_val", "finetune_test", "ood_eval"]


def assert_nb11_setup_is_available() -> None:
    """
    Check that the NB11 setup cell has been executed.

    This baseline cell depends on setup for imports and artifact paths.
    It intentionally does not repeat imports.
    """
    required_names = [
        "DEVICE",
        "ORIGINAL_CHECKPOINT_PATH",
        "BASELINE_PREDICTIONS_PATH",
        "MATERIALIZED_TENSOR_PATHS",
        "MATERIALIZED_METADATA_PATHS",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run the NB11 setup cell before this baseline cell. "
            f"Missing names: {missing}"
        )


class MaterializedTensorDataset(Dataset):
    """
    Dataset backed by materialized live-compatible source-FPS tensors.
    """

    def __init__(self, role: str, tensor_path: Path, metadata_path: Path) -> None:
        self.role = role
        self.tensor_path = Path(tensor_path)
        self.metadata_path = Path(metadata_path)

        loaded = np.load(self.tensor_path)
        self.x = loaded["x"].astype(np.float32, copy=False)
        self.y = loaded["y"].astype(np.float32, copy=False)
        self.metadata = pd.read_csv(self.metadata_path)

        if self.x.ndim != 3:
            raise ValueError(f"{role}: expected x with 3 dimensions, got {self.x.shape}.")

        if self.x.shape[1:] != (3, 240):
            raise ValueError(f"{role}: expected x shape (*, 3, 240), got {self.x.shape}.")

        if self.y.shape != (self.x.shape[0],):
            raise ValueError(f"{role}: y shape {self.y.shape} does not match x rows {self.x.shape[0]}.")

        if len(self.metadata) != self.x.shape[0]:
            raise ValueError(
                f"{role}: metadata rows {len(self.metadata)} do not match tensor rows {self.x.shape[0]}."
            )

        if not np.isfinite(self.x).all():
            raise ValueError(f"{role}: x contains non-finite values.")

        if not np.isfinite(self.y).all():
            raise ValueError(f"{role}: y contains non-finite values.")

    def __len__(self) -> int:
        return int(self.x.shape[0])

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return torch.from_numpy(self.x[index]), torch.tensor(self.y[index], dtype=torch.float32)


def make_eval_loader(dataset: MaterializedTensorDataset) -> DataLoader:
    """
    Create a deterministic evaluation DataLoader.
    """
    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding for transformer time tokens.
    """

    def __init__(self, d_model: int, max_len: int = 300, dropout: float = 0.1) -> None:
        super().__init__()

        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.size(1) > self.pe.size(1):
            raise ValueError(
                f"Input sequence length {x.size(1)} exceeds positional encoding max length {self.pe.size(1)}."
            )

        return self.dropout(x + self.pe[:, : x.size(1), :])


class TransformerEncoderLayerCustom(nn.Module):
    """
    Custom pre-norm transformer encoder layer used by the CRVSE checkpoint.
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError(f"d_model={d_model} must be divisible by n_heads={n_heads}.")

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def _attention(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, time_steps, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, time_steps, self.n_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, time_steps, -1)

        return self.out_proj(out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.dropout(self._attention(self.norm1(x)))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x


class CRVSEPhysFormer(nn.Module):
    """
    CNN + FFT + Transformer model for live-compatible rPPG HR estimation.
    """

    def __init__(
        self,
        in_channels: int = 3,
        cnn_channels: int = 16,
        freq_channels: int = 64,
        n_heads: int = 4,
        n_layers: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.11331939348791525,
        hr_min: float = 40.0,
        hr_max: float = 180.0,
        target_frames: int = 240,
        max_positional_length: int = 300,
    ) -> None:
        super().__init__()

        self.in_channels = in_channels
        self.target_frames = target_frames
        self.hr_min = hr_min
        self.hr_max = hr_max
        self.d_model = cnn_channels + freq_channels

        if self.d_model % n_heads != 0:
            raise ValueError(f"d_model={self.d_model} must be divisible by n_heads={n_heads}.")

        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, cnn_channels // 2, kernel_size=7, padding=3),
            nn.BatchNorm1d(cnn_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels // 2, cnn_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        n_fft_bins = target_frames // 2 + 1

        self.freq_proj = nn.Sequential(
            nn.Linear(n_fft_bins, freq_channels * 4),
            nn.ReLU(),
            nn.Linear(freq_channels * 4, freq_channels),
        )

        self.pos_enc = PositionalEncoding(
            d_model=self.d_model,
            max_len=max_positional_length,
            dropout=dropout,
        )

        self.transformer_layers = nn.ModuleList(
            [
                TransformerEncoderLayerCustom(
                    d_model=self.d_model,
                    n_heads=n_heads,
                    dim_feedforward=dim_feedforward,
                    dropout=dropout,
                )
                for _ in range(n_layers)
            ]
        )

        self.head = nn.Sequential(
            nn.Linear(self.d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError(f"Expected x shape (batch, channels, time), got {tuple(x.shape)}.")

        _, channels, time_steps = x.shape

        if channels != self.in_channels:
            raise ValueError(f"Expected {self.in_channels} channels, got {channels}.")

        if time_steps != self.target_frames:
            raise ValueError(f"Expected {self.target_frames} frames, got {time_steps}.")

        time_feat = self.encoder(x).permute(0, 2, 1)

        freq = torch.fft.rfft(x, norm="ortho")
        freq_mag = freq.abs().mean(dim=1)
        freq_feat = self.freq_proj(freq_mag)
        freq_feat = freq_feat.unsqueeze(1).expand(-1, time_steps, -1)

        combined = torch.cat([time_feat, freq_feat], dim=-1)
        combined = self.pos_enc(combined)

        for layer in self.transformer_layers:
            combined = layer(combined)

        out = self.head(combined.mean(dim=1)).squeeze(-1)

        if not self.training:
            out = out.clamp(self.hr_min, self.hr_max)

        return out


def build_frozen_checkpoint_model(checkpoint_path: Path) -> tuple[CRVSEPhysFormer, dict, dict]:
    """
    Build the app-compatible CRVSE PhysFormer and load frozen checkpoint weights.
    """
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

    if "model_state" not in checkpoint:
        raise KeyError("Checkpoint is missing required key: model_state.")

    state_dict = checkpoint["model_state"]
    best_params = checkpoint.get("best_params", {})

    layer_indices = [
        int(match.group(1))
        for key in state_dict
        for match in [re.match(r"transformer_layers\.(\d+)\.", key)]
        if match is not None
    ]

    model_config = {
        "in_channels": int(state_dict["encoder.0.weight"].shape[1]),
        "cnn_channels": int(state_dict["encoder.8.weight"].shape[0]),
        "freq_channels": int(state_dict["freq_proj.2.weight"].shape[0]),
        "n_heads": int(best_params.get("n_heads", 4)),
        "n_layers": max(layer_indices) + 1,
        "dim_feedforward": int(state_dict["transformer_layers.0.ff.0.weight"].shape[0]),
        "dropout": float(best_params.get("dropout", 0.11331939348791525)),
        "hr_min": 40.0,
        "hr_max": 180.0,
        "target_frames": int((state_dict["freq_proj.0.weight"].shape[1] - 1) * 2),
        "max_positional_length": 300,
    }

    model = CRVSEPhysFormer(**model_config)
    model.load_state_dict(state_dict, strict=True)
    model.to(DEVICE)
    model.eval()

    for parameter in model.parameters():
        parameter.requires_grad = False

    return model, checkpoint, model_config


def evaluate_hr_model(
    model: nn.Module,
    dataset: MaterializedTensorDataset,
    loader: DataLoader,
) -> tuple[pd.DataFrame, dict]:
    """
    Evaluate one HR model on one materialized tensor dataset.
    """
    prediction_batches = []
    target_batches = []

    model.eval()

    with torch.inference_mode():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=True)
            batch_pred = model(batch_x).detach().cpu().numpy().astype(np.float32)

            prediction_batches.append(batch_pred)
            target_batches.append(batch_y.numpy().astype(np.float32))

    pred_hr = np.concatenate(prediction_batches)
    target_hr = np.concatenate(target_batches)
    abs_error = np.abs(pred_hr - target_hr)

    rows = dataset.metadata.copy()
    rows["notebook_model_hr"] = pred_hr
    rows["target_hr_from_tensor"] = target_hr
    rows["notebook_abs_error"] = abs_error
    rows["role"] = dataset.role

    metrics = {
        "role": dataset.role,
        "rows": int(len(rows)),
        "mae_mean": float(abs_error.mean()),
        "mae_median": float(np.median(abs_error)),
        "mae_p90": float(np.quantile(abs_error, 0.90)),
        "mae_p95": float(np.quantile(abs_error, 0.95)),
        "mae_max": float(abs_error.max()),
    }

    return rows, metrics


def summarize_notebook_predictions_by_dataset_role(predictions: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize notebook frozen predictions by dataset and manifest role.
    """
    summary = (
        predictions.groupby(["dataset", "window_role"], dropna=False)
        .agg(
            rows=("notebook_abs_error", "size"),
            mae_mean=("notebook_abs_error", "mean"),
            mae_median=("notebook_abs_error", "median"),
            mae_p90=("notebook_abs_error", lambda s: float(s.quantile(0.90))),
            target_hr_mean=("target_hr_from_tensor", "mean"),
        )
        .reset_index()
        .sort_values(["window_role", "dataset"])
    )

    return summary


def summarize_saved_sourcefps_baseline(
    baseline_predictions_path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Summarize the saved frozen source-FPS baseline by role and dataset-role.
    """
    baseline_predictions = pd.read_csv(baseline_predictions_path, low_memory=False)

    sourcefps_baseline = baseline_predictions[
        (baseline_predictions["preprocessing_mode"] == "training_buffer_source_fps")
        & (baseline_predictions["preprocessing_status"] == "ok")
    ].copy()

    role_summary = (
        sourcefps_baseline.groupby("window_role", dropna=False)
        .agg(
            rows=("model_abs_error", "size"),
            saved_mae_mean=("model_abs_error", "mean"),
            saved_mae_median=("model_abs_error", "median"),
            saved_mae_p90=("model_abs_error", lambda s: float(s.quantile(0.90))),
            saved_mae_p95=("model_abs_error", lambda s: float(s.quantile(0.95))),
            saved_mae_max=("model_abs_error", "max"),
        )
        .reset_index()
        .rename(columns={"window_role": "role"})
    )

    dataset_role_summary = (
        sourcefps_baseline.groupby(["dataset", "window_role"], dropna=False)
        .agg(
            rows=("model_abs_error", "size"),
            saved_mae_mean=("model_abs_error", "mean"),
            saved_mae_median=("model_abs_error", "median"),
            saved_mae_p90=("model_abs_error", lambda s: float(s.quantile(0.90))),
            saved_target_hr_mean=("target_hr_mean", "mean"),
        )
        .reset_index()
        .sort_values(["window_role", "dataset"])
    )

    return role_summary, dataset_role_summary


def compare_role_summaries(
    notebook_summary: pd.DataFrame,
    saved_summary: pd.DataFrame,
) -> pd.DataFrame:
    """
    Compare notebook frozen metrics against saved source-FPS role metrics.
    """
    comparison = notebook_summary.merge(
        saved_summary,
        on=["role", "rows"],
        how="outer",
        indicator=True,
    )

    if not comparison["_merge"].eq("both").all():
        display(comparison)
        raise AssertionError("Notebook and saved baseline role summaries do not align.")

    comparison["delta_mae_mean"] = comparison["mae_mean"] - comparison["saved_mae_mean"]
    comparison["delta_mae_median"] = comparison["mae_median"] - comparison["saved_mae_median"]
    comparison["delta_mae_p90"] = comparison["mae_p90"] - comparison["saved_mae_p90"]
    comparison["delta_mae_p95"] = comparison["mae_p95"] - comparison["saved_mae_p95"]
    comparison["delta_mae_max"] = comparison["mae_max"] - comparison["saved_mae_max"]

    return comparison.drop(columns=["_merge"])


def compare_dataset_role_summaries(
    notebook_summary: pd.DataFrame,
    saved_summary: pd.DataFrame,
) -> pd.DataFrame:
    """
    Compare notebook frozen metrics against saved source-FPS dataset-role metrics.
    """
    comparison = notebook_summary.merge(
        saved_summary,
        on=["dataset", "window_role", "rows"],
        how="outer",
        indicator=True,
    )

    if not comparison["_merge"].eq("both").all():
        display(comparison)
        raise AssertionError("Notebook and saved baseline dataset-role summaries do not align.")

    comparison["delta_mae_mean"] = comparison["mae_mean"] - comparison["saved_mae_mean"]
    comparison["delta_mae_median"] = comparison["mae_median"] - comparison["saved_mae_median"]
    comparison["delta_mae_p90"] = comparison["mae_p90"] - comparison["saved_mae_p90"]
    comparison["delta_target_hr_mean"] = (
        comparison["target_hr_mean"] - comparison["saved_target_hr_mean"]
    )

    return comparison.drop(columns=["_merge"])


def round_metric_table(table: pd.DataFrame) -> pd.DataFrame:
    """
    Return a display copy with numeric metric columns rounded.
    """
    rounded = table.copy()

    for column in rounded.columns:
        if pd.api.types.is_numeric_dtype(rounded[column]):
            rounded[column] = rounded[column].round(4)

    return rounded


assert_nb11_setup_is_available()

datasets = {
    role: MaterializedTensorDataset(
        role=role,
        tensor_path=MATERIALIZED_TENSOR_PATHS[role],
        metadata_path=MATERIALIZED_METADATA_PATHS[role],
    )
    for role in ROLE_ORDER
}

loaders = {
    role: make_eval_loader(datasets[role])
    for role in ROLE_ORDER
}

frozen_model, frozen_checkpoint, frozen_model_config = build_frozen_checkpoint_model(
    ORIGINAL_CHECKPOINT_PATH
)

frozen_prediction_frames = []
frozen_metric_rows = []

for role in ROLE_ORDER:
    prediction_rows, metric_row = evaluate_hr_model(
        model=frozen_model,
        dataset=datasets[role],
        loader=loaders[role],
    )

    frozen_prediction_frames.append(prediction_rows)
    frozen_metric_rows.append(metric_row)

frozen_predictions = pd.concat(frozen_prediction_frames, ignore_index=True)
frozen_role_summary = pd.DataFrame(frozen_metric_rows)
frozen_dataset_role_summary = summarize_notebook_predictions_by_dataset_role(
    frozen_predictions
)

saved_role_summary, saved_dataset_role_summary = summarize_saved_sourcefps_baseline(
    BASELINE_PREDICTIONS_PATH
)

role_comparison = compare_role_summaries(
    notebook_summary=frozen_role_summary,
    saved_summary=saved_role_summary,
)

dataset_role_comparison = compare_dataset_role_summaries(
    notebook_summary=frozen_dataset_role_summary,
    saved_summary=saved_dataset_role_summary,
)

max_role_delta = float(
    role_comparison[
        [
            "delta_mae_mean",
            "delta_mae_median",
            "delta_mae_p90",
            "delta_mae_p95",
            "delta_mae_max",
        ]
    ].abs().max().max()
)

max_dataset_role_delta = float(
    dataset_role_comparison[
        [
            "delta_mae_mean",
            "delta_mae_median",
            "delta_mae_p90",
            "delta_target_hr_mean",
        ]
    ].abs().max().max()
)

baseline_reproduction_report = {
    "notebook_prediction_rows": int(len(frozen_predictions)),
    "saved_sourcefps_rows": int(saved_role_summary["rows"].sum()),
    "max_role_metric_delta": max_role_delta,
    "max_dataset_role_metric_delta": max_dataset_role_delta,
    "comparison_level": "role_and_dataset_role_aggregate",
    "row_level_join_used_as_gate": False,
}

if baseline_reproduction_report["notebook_prediction_rows"] != baseline_reproduction_report["saved_sourcefps_rows"]:
    raise AssertionError("Notebook prediction row count does not match saved source-FPS baseline row count.")

if max_role_delta > 1e-3:
    raise AssertionError("Frozen role-level metrics do not reproduce the saved source-FPS baseline.")

if max_dataset_role_delta > 1e-3:
    raise AssertionError("Frozen dataset-role metrics do not reproduce the saved source-FPS baseline.")

checkpoint_report = {
    "checkpoint_input_mode": frozen_checkpoint.get("input_mode"),
    "checkpoint_in_channels": frozen_checkpoint.get("in_channels"),
    "checkpoint_best_val_mae": frozen_checkpoint.get("best_val_mae"),
    "checkpoint_best_n_epochs": frozen_checkpoint.get("best_n_epochs"),
    "model_config": frozen_model_config,
    "trainable_parameters": int(
        sum(parameter.numel() for parameter in frozen_model.parameters() if parameter.requires_grad)
    ),
    "total_parameters": int(sum(parameter.numel() for parameter in frozen_model.parameters())),
    "device": str(DEVICE),
}

DATASETS = datasets
LOADERS = loaders
FROZEN_MODEL = frozen_model
FROZEN_CHECKPOINT = frozen_checkpoint
FROZEN_MODEL_CONFIG = frozen_model_config
FROZEN_PREDICTIONS = frozen_predictions
FROZEN_ROLE_SUMMARY = frozen_role_summary
FROZEN_DATASET_ROLE_SUMMARY = frozen_dataset_role_summary
SAVED_SOURCEFPS_ROLE_SUMMARY = saved_role_summary
SAVED_SOURCEFPS_DATASET_ROLE_SUMMARY = saved_dataset_role_summary
FROZEN_ROLE_COMPARISON = role_comparison
FROZEN_DATASET_ROLE_COMPARISON = dataset_role_comparison
FROZEN_BASELINE_REPRODUCTION_REPORT = baseline_reproduction_report

print("Frozen checkpoint report:")
print(json.dumps(checkpoint_report, indent=2))

print("\nFrozen cached evaluation by role:")
display(round_metric_table(FROZEN_ROLE_SUMMARY))

print("\nSaved source-FPS baseline by role:")
display(round_metric_table(SAVED_SOURCEFPS_ROLE_SUMMARY))

print("\nFrozen vs saved baseline role comparison:")
display(round_metric_table(FROZEN_ROLE_COMPARISON))

print("\nFrozen cached evaluation by dataset and role:")
display(round_metric_table(FROZEN_DATASET_ROLE_SUMMARY))

print("\nFrozen vs saved baseline dataset-role comparison:")
display(round_metric_table(FROZEN_DATASET_ROLE_COMPARISON))

print("\nBaseline reproduction report:")
print(json.dumps(FROZEN_BASELINE_REPRODUCTION_REPORT, indent=2))

print("\nPASS: NB11 frozen source-FPS aggregate evaluation reproduces the saved baseline.")

Frozen checkpoint report:
{
  "checkpoint_input_mode": "multichannel",
  "checkpoint_in_channels": 3,
  "checkpoint_best_val_mae": 6.900240182876587,
  "checkpoint_best_n_epochs": 50,
  "model_config": {
    "in_channels": 3,
    "cnn_channels": 16,
    "freq_channels": 64,
    "n_heads": 4,
    "n_layers": 4,
    "dim_feedforward": 256,
    "dropout": 0.11331939348791525,
    "hr_min": 40.0,
    "hr_max": 180.0,
    "target_frames": 240,
    "max_positional_length": 300
  },
  "trainable_parameters": 0,
  "total_parameters": 322145,
  "device": "cuda"
}

Frozen cached evaluation by role:


,role,rows,mae_mean,mae_median,mae_p90,mae_p95,mae_max
0,finetune_train,12503,5.8236,3.7437,13.7192,18.9426,54.9838
1,finetune_val,2839,6.0275,3.8533,14.0701,19.2319,50.2965
2,finetune_test,2828,5.7696,3.6274,13.6984,19.0461,72.2401
3,ood_eval,882,14.0355,7.6838,38.1498,49.9899,89.9549



Saved source-FPS baseline by role:


,role,rows,saved_mae_mean,saved_mae_median,saved_mae_p90,saved_mae_p95,saved_mae_max
0,finetune_test,2828,5.7696,3.6274,13.6984,19.0461,72.2401
1,finetune_train,12503,5.8236,3.7437,13.7192,18.9426,54.9838
2,finetune_val,2839,6.0275,3.8533,14.0701,19.2319,50.2965
3,ood_eval,882,14.0355,7.6839,38.1499,49.9899,89.9549



Frozen vs saved baseline role comparison:


...


Frozen cached evaluation by dataset and role:


,dataset,window_role,rows,mae_mean,mae_median,mae_p90,target_hr_mean
1,mcd_rppg,finetune_test,2094,4.8847,3.1502,10.9700,79.853302
4,ubfc_phys,finetune_test,658,8.8584,5.6735,20.2862,77.420601
7,ubfc_rppg,finetune_test,76,3.4085,3.1453,6.0835,98.269302
2,mcd_rppg,finetune_train,7446,4.6876,3.1639,10.3722,79.064796
5,ubfc_phys,finetune_train,4595,7.7819,5.4248,18.5928,80.305496
8,ubfc_rppg,finetune_train,462,4.6548,3.1447,9.9189,98.449203
3,mcd_rppg,finetune_val,1743,5.1325,3.3746,11.5368,79.108597
6,ubfc_phys,finetune_val,986,7.7629,4.8461,18.2725,80.002403
9,ubfc_rppg,finetune_val,110,4.6533,2.4438,12.3774,98.277199
0,ecg_fitness,ood_eval,882,14.0355,7.6838,38.1498,109.204803



Frozen vs saved baseline dataset-role comparison:


,dataset,window_role,rows,mae_mean,mae_median,mae_p90,target_hr_mean,saved_mae_mean,saved_mae_median,saved_mae_p90,saved_target_hr_mean,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_target_hr_mean
0,ecg_fitness,ood_eval,882,14.0355,7.6838,38.1498,109.204803,14.0355,7.6839,38.1499,109.2048,-0.0,-0.0,-0.0,0.0
1,mcd_rppg,finetune_test,2094,4.8847,3.1502,10.9700,79.853302,4.8847,3.1502,10.9700,79.8533,-0.0,0.0,0.0,0.0
2,mcd_rppg,finetune_train,7446,4.6876,3.1639,10.3722,79.064796,4.6876,3.1639,10.3722,79.0648,-0.0,-0.0,-0.0,-0.0
3,mcd_rppg,finetune_val,1743,5.1325,3.3746,11.5368,79.108597,5.1325,3.3746,11.5368,79.1086,-0.0,-0.0,0.0,-0.0
4,ubfc_phys,finetune_test,658,8.8584,5.6735,20.2862,77.420601,8.8584,5.6735,20.2862,77.4205,0.0,0.0,0.0,0.0
5,ubfc_phys,finetune_train,4595,7.7819,5.4248,18.5928,80.305496,7.7819,5.4248,18.5928,80.3055,0.0,0.0,-0.0,0.0
6,ubfc_phys,finetune_val,986,7.7629,4.8461,18.2725,80.002403,7.7629,4.8461,18.2725,80.0024,0.0,0.0,-0.0,0.0
7,ubfc_rppg,finetune_test,76,3.4085,3.1453,6.0835,98.269302,3.4085,3.1453,6.0835,98.2693,0.0,0.0,-0.0,-0.0
8,ubfc_rppg,finetune_train,462,4.6548,3.1447,9.9189,98.449203,4.6548,3.1447,9.9189,98.4492,0.0,-0.0,-0.0,-0.0
9,ubfc_rppg,finetune_val,110,4.6533,2.4438,12.3774,98.277199,4.6533,2.4438,12.3774,98.2772,0.0,0.0,0.0,-0.0



Baseline reproduction report:
{
  "notebook_prediction_rows": 19052,
  "saved_sourcefps_rows": 19052,
  "max_role_metric_delta": 4.0435791014203915e-05,
  "max_dataset_role_metric_delta": 4.57763671875e-05,
  "comparison_level": "role_and_dataset_role_aggregate",
  "row_level_join_used_as_gate": false
}

PASS: NB11 frozen source-FPS aggregate evaluation reproduces the saved baseline.


## Tiny Full-Model Transfer Sanity Check

### What is being tested

This section checks whether the original PhysFormer checkpoint can reduce loss on a small live-compatible training subset when all model parameters are trainable.

### Why this matters

Before running a full transfer-learning experiment, the notebook should prove that:

```text
cached live-compatible tensors -> trainable PhysFormer -> optimizer step -> lower training error

In [3]:
TINY_TRANSFER_ROWS_PER_DATASET = 32
TINY_TRANSFER_EPOCHS = 40
TINY_TRANSFER_LR = 1e-4
TINY_TRANSFER_WEIGHT_DECAY = 1e-4
TINY_TRANSFER_HUBER_DELTA = 6.5


def build_trainable_checkpoint_model(checkpoint_path: Path) -> CRVSEPhysFormer:
    """
    Build a CRVSE PhysFormer initialized from the original checkpoint.

    Unlike FROZEN_MODEL, all parameters are left trainable. This is used for
    transfer-learning experiments.
    """
    model, _, _ = build_frozen_checkpoint_model(checkpoint_path)

    for parameter in model.parameters():
        parameter.requires_grad = True

    return model


def count_parameters_by_trainability(model: nn.Module) -> pd.DataFrame:
    """
    Count trainable and frozen parameters by top-level module name.
    """
    rows = []

    for name, parameter in model.named_parameters():
        module = name.split(".")[0]

        rows.append(
            {
                "module": module,
                "trainable": bool(parameter.requires_grad),
                "parameters": int(parameter.numel()),
            }
        )

    return (
        pd.DataFrame(rows)
        .groupby(["module", "trainable"], as_index=False)["parameters"]
        .sum()
        .sort_values(["module", "trainable"])
    )


def select_tiny_balanced_indices(
    dataset: MaterializedTensorDataset,
    rows_per_dataset: int,
) -> np.ndarray:
    """
    Select a small deterministic dataset-balanced subset from finetune_train.

    The goal is not statistical representativeness. The goal is to avoid a tiny
    sanity check that accidentally uses only one dataset.
    """
    selected = []

    for dataset_name in sorted(dataset.metadata["dataset"].unique()):
        dataset_indices = np.flatnonzero(dataset.metadata["dataset"].to_numpy() == dataset_name)

        if len(dataset_indices) == 0:
            continue

        selected.extend(dataset_indices[:rows_per_dataset].tolist())

    return np.asarray(selected, dtype=np.int64)


def make_tiny_training_batch(
    dataset: MaterializedTensorDataset,
    indices: np.ndarray,
) -> tuple[torch.Tensor, torch.Tensor, pd.DataFrame]:
    """
    Build one small full-batch training tensor from selected cached rows.
    """
    x = torch.from_numpy(dataset.x[indices]).to(DEVICE)
    y = torch.from_numpy(dataset.y[indices]).to(DEVICE)

    metadata = dataset.metadata.iloc[indices].copy()
    metadata.insert(0, "cache_row_index", indices)

    return x, y, metadata


def compute_batch_metrics(
    model: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    loss_fn: nn.Module,
) -> dict:
    """
    Compute loss and MAE for one in-memory batch.
    """
    model.eval()

    with torch.inference_mode():
        pred = model(x)
        loss = loss_fn(pred, y)
        mae = torch.mean(torch.abs(pred - y))

    return {
        "loss": float(loss.item()),
        "mae": float(mae.item()),
        "pred_min": float(pred.min().item()),
        "pred_max": float(pred.max().item()),
    }


def run_tiny_full_model_transfer_check(
    train_dataset: MaterializedTensorDataset,
) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    """
    Run a tiny full-model transfer-learning sanity check.

    This trains on a small fixed subset only. It is expected to reduce training
    loss if the model, optimizer, labels, and gradients are wired correctly.
    """
    tiny_indices = select_tiny_balanced_indices(
        dataset=train_dataset,
        rows_per_dataset=TINY_TRANSFER_ROWS_PER_DATASET,
    )

    x, y, metadata = make_tiny_training_batch(
        dataset=train_dataset,
        indices=tiny_indices,
    )

    model = build_trainable_checkpoint_model(ORIGINAL_CHECKPOINT_PATH).to(DEVICE)

    parameter_summary = count_parameters_by_trainability(model)

    optimizer = AdamW(
        model.parameters(),
        lr=TINY_TRANSFER_LR,
        weight_decay=TINY_TRANSFER_WEIGHT_DECAY,
    )

    loss_fn = nn.HuberLoss(delta=TINY_TRANSFER_HUBER_DELTA)

    initial_metrics = compute_batch_metrics(model, x, y, loss_fn)

    history_rows = []

    for epoch in range(1, TINY_TRANSFER_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        pred = model(x)
        loss = loss_fn(pred, y)
        mae = torch.mean(torch.abs(pred - y))

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=10.0,
        )

        optimizer.step()

        history_rows.append(
            {
                "epoch": epoch,
                "loss": float(loss.item()),
                "mae": float(mae.item()),
                "grad_norm_before_clip": float(grad_norm.item()),
            }
        )

    final_metrics = compute_batch_metrics(model, x, y, loss_fn)

    history = pd.DataFrame(history_rows)

    report = {
        "tiny_rows": int(len(tiny_indices)),
        "datasets": metadata["dataset"].value_counts().sort_index().to_dict(),
        "x_shape": tuple(x.shape),
        "y_shape": tuple(y.shape),
        "initial_loss": initial_metrics["loss"],
        "final_loss": final_metrics["loss"],
        "initial_mae": initial_metrics["mae"],
        "final_mae": final_metrics["mae"],
        "min_training_mae_during_updates": float(history["mae"].min()),
        "loss_reduced": bool(final_metrics["loss"] < initial_metrics["loss"]),
        "mae_reduced": bool(final_metrics["mae"] < initial_metrics["mae"]),
        "lr": TINY_TRANSFER_LR,
        "weight_decay": TINY_TRANSFER_WEIGHT_DECAY,
        "huber_delta": TINY_TRANSFER_HUBER_DELTA,
    }

    if not report["loss_reduced"]:
        raise AssertionError(
            "Tiny full-model transfer check did not reduce loss. "
            "Do not proceed to full training until this is understood."
        )

    return history, report, parameter_summary


tiny_transfer_history, tiny_transfer_report, tiny_transfer_parameter_summary = (
    run_tiny_full_model_transfer_check(DATASETS["finetune_train"])
)

print("Tiny full-model transfer parameter summary:")
display(tiny_transfer_parameter_summary)

print("\nTiny full-model transfer report:")
print(json.dumps(tiny_transfer_report, indent=2))

print("\nFirst 10 epochs:")
display(round_metric_table(tiny_transfer_history.head(10)))

print("\nLast 10 epochs:")
display(round_metric_table(tiny_transfer_history.tail(10)))

print("\nPASS: Tiny full-model transfer sanity check reduced loss on live-compatible tensors.")

Tiny full-model transfer parameter summary:


,module,trainable,parameters
0,encoder,True,1696
1,freq_proj,True,47680
2,head,True,2625
3,transformer_layers,True,270144



Tiny full-model transfer report:
{
  "tiny_rows": 96,
  "datasets": {
    "mcd_rppg": 32,
    "ubfc_phys": 32,
    "ubfc_rppg": 32
  },
  "x_shape": [
    96,
    3,
    240
  ],
  "y_shape": [
    96
  ],
  "initial_loss": 27.2802734375,
  "final_loss": 17.433170318603516,
  "initial_mae": 6.462228298187256,
  "final_mae": 5.240494728088379,
  "min_training_mae_during_updates": 7.200870037078857,
  "loss_reduced": true,
  "mae_reduced": true,
  "lr": 0.0001,
  "weight_decay": 0.0001,
  "huber_delta": 6.5
}

First 10 epochs:


,epoch,loss,mae,grad_norm_before_clip
0,1,53.2894,11.0160,256.1584
1,2,43.7195,9.5950,175.2224
2,3,52.7387,11.0356,185.5040
3,4,47.3262,10.2055,411.6986
4,5,40.4373,9.1346,193.1730
5,6,42.0115,9.2768,146.9765
6,7,43.8872,9.4922,284.1102
7,8,38.2273,8.5477,109.3850
8,9,41.9626,9.2064,144.0251
9,10,38.2162,8.6539,401.9509



Last 10 epochs:


,epoch,loss,mae,grad_norm_before_clip
30,31,36.5114,8.1205,140.9918
31,32,35.9199,8.2483,158.5467
32,33,33.1801,7.7467,201.0106
33,34,40.0733,8.9736,182.6833
34,35,36.2399,8.1966,275.9403
35,36,36.7230,8.4189,103.2319
36,37,32.7319,7.7261,97.1149
37,38,36.6848,8.4471,183.3607
38,39,35.9153,8.3139,252.0083
39,40,39.4137,8.9511,195.0959



PASS: Tiny full-model transfer sanity check reduced loss on live-compatible tensors.


## Full-Model Transfer Learning From Original Checkpoint

### What is being tested

This section tests whether the original multichannel CRVSE PhysFormer checkpoint improves when all model parameters are trained on the live-compatible source-FPS tensor distribution.

### Why this matters

`NB_P2_10` showed that shallow adaptation was not enough.

This experiment allows the full model to adapt:

```text
encoder + FFT branch + transformer layers + HR head

In [4]:
FULL_TRANSFER_BATCH_SIZE = 256
FULL_TRANSFER_MAX_EPOCHS = 25
FULL_TRANSFER_PATIENCE = 6
FULL_TRANSFER_MIN_DELTA = 1e-3
FULL_TRANSFER_LR = 3e-5
FULL_TRANSFER_WEIGHT_DECAY = 1e-4
FULL_TRANSFER_HUBER_DELTA = 6.5
FULL_TRANSFER_GRAD_CLIP = 5.0

FULL_TRANSFER_DIR = KAGGLE_WORKING / "crvse_full_model_transfer_sourcefps"
FULL_TRANSFER_DIR.mkdir(parents=True, exist_ok=True)


def assert_full_transfer_prerequisites() -> None:
    """
    Check that the frozen baseline section has been executed successfully.
    """
    required_names = [
        "DATASETS",
        "FROZEN_ROLE_SUMMARY",
        "FROZEN_DATASET_ROLE_SUMMARY",
        "build_frozen_checkpoint_model",
        "evaluate_hr_model",
        "round_metric_table",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run the frozen baseline reproduction cell before this training cell. "
            f"Missing names: {missing}"
        )


def build_full_transfer_model(checkpoint_path: Path) -> CRVSEPhysFormer:
    """
    Build a CRVSE PhysFormer initialized from the original checkpoint.

    All parameters are made trainable for full-model transfer learning.
    """
    model, _, _ = build_frozen_checkpoint_model(checkpoint_path)

    for parameter in model.parameters():
        parameter.requires_grad = True

    return model


def count_parameters_by_trainability(model: nn.Module) -> pd.DataFrame:
    """
    Count trainable and frozen parameters by top-level module.
    """
    rows = []

    for name, parameter in model.named_parameters():
        rows.append(
            {
                "module": name.split(".")[0],
                "trainable": bool(parameter.requires_grad),
                "parameters": int(parameter.numel()),
            }
        )

    return (
        pd.DataFrame(rows)
        .groupby(["module", "trainable"], as_index=False)["parameters"]
        .sum()
        .sort_values(["module", "trainable"])
    )


def make_training_loader(dataset: MaterializedTensorDataset) -> DataLoader:
    """
    Create the shuffled training DataLoader for natural manifest distribution.
    """
    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=FULL_TRANSFER_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        generator=generator,
    )


def train_one_full_transfer_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
) -> dict:
    """
    Train one epoch and return training-mode aggregate metrics.
    """
    model.train()

    total_loss = 0.0
    total_mae = 0.0
    total_rows = 0
    grad_norms = []

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        pred = model(batch_x)
        loss = loss_fn(pred, batch_y)
        mae = torch.mean(torch.abs(pred - batch_y))

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=FULL_TRANSFER_GRAD_CLIP,
        )

        optimizer.step()

        batch_size = int(batch_x.shape[0])
        total_loss += float(loss.item()) * batch_size
        total_mae += float(mae.item()) * batch_size
        total_rows += batch_size
        grad_norms.append(float(grad_norm.item()))

    return {
        "train_loss": total_loss / total_rows,
        "train_mae": total_mae / total_rows,
        "grad_norm_mean": float(np.mean(grad_norms)),
        "grad_norm_max": float(np.max(grad_norms)),
    }


def evaluate_candidate_on_all_roles(
    model: nn.Module,
    datasets: dict[str, MaterializedTensorDataset],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Evaluate a candidate model on train, validation, test, and OOD roles.
    """
    prediction_frames = []
    metric_rows = []

    for role in ROLE_ORDER:
        loader = make_eval_loader(datasets[role])

        prediction_rows, metric_row = evaluate_hr_model(
            model=model,
            dataset=datasets[role],
            loader=loader,
        )

        prediction_frames.append(prediction_rows)
        metric_rows.append(metric_row)

    predictions = pd.concat(prediction_frames, ignore_index=True)
    role_summary = pd.DataFrame(metric_rows)
    dataset_role_summary = summarize_notebook_predictions_by_dataset_role(predictions)

    return predictions, role_summary, dataset_role_summary


def compare_candidate_to_frozen_by_role(
    candidate_summary: pd.DataFrame,
) -> pd.DataFrame:
    """
    Compare candidate role metrics against the frozen source-FPS baseline.
    """
    frozen = FROZEN_ROLE_SUMMARY.rename(
        columns={
            "mae_mean": "frozen_mae_mean",
            "mae_median": "frozen_mae_median",
            "mae_p90": "frozen_mae_p90",
            "mae_p95": "frozen_mae_p95",
            "mae_max": "frozen_mae_max",
        }
    )

    candidate = candidate_summary.rename(
        columns={
            "mae_mean": "candidate_mae_mean",
            "mae_median": "candidate_mae_median",
            "mae_p90": "candidate_mae_p90",
            "mae_p95": "candidate_mae_p95",
            "mae_max": "candidate_mae_max",
        }
    )

    comparison = frozen.merge(candidate, on=["role", "rows"], how="inner")

    comparison["delta_mae_mean"] = (
        comparison["candidate_mae_mean"] - comparison["frozen_mae_mean"]
    )
    comparison["delta_mae_median"] = (
        comparison["candidate_mae_median"] - comparison["frozen_mae_median"]
    )
    comparison["delta_mae_p90"] = (
        comparison["candidate_mae_p90"] - comparison["frozen_mae_p90"]
    )
    comparison["delta_mae_p95"] = (
        comparison["candidate_mae_p95"] - comparison["frozen_mae_p95"]
    )
    comparison["delta_mae_max"] = (
        comparison["candidate_mae_max"] - comparison["frozen_mae_max"]
    )

    return comparison


def compare_candidate_to_frozen_by_dataset_role(
    candidate_summary: pd.DataFrame,
) -> pd.DataFrame:
    """
    Compare candidate dataset-role metrics against the frozen baseline.
    """
    frozen = FROZEN_DATASET_ROLE_SUMMARY.rename(
        columns={
            "mae_mean": "frozen_mae_mean",
            "mae_median": "frozen_mae_median",
            "mae_p90": "frozen_mae_p90",
        }
    )

    candidate = candidate_summary.rename(
        columns={
            "mae_mean": "candidate_mae_mean",
            "mae_median": "candidate_mae_median",
            "mae_p90": "candidate_mae_p90",
        }
    )

    comparison = frozen.merge(
        candidate,
        on=["dataset", "window_role", "rows", "target_hr_mean"],
        how="inner",
    )

    comparison["delta_mae_mean"] = (
        comparison["candidate_mae_mean"] - comparison["frozen_mae_mean"]
    )
    comparison["delta_mae_median"] = (
        comparison["candidate_mae_median"] - comparison["frozen_mae_median"]
    )
    comparison["delta_mae_p90"] = (
        comparison["candidate_mae_p90"] - comparison["frozen_mae_p90"]
    )

    return comparison


def get_role_delta(comparison: pd.DataFrame, role: str, column: str) -> float:
    """
    Extract one role-level delta metric from a comparison table.
    """
    matching = comparison.loc[comparison["role"] == role, column]

    if len(matching) != 1:
        raise ValueError(f"Expected one row for role={role!r}, found {len(matching)}.")

    return float(matching.iloc[0])


def run_full_model_transfer_experiment() -> dict:
    """
    Run full-model transfer learning from the original checkpoint.

    The best state is selected by validation mean MAE relative to the frozen
    source-FPS baseline.
    """
    assert_full_transfer_prerequisites()

    model = build_full_transfer_model(ORIGINAL_CHECKPOINT_PATH)
    parameter_summary = count_parameters_by_trainability(model)

    train_loader = make_training_loader(DATASETS["finetune_train"])
    val_loader = make_eval_loader(DATASETS["finetune_val"])

    optimizer = AdamW(
        model.parameters(),
        lr=FULL_TRANSFER_LR,
        weight_decay=FULL_TRANSFER_WEIGHT_DECAY,
    )

    loss_fn = nn.HuberLoss(delta=FULL_TRANSFER_HUBER_DELTA)

    frozen_val_mae = get_role_delta(
        FROZEN_ROLE_SUMMARY.assign(delta_mae_mean=FROZEN_ROLE_SUMMARY["mae_mean"]),
        role="finetune_val",
        column="delta_mae_mean",
    )

    best_val_mae = float(
        FROZEN_ROLE_SUMMARY.loc[
            FROZEN_ROLE_SUMMARY["role"] == "finetune_val",
            "mae_mean",
        ].iloc[0]
    )

    best_epoch = 0
    best_state_dict = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    history_rows = []

    print(f"Frozen validation MAE baseline: {best_val_mae:.4f} BPM")

    for epoch in range(1, FULL_TRANSFER_MAX_EPOCHS + 1):
        train_metrics = train_one_full_transfer_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
        )

        _, val_metrics = evaluate_hr_model(
            model=model,
            dataset=DATASETS["finetune_val"],
            loader=val_loader,
        )

        improved = val_metrics["mae_mean"] < (best_val_mae - FULL_TRANSFER_MIN_DELTA)

        if improved:
            best_val_mae = float(val_metrics["mae_mean"])
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history_rows.append(
            {
                "epoch": epoch,
                **train_metrics,
                "val_mae": float(val_metrics["mae_mean"]),
                "val_median": float(val_metrics["mae_median"]),
                "val_p90": float(val_metrics["mae_p90"]),
                "improved": bool(improved),
                "best_epoch_so_far": int(best_epoch),
                "best_val_mae_so_far": float(best_val_mae),
            }
        )

        print(
            f"Epoch {epoch:02d} | "
            f"train MAE {train_metrics['train_mae']:.4f} | "
            f"val MAE {val_metrics['mae_mean']:.4f} | "
            f"best {best_val_mae:.4f} @ epoch {best_epoch}"
        )

        if epochs_without_improvement >= FULL_TRANSFER_PATIENCE:
            print(f"Early stopping after {epoch} epochs.")
            break

    model.load_state_dict(best_state_dict)

    candidate_predictions, candidate_role_summary, candidate_dataset_role_summary = (
        evaluate_candidate_on_all_roles(model=model, datasets=DATASETS)
    )

    role_comparison = compare_candidate_to_frozen_by_role(candidate_role_summary)
    dataset_role_comparison = compare_candidate_to_frozen_by_dataset_role(
        candidate_dataset_role_summary
    )

    history = pd.DataFrame(history_rows)

    val_delta = get_role_delta(role_comparison, "finetune_val", "delta_mae_mean")
    test_delta = get_role_delta(role_comparison, "finetune_test", "delta_mae_mean")
    ood_delta = get_role_delta(role_comparison, "ood_eval", "delta_mae_mean")

    decision = (
        "candidate_for_analysis"
        if val_delta <= -0.20 and test_delta <= -0.10 and ood_delta <= 0.50
        else "research_result_only"
    )

    history_path = FULL_TRANSFER_DIR / "full_model_transfer_history.csv"
    comparison_path = FULL_TRANSFER_DIR / "full_model_transfer_role_comparison.csv"
    dataset_role_path = FULL_TRANSFER_DIR / "full_model_transfer_dataset_role_comparison.csv"
    checkpoint_path = FULL_TRANSFER_DIR / "full_model_transfer_best.pt"

    history.to_csv(history_path, index=False)
    role_comparison.to_csv(comparison_path, index=False)
    dataset_role_comparison.to_csv(dataset_role_path, index=False)

    torch.save(
        {
            "experiment": "NB_P2_11_full_model_transfer_sourcefps",
            "model_state": model.state_dict(),
            "model_config": FROZEN_MODEL_CONFIG,
            "best_epoch": best_epoch,
            "frozen_val_mae": float(
                FROZEN_ROLE_SUMMARY.loc[
                    FROZEN_ROLE_SUMMARY["role"] == "finetune_val",
                    "mae_mean",
                ].iloc[0]
            ),
            "best_val_mae": float(best_val_mae),
            "role_comparison": role_comparison.to_dict("records"),
            "dataset_role_comparison": dataset_role_comparison.to_dict("records"),
        },
        checkpoint_path,
    )

    result_summary = {
        "best_epoch": int(best_epoch),
        "frozen_val_mae": float(
            FROZEN_ROLE_SUMMARY.loc[
                FROZEN_ROLE_SUMMARY["role"] == "finetune_val",
                "mae_mean",
            ].iloc[0]
        ),
        "best_val_mae": float(best_val_mae),
        "val_delta_vs_frozen": float(val_delta),
        "test_delta_vs_frozen": float(test_delta),
        "ood_delta_vs_frozen": float(ood_delta),
        "decision": decision,
        "history_path": str(history_path),
        "comparison_path": str(comparison_path),
        "dataset_role_path": str(dataset_role_path),
        "checkpoint_path": str(checkpoint_path),
    }

    return {
        "model": model,
        "parameter_summary": parameter_summary,
        "history": history,
        "candidate_predictions": candidate_predictions,
        "candidate_role_summary": candidate_role_summary,
        "candidate_dataset_role_summary": candidate_dataset_role_summary,
        "role_comparison": role_comparison,
        "dataset_role_comparison": dataset_role_comparison,
        "result_summary": result_summary,
    }


full_transfer_result = run_full_model_transfer_experiment()

FULL_TRANSFER_MODEL = full_transfer_result["model"]
FULL_TRANSFER_HISTORY = full_transfer_result["history"]
FULL_TRANSFER_ROLE_COMPARISON = full_transfer_result["role_comparison"]
FULL_TRANSFER_DATASET_ROLE_COMPARISON = full_transfer_result["dataset_role_comparison"]
FULL_TRANSFER_RESULT_SUMMARY = full_transfer_result["result_summary"]

print("\nFull-model transfer parameter summary:")
display(full_transfer_result["parameter_summary"])

print("\nFull-model transfer history:")
display(round_metric_table(FULL_TRANSFER_HISTORY))

print("\nFull-model transfer vs frozen role comparison:")
display(round_metric_table(FULL_TRANSFER_ROLE_COMPARISON))

print("\nFull-model transfer vs frozen dataset-role comparison:")
display(round_metric_table(FULL_TRANSFER_DATASET_ROLE_COMPARISON))

print("\nFull-model transfer result summary:")
print(json.dumps(FULL_TRANSFER_RESULT_SUMMARY, indent=2))

print("\nDONE: Full-model transfer-learning experiment completed.")

Frozen validation MAE baseline: 6.0275 BPM
Epoch 01 | train MAE 8.5642 | val MAE 5.9458 | best 5.9458 @ epoch 1
Epoch 02 | train MAE 8.4441 | val MAE 5.9510 | best 5.9458 @ epoch 1
Epoch 03 | train MAE 8.5966 | val MAE 5.9104 | best 5.9104 @ epoch 3
Epoch 04 | train MAE 8.5323 | val MAE 5.9497 | best 5.9104 @ epoch 3
Epoch 05 | train MAE 8.5699 | val MAE 5.8709 | best 5.8709 @ epoch 5
Epoch 06 | train MAE 8.5696 | val MAE 5.8781 | best 5.8709 @ epoch 5
Epoch 07 | train MAE 8.5512 | val MAE 5.8677 | best 5.8677 @ epoch 7
Epoch 08 | train MAE 8.4783 | val MAE 5.9374 | best 5.8677 @ epoch 7
Epoch 09 | train MAE 8.4436 | val MAE 5.8482 | best 5.8482 @ epoch 9
Epoch 10 | train MAE 8.4256 | val MAE 5.9000 | best 5.8482 @ epoch 9
Epoch 11 | train MAE 8.4432 | val MAE 5.9172 | best 5.8482 @ epoch 9
Epoch 12 | train MAE 8.3415 | val MAE 5.9500 | best 5.8482 @ epoch 9
Epoch 13 | train MAE 8.4173 | val MAE 5.8709 | best 5.8482 @ epoch 9
Epoch 14 | train MAE 8.5273 | val MAE 5.8542 | best 5.8482 @

,module,trainable,parameters
0,encoder,True,1696
1,freq_proj,True,47680
2,head,True,2625
3,transformer_layers,True,270144



Full-model transfer history:


,epoch,train_loss,train_mae,grad_norm_mean,grad_norm_max,val_mae,val_median,val_p90,improved,best_epoch_so_far,best_val_mae_so_far
0,1,38.1884,8.5642,114.6364,228.1400,5.9458,3.8317,14.1410,True,1,5.9458
1,2,37.4427,8.4441,121.0330,243.8736,5.9510,3.8103,13.9917,False,1,5.9458
2,3,38.3450,8.5966,120.6999,232.8223,5.9104,3.7972,14.1963,True,3,5.9104
3,4,38.0243,8.5323,105.6362,193.6909,5.9497,3.8186,13.8704,False,3,5.9104
4,5,38.2142,8.5699,125.3162,281.8083,5.8709,3.7656,13.8742,True,5,5.8709
5,6,38.2079,8.5696,117.9732,295.7976,5.8781,3.7377,13.8487,False,5,5.8709
6,7,38.0939,8.5512,126.4675,331.4384,5.8677,3.7333,13.9253,True,7,5.8677
7,8,37.7046,8.4783,108.2482,243.6109,5.9374,3.8344,13.8510,False,7,5.8677
8,9,37.4146,8.4436,107.9326,208.6938,5.8482,3.7303,13.7801,True,9,5.8482
9,10,37.3433,8.4256,111.7760,287.3383,5.9000,3.7363,13.9709,False,9,5.8482



Full-model transfer vs frozen role comparison:


,role,rows,frozen_mae_mean,frozen_mae_median,frozen_mae_p90,frozen_mae_p95,frozen_mae_max,candidate_mae_mean,candidate_mae_median,candidate_mae_p90,candidate_mae_p95,candidate_mae_max,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_mae_max
0,finetune_train,12503,5.8236,3.7437,13.7192,18.9426,54.9838,5.5693,3.5789,13.0876,17.9455,59.8709,-0.2543,-0.1648,-0.6316,-0.9972,4.8871
1,finetune_val,2839,6.0275,3.8533,14.0701,19.2319,50.2965,5.8482,3.7303,13.7801,18.8157,49.8279,-0.1793,-0.1230,-0.2900,-0.4162,-0.4686
2,finetune_test,2828,5.7696,3.6274,13.6984,19.0461,72.2401,5.5868,3.4523,13.0519,18.7095,72.6520,-0.1829,-0.1751,-0.6465,-0.3366,0.4120
3,ood_eval,882,14.0355,7.6838,38.1498,49.9899,89.9549,15.1386,8.1842,40.2203,52.5953,89.4354,1.1031,0.5003,2.0705,2.6054,-0.5195



Full-model transfer vs frozen dataset-role comparison:


,dataset,window_role,rows,frozen_mae_mean,frozen_mae_median,frozen_mae_p90,target_hr_mean,candidate_mae_mean,candidate_mae_median,candidate_mae_p90,delta_mae_mean,delta_mae_median,delta_mae_p90
0,mcd_rppg,finetune_test,2094,4.8847,3.1502,10.9700,79.853302,4.6917,2.9469,10.5556,-0.1930,-0.2033,-0.4144
1,ubfc_phys,finetune_test,658,8.8584,5.6735,20.2862,77.420601,8.6892,5.5384,20.1073,-0.1692,-0.1351,-0.1789
2,ubfc_rppg,finetune_test,76,3.4085,3.1453,6.0835,98.269302,3.3874,3.1672,5.9411,-0.0211,0.0219,-0.1424
3,mcd_rppg,finetune_train,7446,4.6876,3.1639,10.3722,79.064796,4.3588,2.9670,9.6120,-0.3289,-0.1969,-0.7602
4,ubfc_phys,finetune_train,4595,7.7819,5.4248,18.5928,80.305496,7.6382,5.3805,18.0448,-0.1437,-0.0443,-0.5481
5,ubfc_rppg,finetune_train,462,4.6548,3.1447,9.9189,98.449203,4.5014,3.2839,9.6632,-0.1534,0.1392,-0.2557
6,mcd_rppg,finetune_val,1743,5.1325,3.3746,11.5368,79.108597,4.8776,3.2282,11.0521,-0.2549,-0.1465,-0.4848
7,ubfc_phys,finetune_val,986,7.7629,4.8461,18.2725,80.002403,7.7043,5.0439,18.2872,-0.0586,0.1978,0.0147
8,ubfc_rppg,finetune_val,110,4.6533,2.4438,12.3774,98.277199,4.5915,2.4584,11.5249,-0.0618,0.0146,-0.8525
9,ecg_fitness,ood_eval,882,14.0355,7.6838,38.1498,109.204803,15.1386,8.1842,40.2203,1.1031,0.5003,2.0705



Full-model transfer result summary:
{
  "best_epoch": 9,
  "frozen_val_mae": 6.027502059936523,
  "best_val_mae": 5.848221302032471,
  "val_delta_vs_frozen": -0.17928075790405273,
  "test_delta_vs_frozen": -0.18285274505615234,
  "ood_delta_vs_frozen": 1.1030645370483398,
  "decision": "research_result_only",
  "history_path": "/kaggle/working/crvse_full_model_transfer_sourcefps/full_model_transfer_history.csv",
  "comparison_path": "/kaggle/working/crvse_full_model_transfer_sourcefps/full_model_transfer_role_comparison.csv",
  "dataset_role_path": "/kaggle/working/crvse_full_model_transfer_sourcefps/full_model_transfer_dataset_role_comparison.csv",
  "checkpoint_path": "/kaggle/working/crvse_full_model_transfer_sourcefps/full_model_transfer_best.pt"
}

DONE: Full-model transfer-learning experiment completed.


## Balanced Full-Model Transfer Learning

### What is being tested

This section tests whether dataset-balanced full-model transfer learning improves live-compatible validation and test performance while reducing the out-of-domain degradation seen in natural full-model transfer.

### Why this matters

The natural training distribution is not dataset-balanced.

A model can improve aggregate validation and test MAE while quietly adapting too strongly to the dominant datasets. Balanced sampling gives each training dataset more equal influence during optimization.

In [5]:
BALANCED_FULL_TRANSFER_BATCH_SIZE = 256
BALANCED_FULL_TRANSFER_MAX_EPOCHS = 25
BALANCED_FULL_TRANSFER_PATIENCE = 6
BALANCED_FULL_TRANSFER_MIN_DELTA = 1e-3
BALANCED_FULL_TRANSFER_LR = 2e-5
BALANCED_FULL_TRANSFER_WEIGHT_DECAY = 1e-4
BALANCED_FULL_TRANSFER_HUBER_DELTA = 6.5
BALANCED_FULL_TRANSFER_GRAD_CLIP = 5.0

BALANCED_FULL_TRANSFER_DIR = KAGGLE_WORKING / "crvse_balanced_full_model_transfer_sourcefps"
BALANCED_FULL_TRANSFER_DIR.mkdir(parents=True, exist_ok=True)


def assert_balanced_full_transfer_prerequisites() -> None:
    """
    Check that the frozen baseline and natural full-transfer cells have run.
    """
    required_names = [
        "DATASETS",
        "FROZEN_ROLE_SUMMARY",
        "FROZEN_DATASET_ROLE_SUMMARY",
        "build_full_transfer_model",
        "count_parameters_by_trainability",
        "evaluate_hr_model",
        "evaluate_candidate_on_all_roles",
        "compare_candidate_to_frozen_by_role",
        "compare_candidate_to_frozen_by_dataset_role",
        "get_role_delta",
        "round_metric_table",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run the frozen baseline and full-transfer cells before this cell. "
            f"Missing names: {missing}"
        )


def make_dataset_balanced_sampler(dataset: MaterializedTensorDataset) -> WeightedRandomSampler:
    """
    Build an inverse-frequency sampler over dataset names.

    Each row receives weight 1 / count(dataset). This makes small datasets appear
    more often during training without changing the validation or test sets.
    """
    dataset_names = dataset.metadata["dataset"].astype(str).to_numpy()
    counts = pd.Series(dataset_names).value_counts().to_dict()

    weights = np.asarray(
        [1.0 / counts[name] for name in dataset_names],
        dtype=np.float64,
    )

    return WeightedRandomSampler(
        weights=torch.as_tensor(weights, dtype=torch.double),
        num_samples=len(weights),
        replacement=True,
    )


def make_balanced_training_loader(dataset: MaterializedTensorDataset) -> DataLoader:
    """
    Create a dataset-balanced training DataLoader.
    """
    sampler = make_dataset_balanced_sampler(dataset)

    return DataLoader(
        dataset,
        batch_size=BALANCED_FULL_TRANSFER_BATCH_SIZE,
        sampler=sampler,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def preview_balanced_batches(
    dataset: MaterializedTensorDataset,
    loader: DataLoader,
    n_batches: int = 5,
) -> pd.DataFrame:
    """
    Show approximate dataset composition from the first sampled batches.
    """
    rows = []

    for batch_index, (_, _) in enumerate(loader):
        if batch_index >= n_batches:
            break

        start = batch_index * BALANCED_FULL_TRANSFER_BATCH_SIZE
        end = start + BALANCED_FULL_TRANSFER_BATCH_SIZE

        sampled_indices = list(loader.sampler)[start:end]
        sampled_metadata = dataset.metadata.iloc[sampled_indices]

        counts = sampled_metadata["dataset"].value_counts().sort_index()

        for dataset_name, count in counts.items():
            rows.append(
                {
                    "batch": batch_index,
                    "dataset": dataset_name,
                    "rows": int(count),
                }
            )

    return pd.DataFrame(rows)


def train_one_balanced_full_transfer_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
) -> dict:
    """
    Train one epoch using the balanced sampler.
    """
    model.train()

    total_loss = 0.0
    total_mae = 0.0
    total_rows = 0
    grad_norms = []

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        pred = model(batch_x)
        loss = loss_fn(pred, batch_y)
        mae = torch.mean(torch.abs(pred - batch_y))

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=BALANCED_FULL_TRANSFER_GRAD_CLIP,
        )

        optimizer.step()

        batch_size = int(batch_x.shape[0])
        total_loss += float(loss.item()) * batch_size
        total_mae += float(mae.item()) * batch_size
        total_rows += batch_size
        grad_norms.append(float(grad_norm.item()))

    return {
        "train_loss": total_loss / total_rows,
        "train_mae": total_mae / total_rows,
        "grad_norm_mean": float(np.mean(grad_norms)),
        "grad_norm_max": float(np.max(grad_norms)),
    }


def run_balanced_full_model_transfer_experiment() -> dict:
    """
    Run full-model transfer learning with dataset-balanced sampling.
    """
    assert_balanced_full_transfer_prerequisites()

    model = build_full_transfer_model(ORIGINAL_CHECKPOINT_PATH)
    parameter_summary = count_parameters_by_trainability(model)

    train_loader = make_balanced_training_loader(DATASETS["finetune_train"])
    val_loader = make_eval_loader(DATASETS["finetune_val"])

    sampler_preview = preview_balanced_batches(
        dataset=DATASETS["finetune_train"],
        loader=train_loader,
        n_batches=5,
    )

    optimizer = AdamW(
        model.parameters(),
        lr=BALANCED_FULL_TRANSFER_LR,
        weight_decay=BALANCED_FULL_TRANSFER_WEIGHT_DECAY,
    )

    loss_fn = nn.HuberLoss(delta=BALANCED_FULL_TRANSFER_HUBER_DELTA)

    best_val_mae = float(
        FROZEN_ROLE_SUMMARY.loc[
            FROZEN_ROLE_SUMMARY["role"] == "finetune_val",
            "mae_mean",
        ].iloc[0]
    )

    frozen_val_mae = best_val_mae
    best_epoch = 0
    best_state_dict = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    history_rows = []

    print(f"Frozen validation MAE baseline: {best_val_mae:.4f} BPM")

    for epoch in range(1, BALANCED_FULL_TRANSFER_MAX_EPOCHS + 1):
        train_metrics = train_one_balanced_full_transfer_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
        )

        _, val_metrics = evaluate_hr_model(
            model=model,
            dataset=DATASETS["finetune_val"],
            loader=val_loader,
        )

        improved = val_metrics["mae_mean"] < (
            best_val_mae - BALANCED_FULL_TRANSFER_MIN_DELTA
        )

        if improved:
            best_val_mae = float(val_metrics["mae_mean"])
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history_rows.append(
            {
                "epoch": epoch,
                **train_metrics,
                "val_mae": float(val_metrics["mae_mean"]),
                "val_median": float(val_metrics["mae_median"]),
                "val_p90": float(val_metrics["mae_p90"]),
                "improved": bool(improved),
                "best_epoch_so_far": int(best_epoch),
                "best_val_mae_so_far": float(best_val_mae),
            }
        )

        print(
            f"Epoch {epoch:02d} | "
            f"train MAE {train_metrics['train_mae']:.4f} | "
            f"val MAE {val_metrics['mae_mean']:.4f} | "
            f"best {best_val_mae:.4f} @ epoch {best_epoch}"
        )

        if epochs_without_improvement >= BALANCED_FULL_TRANSFER_PATIENCE:
            print(f"Early stopping after {epoch} epochs.")
            break

    model.load_state_dict(best_state_dict)

    candidate_predictions, candidate_role_summary, candidate_dataset_role_summary = (
        evaluate_candidate_on_all_roles(model=model, datasets=DATASETS)
    )

    role_comparison = compare_candidate_to_frozen_by_role(candidate_role_summary)
    dataset_role_comparison = compare_candidate_to_frozen_by_dataset_role(
        candidate_dataset_role_summary
    )

    history = pd.DataFrame(history_rows)

    val_delta = get_role_delta(role_comparison, "finetune_val", "delta_mae_mean")
    test_delta = get_role_delta(role_comparison, "finetune_test", "delta_mae_mean")
    ood_delta = get_role_delta(role_comparison, "ood_eval", "delta_mae_mean")

    decision = (
        "candidate_for_analysis"
        if val_delta <= -0.20 and test_delta <= -0.10 and ood_delta <= 0.50
        else "research_result_only"
    )

    history_path = BALANCED_FULL_TRANSFER_DIR / "balanced_full_model_transfer_history.csv"
    comparison_path = BALANCED_FULL_TRANSFER_DIR / "balanced_full_model_transfer_role_comparison.csv"
    dataset_role_path = BALANCED_FULL_TRANSFER_DIR / "balanced_full_model_transfer_dataset_role_comparison.csv"
    checkpoint_path = BALANCED_FULL_TRANSFER_DIR / "balanced_full_model_transfer_best.pt"

    history.to_csv(history_path, index=False)
    role_comparison.to_csv(comparison_path, index=False)
    dataset_role_comparison.to_csv(dataset_role_path, index=False)

    torch.save(
        {
            "experiment": "NB_P2_11_balanced_full_model_transfer_sourcefps",
            "model_state": model.state_dict(),
            "model_config": FROZEN_MODEL_CONFIG,
            "best_epoch": best_epoch,
            "frozen_val_mae": float(frozen_val_mae),
            "best_val_mae": float(best_val_mae),
            "role_comparison": role_comparison.to_dict("records"),
            "dataset_role_comparison": dataset_role_comparison.to_dict("records"),
        },
        checkpoint_path,
    )

    result_summary = {
        "best_epoch": int(best_epoch),
        "frozen_val_mae": float(frozen_val_mae),
        "best_val_mae": float(best_val_mae),
        "val_delta_vs_frozen": float(val_delta),
        "test_delta_vs_frozen": float(test_delta),
        "ood_delta_vs_frozen": float(ood_delta),
        "decision": decision,
        "history_path": str(history_path),
        "comparison_path": str(comparison_path),
        "dataset_role_path": str(dataset_role_path),
        "checkpoint_path": str(checkpoint_path),
    }

    return {
        "model": model,
        "parameter_summary": parameter_summary,
        "sampler_preview": sampler_preview,
        "history": history,
        "candidate_predictions": candidate_predictions,
        "candidate_role_summary": candidate_role_summary,
        "candidate_dataset_role_summary": candidate_dataset_role_summary,
        "role_comparison": role_comparison,
        "dataset_role_comparison": dataset_role_comparison,
        "result_summary": result_summary,
    }


balanced_full_transfer_result = run_balanced_full_model_transfer_experiment()

BALANCED_FULL_TRANSFER_MODEL = balanced_full_transfer_result["model"]
BALANCED_FULL_TRANSFER_HISTORY = balanced_full_transfer_result["history"]
BALANCED_FULL_TRANSFER_ROLE_COMPARISON = balanced_full_transfer_result["role_comparison"]
BALANCED_FULL_TRANSFER_DATASET_ROLE_COMPARISON = balanced_full_transfer_result["dataset_role_comparison"]
BALANCED_FULL_TRANSFER_RESULT_SUMMARY = balanced_full_transfer_result["result_summary"]

print("\nBalanced sampler preview from first 5 batches:")
display(balanced_full_transfer_result["sampler_preview"])

print("\nBalanced full-model transfer parameter summary:")
display(balanced_full_transfer_result["parameter_summary"])

print("\nBalanced full-model transfer history:")
display(round_metric_table(BALANCED_FULL_TRANSFER_HISTORY))

print("\nBalanced full-model transfer vs frozen role comparison:")
display(round_metric_table(BALANCED_FULL_TRANSFER_ROLE_COMPARISON))

print("\nBalanced full-model transfer vs frozen dataset-role comparison:")
display(round_metric_table(BALANCED_FULL_TRANSFER_DATASET_ROLE_COMPARISON))

print("\nBalanced full-model transfer result summary:")
print(json.dumps(BALANCED_FULL_TRANSFER_RESULT_SUMMARY, indent=2))

print("\nDONE: Balanced full-model transfer-learning experiment completed.")

Frozen validation MAE baseline: 6.0275 BPM
Epoch 01 | train MAE 8.8339 | val MAE 6.0141 | best 6.0141 @ epoch 1
Epoch 02 | train MAE 8.9567 | val MAE 6.0741 | best 6.0141 @ epoch 1
Epoch 03 | train MAE 8.8502 | val MAE 5.9297 | best 5.9297 @ epoch 3
Epoch 04 | train MAE 8.7570 | val MAE 5.9451 | best 5.9297 @ epoch 3
Epoch 05 | train MAE 8.9454 | val MAE 5.8359 | best 5.8359 @ epoch 5
Epoch 06 | train MAE 8.6779 | val MAE 5.8636 | best 5.8359 @ epoch 5
Epoch 07 | train MAE 8.8219 | val MAE 5.8875 | best 5.8359 @ epoch 5
Epoch 08 | train MAE 8.8160 | val MAE 5.8652 | best 5.8359 @ epoch 5
Epoch 09 | train MAE 8.7697 | val MAE 5.8462 | best 5.8359 @ epoch 5
Epoch 10 | train MAE 8.7968 | val MAE 5.8932 | best 5.8359 @ epoch 5
Epoch 11 | train MAE 8.7788 | val MAE 5.8646 | best 5.8359 @ epoch 5
Early stopping after 11 epochs.

Balanced sampler preview from first 5 batches:


,batch,dataset,rows
0,0,mcd_rppg,84
1,0,ubfc_phys,89
2,0,ubfc_rppg,83
3,1,mcd_rppg,72
4,1,ubfc_phys,85
5,1,ubfc_rppg,99
6,2,mcd_rppg,91
7,2,ubfc_phys,85
8,2,ubfc_rppg,80
9,3,mcd_rppg,79



Balanced full-model transfer parameter summary:


,module,trainable,parameters
0,encoder,True,1696
1,freq_proj,True,47680
2,head,True,2625
3,transformer_layers,True,270144



Balanced full-model transfer history:


...


Balanced full-model transfer vs frozen role comparison:


,role,rows,frozen_mae_mean,frozen_mae_median,frozen_mae_p90,frozen_mae_p95,frozen_mae_max,candidate_mae_mean,candidate_mae_median,candidate_mae_p90,candidate_mae_p95,candidate_mae_max,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_mae_max
0,finetune_train,12503,5.8236,3.7437,13.7192,18.9426,54.9838,5.6815,3.6093,13.4336,18.4911,59.8977,-0.1422,-0.1344,-0.2856,-0.4515,4.9140
1,finetune_val,2839,6.0275,3.8533,14.0701,19.2319,50.2965,5.8359,3.7266,13.7003,18.5688,52.5472,-0.1916,-0.1267,-0.3699,-0.6631,2.2507
2,finetune_test,2828,5.7696,3.6274,13.6984,19.0461,72.2401,5.7187,3.5311,13.4866,19.2105,72.6248,-0.0509,-0.0963,-0.2118,0.1645,0.3847
3,ood_eval,882,14.0355,7.6838,38.1498,49.9899,89.9549,14.5699,7.7757,39.3819,50.9697,89.2260,0.5344,0.0919,1.2321,0.9799,-0.7290



Balanced full-model transfer vs frozen dataset-role comparison:


,dataset,window_role,rows,frozen_mae_mean,frozen_mae_median,frozen_mae_p90,target_hr_mean,candidate_mae_mean,candidate_mae_median,candidate_mae_p90,delta_mae_mean,delta_mae_median,delta_mae_p90
0,mcd_rppg,finetune_test,2094,4.8847,3.1502,10.9700,79.853302,4.8131,3.0935,10.7764,-0.0716,-0.0566,-0.1936
1,ubfc_phys,finetune_test,658,8.8584,5.6735,20.2862,77.420601,8.9021,5.7405,20.4269,0.0437,0.0670,0.1407
2,ubfc_rppg,finetune_test,76,3.4085,3.1453,6.0835,98.269302,3.1105,2.7118,6.3532,-0.2979,-0.4336,0.2697
3,mcd_rppg,finetune_train,7446,4.6876,3.1639,10.3722,79.064796,4.4661,3.0625,9.9697,-0.2216,-0.1014,-0.4025
4,ubfc_phys,finetune_train,4595,7.7819,5.4248,18.5928,80.305496,7.8180,5.3367,18.6172,0.0361,-0.0881,0.0244
5,ubfc_rppg,finetune_train,462,4.6548,3.1447,9.9189,98.449203,4.0194,2.9230,8.5612,-0.6354,-0.2216,-1.3577
6,mcd_rppg,finetune_val,1743,5.1325,3.3746,11.5368,79.108597,4.8694,3.3076,10.8467,-0.2631,-0.0671,-0.6902
7,ubfc_phys,finetune_val,986,7.7629,4.8461,18.2725,80.002403,7.7093,4.9503,18.3111,-0.0536,0.1042,0.0386
8,ubfc_rppg,finetune_val,110,4.6533,2.4438,12.3774,98.277199,4.3588,3.1201,9.7114,-0.2945,0.6763,-2.6660
9,ecg_fitness,ood_eval,882,14.0355,7.6838,38.1498,109.204803,14.5699,7.7757,39.3819,0.5344,0.0919,1.2321



Balanced full-model transfer result summary:
{
  "best_epoch": 5,
  "frozen_val_mae": 6.027502059936523,
  "best_val_mae": 5.835920333862305,
  "val_delta_vs_frozen": -0.19158172607421875,
  "test_delta_vs_frozen": -0.05088043212890625,
  "ood_delta_vs_frozen": 0.5343828201293945,
  "decision": "research_result_only",
  "history_path": "/kaggle/working/crvse_balanced_full_model_transfer_sourcefps/balanced_full_model_transfer_history.csv",
  "comparison_path": "/kaggle/working/crvse_balanced_full_model_transfer_sourcefps/balanced_full_model_transfer_role_comparison.csv",
  "dataset_role_path": "/kaggle/working/crvse_balanced_full_model_transfer_sourcefps/balanced_full_model_transfer_dataset_role_comparison.csv",
  "checkpoint_path": "/kaggle/working/crvse_balanced_full_model_transfer_sourcefps/balanced_full_model_transfer_best.pt"
}

DONE: Balanced full-model transfer-learning experiment completed.


In [6]:
NB11_EXPERIMENT_ROWS = [
    {
        "experiment": "frozen_sourcefps_baseline",
        "best_epoch": 0,
        "val_delta_mean": 0.0,
        "test_delta_mean": 0.0,
        "ood_delta_mean": 0.0,
        "val_delta_p90": 0.0,
        "test_delta_p90": 0.0,
        "ood_delta_p90": 0.0,
        "status": "baseline",
    },
    {
        "experiment": "full_model_transfer",
        "best_epoch": FULL_TRANSFER_RESULT_SUMMARY["best_epoch"],
        "val_delta_mean": FULL_TRANSFER_RESULT_SUMMARY["val_delta_vs_frozen"],
        "test_delta_mean": FULL_TRANSFER_RESULT_SUMMARY["test_delta_vs_frozen"],
        "ood_delta_mean": FULL_TRANSFER_RESULT_SUMMARY["ood_delta_vs_frozen"],
        "val_delta_p90": get_role_delta(FULL_TRANSFER_ROLE_COMPARISON, "finetune_val", "delta_mae_p90"),
        "test_delta_p90": get_role_delta(FULL_TRANSFER_ROLE_COMPARISON, "finetune_test", "delta_mae_p90"),
        "ood_delta_p90": get_role_delta(FULL_TRANSFER_ROLE_COMPARISON, "ood_eval", "delta_mae_p90"),
        "status": FULL_TRANSFER_RESULT_SUMMARY["decision"],
    },
    {
        "experiment": "balanced_full_model_transfer",
        "best_epoch": BALANCED_FULL_TRANSFER_RESULT_SUMMARY["best_epoch"],
        "val_delta_mean": BALANCED_FULL_TRANSFER_RESULT_SUMMARY["val_delta_vs_frozen"],
        "test_delta_mean": BALANCED_FULL_TRANSFER_RESULT_SUMMARY["test_delta_vs_frozen"],
        "ood_delta_mean": BALANCED_FULL_TRANSFER_RESULT_SUMMARY["ood_delta_vs_frozen"],
        "val_delta_p90": get_role_delta(BALANCED_FULL_TRANSFER_ROLE_COMPARISON, "finetune_val", "delta_mae_p90"),
        "test_delta_p90": get_role_delta(BALANCED_FULL_TRANSFER_ROLE_COMPARISON, "finetune_test", "delta_mae_p90"),
        "ood_delta_p90": get_role_delta(BALANCED_FULL_TRANSFER_ROLE_COMPARISON, "ood_eval", "delta_mae_p90"),
        "status": BALANCED_FULL_TRANSFER_RESULT_SUMMARY["decision"],
    },
]

nb11_scoreboard = pd.DataFrame(NB11_EXPERIMENT_ROWS)

nb11_scoreboard["in_distribution_mean_gain"] = -(
    nb11_scoreboard["val_delta_mean"] + nb11_scoreboard["test_delta_mean"]
) / 2.0

nb11_scoreboard["ood_penalty"] = nb11_scoreboard["ood_delta_mean"]

nb11_scoreboard["adoption_candidate"] = (
    (nb11_scoreboard["val_delta_mean"] <= -0.20)
    & (nb11_scoreboard["test_delta_mean"] <= -0.10)
    & (nb11_scoreboard["ood_delta_mean"] <= 0.50)
)

nb11_scoreboard = nb11_scoreboard.sort_values(
    ["adoption_candidate", "in_distribution_mean_gain"],
    ascending=[False, False],
).reset_index(drop=True)

nb11_current_conclusion = {
    "best_in_distribution_experiment": str(nb11_scoreboard.iloc[0]["experiment"]),
    "adopt_checkpoint_now": bool(nb11_scoreboard.iloc[0]["adoption_candidate"]),
    "interpretation": (
        "Full-model transfer learning improves validation and held-out test performance on "
        "live-compatible source-FPS tensors. Balanced sampling gives the strongest validation "
        "gain and slightly reduces ECG-Fitness degradation compared with natural full transfer, "
        "but ECG-Fitness OOD performance still worsens. This suggests useful app-distribution "
        "adaptation with unresolved high-HR/exercise domain-shift risk."
    ),
    "recommended_next_step": (
        "Run scratch PhysFormer training on the live-compatible tensors as the next experiment, "
        "then compare scratch training against full-transfer and balanced full-transfer results."
    ),
}

print("NB11 experiment scoreboard:")
display(round_metric_table(nb11_scoreboard))

print("\nNB11 current conclusion:")
print(json.dumps(nb11_current_conclusion, indent=2))

NB11 experiment scoreboard:


,experiment,best_epoch,val_delta_mean,test_delta_mean,ood_delta_mean,val_delta_p90,test_delta_p90,ood_delta_p90,status,in_distribution_mean_gain,ood_penalty,adoption_candidate
0,full_model_transfer,9,-0.1793,-0.1829,1.1031,-0.2900,-0.6465,2.0705,research_result_only,0.1811,1.1031,False
1,balanced_full_model_transfer,5,-0.1916,-0.0509,0.5344,-0.3699,-0.2118,1.2321,research_result_only,0.1212,0.5344,False
2,frozen_sourcefps_baseline,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,baseline,-0.0000,0.0000,False



NB11 current conclusion:
{
  "best_in_distribution_experiment": "full_model_transfer",
  "adopt_checkpoint_now": false,
  "interpretation": "Full-model transfer learning improves validation and held-out test performance on live-compatible source-FPS tensors. Balanced sampling gives the strongest validation gain and slightly reduces ECG-Fitness degradation compared with natural full transfer, but ECG-Fitness OOD performance still worsens. This suggests useful app-distribution adaptation with unresolved high-HR/exercise domain-shift risk.",
  "recommended_next_step": "Run scratch PhysFormer training on the live-compatible tensors as the next experiment, then compare scratch training against full-transfer and balanced full-transfer results."
}


## Scratch PhysFormer Training on Live-Compatible Tensors

### What is being tested

This section trains the same CRVSE PhysFormer architecture from random initialization using the live-compatible source-FPS tensors as the primary training distribution.

### Why this matters

The transfer-learning experiments showed that the original checkpoint can adapt to live-compatible tensors, but the improvement was modest and ECG-Fitness OOD behavior worsened.

Scratch training tests a different hypothesis:

```text
The old checkpoint may carry preprocessing-distribution assumptions that limit adaptation.

In [7]:
SCRATCH_BATCH_SIZE = 256
SCRATCH_MAX_EPOCHS = 60
SCRATCH_PATIENCE = 10
SCRATCH_MIN_DELTA = 1e-3
SCRATCH_LR = 3e-4
SCRATCH_WEIGHT_DECAY = 1e-4
SCRATCH_HUBER_DELTA = 6.5
SCRATCH_GRAD_CLIP = 5.0

SCRATCH_DIR = KAGGLE_WORKING / "crvse_scratch_physformer_sourcefps"
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)


def assert_scratch_training_prerequisites() -> None:
    """
    Check that the frozen baseline section has been executed.
    """
    required_names = [
        "DATASETS",
        "FROZEN_ROLE_SUMMARY",
        "FROZEN_DATASET_ROLE_SUMMARY",
        "FROZEN_MODEL_CONFIG",
        "evaluate_hr_model",
        "evaluate_candidate_on_all_roles",
        "compare_candidate_to_frozen_by_role",
        "compare_candidate_to_frozen_by_dataset_role",
        "get_role_delta",
        "round_metric_table",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run the frozen baseline reproduction cell before scratch training. "
            f"Missing names: {missing}"
        )


def build_scratch_physformer_model(model_config: dict) -> CRVSEPhysFormer:
    """
    Build a randomly initialized PhysFormer with the same app-compatible contract.

    The architecture matches the current checkpoint contract, but no pretrained
    weights are loaded.
    """
    torch.manual_seed(SEED)

    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(SEED)

    model = CRVSEPhysFormer(**model_config)
    model.to(DEVICE)

    for parameter in model.parameters():
        parameter.requires_grad = True

    return model


def make_scratch_training_loader(dataset: MaterializedTensorDataset) -> DataLoader:
    """
    Create a shuffled natural-distribution training DataLoader.
    """
    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=SCRATCH_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        generator=generator,
    )


def train_one_scratch_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
) -> dict:
    """
    Train the scratch model for one epoch.
    """
    model.train()

    total_loss = 0.0
    total_mae = 0.0
    total_rows = 0
    grad_norms = []

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        pred = model(batch_x)
        loss = loss_fn(pred, batch_y)
        mae = torch.mean(torch.abs(pred - batch_y))

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=SCRATCH_GRAD_CLIP,
        )

        optimizer.step()

        batch_size = int(batch_x.shape[0])
        total_loss += float(loss.item()) * batch_size
        total_mae += float(mae.item()) * batch_size
        total_rows += batch_size
        grad_norms.append(float(grad_norm.item()))

    return {
        "train_loss": total_loss / total_rows,
        "train_mae": total_mae / total_rows,
        "grad_norm_mean": float(np.mean(grad_norms)),
        "grad_norm_max": float(np.max(grad_norms)),
    }


def run_scratch_physformer_training_experiment() -> dict:
    """
    Train CRVSE PhysFormer from random initialization on live-compatible tensors.
    """
    assert_scratch_training_prerequisites()

    model = build_scratch_physformer_model(FROZEN_MODEL_CONFIG)
    parameter_summary = count_parameters_by_trainability(model)

    train_loader = make_scratch_training_loader(DATASETS["finetune_train"])
    val_loader = make_eval_loader(DATASETS["finetune_val"])

    optimizer = AdamW(
        model.parameters(),
        lr=SCRATCH_LR,
        weight_decay=SCRATCH_WEIGHT_DECAY,
    )

    loss_fn = nn.HuberLoss(delta=SCRATCH_HUBER_DELTA)

    frozen_val_mae = float(
        FROZEN_ROLE_SUMMARY.loc[
            FROZEN_ROLE_SUMMARY["role"] == "finetune_val",
            "mae_mean",
        ].iloc[0]
    )

    best_val_mae = float("inf")
    best_epoch = 0
    best_state_dict = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    history_rows = []

    print(f"Frozen validation MAE baseline: {frozen_val_mae:.4f} BPM")
    print("Training scratch PhysFormer from random initialization.")

    for epoch in range(1, SCRATCH_MAX_EPOCHS + 1):
        train_metrics = train_one_scratch_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
        )

        _, val_metrics = evaluate_hr_model(
            model=model,
            dataset=DATASETS["finetune_val"],
            loader=val_loader,
        )

        improved = val_metrics["mae_mean"] < (best_val_mae - SCRATCH_MIN_DELTA)

        if improved:
            best_val_mae = float(val_metrics["mae_mean"])
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history_rows.append(
            {
                "epoch": epoch,
                **train_metrics,
                "val_mae": float(val_metrics["mae_mean"]),
                "val_median": float(val_metrics["mae_median"]),
                "val_p90": float(val_metrics["mae_p90"]),
                "improved": bool(improved),
                "best_epoch_so_far": int(best_epoch),
                "best_val_mae_so_far": float(best_val_mae),
            }
        )

        print(
            f"Epoch {epoch:02d} | "
            f"train MAE {train_metrics['train_mae']:.4f} | "
            f"val MAE {val_metrics['mae_mean']:.4f} | "
            f"best {best_val_mae:.4f} @ epoch {best_epoch}"
        )

        if epochs_without_improvement >= SCRATCH_PATIENCE:
            print(f"Early stopping after {epoch} epochs.")
            break

    model.load_state_dict(best_state_dict)

    candidate_predictions, candidate_role_summary, candidate_dataset_role_summary = (
        evaluate_candidate_on_all_roles(model=model, datasets=DATASETS)
    )

    role_comparison = compare_candidate_to_frozen_by_role(candidate_role_summary)
    dataset_role_comparison = compare_candidate_to_frozen_by_dataset_role(
        candidate_dataset_role_summary
    )

    history = pd.DataFrame(history_rows)

    val_delta = get_role_delta(role_comparison, "finetune_val", "delta_mae_mean")
    test_delta = get_role_delta(role_comparison, "finetune_test", "delta_mae_mean")
    ood_delta = get_role_delta(role_comparison, "ood_eval", "delta_mae_mean")

    decision = (
        "candidate_for_analysis"
        if val_delta <= -0.20 and test_delta <= -0.10 and ood_delta <= 0.50
        else "research_result_only"
    )

    history_path = SCRATCH_DIR / "scratch_physformer_history.csv"
    comparison_path = SCRATCH_DIR / "scratch_physformer_role_comparison.csv"
    dataset_role_path = SCRATCH_DIR / "scratch_physformer_dataset_role_comparison.csv"
    checkpoint_path = SCRATCH_DIR / "scratch_physformer_best.pt"

    history.to_csv(history_path, index=False)
    role_comparison.to_csv(comparison_path, index=False)
    dataset_role_comparison.to_csv(dataset_role_path, index=False)

    torch.save(
        {
            "experiment": "NB_P2_11_scratch_physformer_sourcefps",
            "model_state": model.state_dict(),
            "model_config": FROZEN_MODEL_CONFIG,
            "best_epoch": best_epoch,
            "frozen_val_mae": float(frozen_val_mae),
            "best_val_mae": float(best_val_mae),
            "role_comparison": role_comparison.to_dict("records"),
            "dataset_role_comparison": dataset_role_comparison.to_dict("records"),
        },
        checkpoint_path,
    )

    result_summary = {
        "best_epoch": int(best_epoch),
        "frozen_val_mae": float(frozen_val_mae),
        "best_val_mae": float(best_val_mae),
        "val_delta_vs_frozen": float(val_delta),
        "test_delta_vs_frozen": float(test_delta),
        "ood_delta_vs_frozen": float(ood_delta),
        "decision": decision,
        "history_path": str(history_path),
        "comparison_path": str(comparison_path),
        "dataset_role_path": str(dataset_role_path),
        "checkpoint_path": str(checkpoint_path),
    }

    return {
        "model": model,
        "parameter_summary": parameter_summary,
        "history": history,
        "candidate_predictions": candidate_predictions,
        "candidate_role_summary": candidate_role_summary,
        "candidate_dataset_role_summary": candidate_dataset_role_summary,
        "role_comparison": role_comparison,
        "dataset_role_comparison": dataset_role_comparison,
        "result_summary": result_summary,
    }


scratch_result = run_scratch_physformer_training_experiment()

SCRATCH_MODEL = scratch_result["model"]
SCRATCH_HISTORY = scratch_result["history"]
SCRATCH_ROLE_COMPARISON = scratch_result["role_comparison"]
SCRATCH_DATASET_ROLE_COMPARISON = scratch_result["dataset_role_comparison"]
SCRATCH_RESULT_SUMMARY = scratch_result["result_summary"]

print("\nScratch PhysFormer parameter summary:")
display(scratch_result["parameter_summary"])

print("\nScratch PhysFormer training history:")
display(round_metric_table(SCRATCH_HISTORY))

print("\nScratch PhysFormer vs frozen role comparison:")
display(round_metric_table(SCRATCH_ROLE_COMPARISON))

print("\nScratch PhysFormer vs frozen dataset-role comparison:")
display(round_metric_table(SCRATCH_DATASET_ROLE_COMPARISON))

print("\nScratch PhysFormer result summary:")
print(json.dumps(SCRATCH_RESULT_SUMMARY, indent=2))

print("\nDONE: Scratch PhysFormer training experiment completed.")

Frozen validation MAE baseline: 6.0275 BPM
Training scratch PhysFormer from random initialization.
Epoch 01 | train MAE 68.2288 | val MAE 40.1617 | best 40.1617 @ epoch 1
Epoch 02 | train MAE 18.6061 | val MAE 11.4871 | best 11.4871 @ epoch 2
Epoch 03 | train MAE 12.3285 | val MAE 9.9201 | best 9.9201 @ epoch 3
Epoch 04 | train MAE 10.8000 | val MAE 8.5242 | best 8.5242 @ epoch 4
Epoch 05 | train MAE 10.3520 | val MAE 8.1529 | best 8.1529 @ epoch 5
Epoch 06 | train MAE 10.0701 | val MAE 7.8157 | best 7.8157 @ epoch 6
Epoch 07 | train MAE 9.8140 | val MAE 7.5943 | best 7.5943 @ epoch 7
Epoch 08 | train MAE 9.4745 | val MAE 7.3537 | best 7.3537 @ epoch 8
Epoch 09 | train MAE 9.5915 | val MAE 7.2305 | best 7.2305 @ epoch 9
Epoch 10 | train MAE 9.3050 | val MAE 7.0647 | best 7.0647 @ epoch 10
Epoch 11 | train MAE 9.2144 | val MAE 6.9318 | best 6.9318 @ epoch 11
Epoch 12 | train MAE 9.1137 | val MAE 7.1057 | best 6.9318 @ epoch 11
Epoch 13 | train MAE 9.1647 | val MAE 7.0117 | best 6.9318 @

,module,trainable,parameters
0,encoder,True,1696
1,freq_proj,True,47680
2,head,True,2625
3,transformer_layers,True,270144



Scratch PhysFormer training history:


,epoch,train_loss,train_mae,grad_norm_mean,grad_norm_max,val_mae,val_median,val_p90,improved,best_epoch_so_far,best_val_mae_so_far
0,1,422.3624,68.2288,252.6989,672.0522,40.1617,41.3984,56.4772,True,1,40.1617
1,2,101.3917,18.6061,467.0268,928.5376,11.4871,10.2909,24.4131,True,2,11.4871
2,3,61.4351,12.3285,115.6411,366.3273,9.9201,7.8474,20.6029,True,3,9.9201
3,4,51.8711,10.8000,126.3664,237.9992,8.5242,6.8295,18.0590,True,4,8.5242
4,5,49.0641,10.3520,135.7264,331.3086,8.1529,6.2976,17.6172,True,5,8.1529
5,6,47.3127,10.0701,129.0771,431.3731,7.8157,5.9889,17.1759,True,6,7.8157
6,7,45.7547,9.8140,152.4220,417.9286,7.5943,5.5932,16.9037,True,7,7.5943
7,8,43.7636,9.4745,106.5768,363.1324,7.3537,5.3339,16.4690,True,8,7.3537
8,9,44.4714,9.5915,205.2454,545.7905,7.2305,5.3595,16.1410,True,9,7.2305
9,10,42.6461,9.3050,133.9920,432.4835,7.0647,5.0231,16.2432,True,10,7.0647



Scratch PhysFormer vs frozen role comparison:


,role,rows,frozen_mae_mean,frozen_mae_median,frozen_mae_p90,frozen_mae_p95,frozen_mae_max,candidate_mae_mean,candidate_mae_median,candidate_mae_p90,candidate_mae_p95,candidate_mae_max,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_mae_max
0,finetune_train,12503,5.8236,3.7437,13.7192,18.9426,54.9838,6.1203,4.0203,14.4158,18.6208,53.2401,0.2967,0.2766,0.6966,-0.3218,-1.7437
1,finetune_val,2839,6.0275,3.8533,14.0701,19.2319,50.2965,6.6764,4.5436,15.6785,21.0263,48.8901,0.6489,0.6903,1.6083,1.7944,-1.4064
2,finetune_test,2828,5.7696,3.6274,13.6984,19.0461,72.2401,6.1079,4.1538,13.6359,18.2967,67.6719,0.3383,0.5264,-0.0625,-0.7493,-4.5682
3,ood_eval,882,14.0355,7.6838,38.1498,49.9899,89.9549,28.9395,26.4972,58.7236,66.3978,101.6132,14.9040,18.8134,20.5738,16.4079,11.6583



Scratch PhysFormer vs frozen dataset-role comparison:


,dataset,window_role,rows,frozen_mae_mean,frozen_mae_median,frozen_mae_p90,target_hr_mean,candidate_mae_mean,candidate_mae_median,candidate_mae_p90,delta_mae_mean,delta_mae_median,delta_mae_p90
0,mcd_rppg,finetune_test,2094,4.8847,3.1502,10.9700,79.853302,5.348200,3.5624,12.1332,0.4635,0.4122,1.1632
1,ubfc_phys,finetune_test,658,8.8584,5.6735,20.2862,77.420601,8.720100,6.3389,19.6536,-0.1384,0.6654,-0.6325
2,ubfc_rppg,finetune_test,76,3.4085,3.1453,6.0835,98.269302,4.424500,3.6241,9.2641,1.0160,0.4787,3.1805
3,mcd_rppg,finetune_train,7446,4.6876,3.1639,10.3722,79.064796,5.010800,3.3988,11.5245,0.3232,0.2348,1.1524
4,ubfc_phys,finetune_train,4595,7.7819,5.4248,18.5928,80.305496,8.011600,5.7633,18.3199,0.2296,0.3384,-0.2729
5,ubfc_rppg,finetune_train,462,4.6548,3.1447,9.9189,98.449203,5.192000,3.7708,12.1805,0.5371,0.6261,2.2616
6,mcd_rppg,finetune_val,1743,5.1325,3.3746,11.5368,79.108597,5.936500,4.1793,12.7745,0.8040,0.8046,1.2376
7,ubfc_phys,finetune_val,986,7.7629,4.8461,18.2725,80.002403,8.022800,5.3179,19.7981,0.2598,0.4717,1.5256
8,ubfc_rppg,finetune_val,110,4.6533,2.4438,12.3774,98.277199,6.331300,4.9826,13.6730,1.6780,2.5389,1.2957
9,ecg_fitness,ood_eval,882,14.0355,7.6838,38.1498,109.204803,28.939501,26.4972,58.7236,14.9040,18.8134,20.5738



Scratch PhysFormer result summary:
{
  "best_epoch": 35,
  "frozen_val_mae": 6.027502059936523,
  "best_val_mae": 6.676396369934082,
  "val_delta_vs_frozen": 0.6488943099975586,
  "test_delta_vs_frozen": 0.33829402923583984,
  "ood_delta_vs_frozen": 14.904014587402344,
  "decision": "research_result_only",
  "history_path": "/kaggle/working/crvse_scratch_physformer_sourcefps/scratch_physformer_history.csv",
  "comparison_path": "/kaggle/working/crvse_scratch_physformer_sourcefps/scratch_physformer_role_comparison.csv",
  "dataset_role_path": "/kaggle/working/crvse_scratch_physformer_sourcefps/scratch_physformer_dataset_role_comparison.csv",
  "checkpoint_path": "/kaggle/working/crvse_scratch_physformer_sourcefps/scratch_physformer_best.pt"
}

DONE: Scratch PhysFormer training experiment completed.


## NB11 Error Anatomy Across Candidate Models

### What is being tested

This section compares the frozen source-FPS baseline, full-model transfer, balanced full-model transfer, and scratch PhysFormer training across clinically and computationally meaningful subgroups.

### Why this matters

Aggregate MAE can hide where a model actually improved or failed.

This analysis checks error behavior by:

- dataset and role,
- label stability,
- target HR range,
- source FPS,
- available live buffer duration,
- HR range inside the label window.

In [8]:
ERROR_ANATOMY_DIR = KAGGLE_WORKING / "crvse_nb11_error_anatomy"
ERROR_ANATOMY_DIR.mkdir(parents=True, exist_ok=True)


def assert_error_anatomy_prerequisites() -> None:
    """
    Check that NB11 candidate model results are available.

    This analysis depends on previous NB11 experiment cells. It does not train
    anything and does not repeat imports.
    """
    required_names = [
        "DATASETS",
        "FROZEN_PREDICTIONS",
        "FULL_TRANSFER_MODEL",
        "BALANCED_FULL_TRANSFER_MODEL",
        "SCRATCH_MODEL",
        "evaluate_candidate_on_all_roles",
        "round_metric_table",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run the frozen, full-transfer, balanced-transfer, and scratch cells first. "
            f"Missing names: {missing}"
        )


def get_or_build_candidate_predictions(
    experiment_name: str,
    result_name: str,
    model_name: str,
) -> pd.DataFrame:
    """
    Return row-level candidate predictions.

    The function first uses an existing result dictionary if present. If not, it
    rebuilds predictions from the stored model object and DATASETS.
    """
    result = globals().get(result_name)

    if isinstance(result, dict) and "candidate_predictions" in result:
        predictions = result["candidate_predictions"].copy()
    else:
        model = globals().get(model_name)

        if model is None:
            raise NameError(
                f"Could not find {result_name}['candidate_predictions'] or {model_name}."
            )

        predictions, _, _ = evaluate_candidate_on_all_roles(
            model=model,
            datasets=DATASETS,
        )

    predictions["experiment"] = experiment_name
    return predictions


def build_nb11_prediction_long_table() -> pd.DataFrame:
    """
    Build one row-level table containing all NB11 candidate predictions.
    """
    frozen = FROZEN_PREDICTIONS.copy()
    frozen["experiment"] = "frozen_sourcefps_baseline"

    full_transfer = get_or_build_candidate_predictions(
        experiment_name="full_model_transfer",
        result_name="full_transfer_result",
        model_name="FULL_TRANSFER_MODEL",
    )

    balanced_transfer = get_or_build_candidate_predictions(
        experiment_name="balanced_full_model_transfer",
        result_name="balanced_full_transfer_result",
        model_name="BALANCED_FULL_TRANSFER_MODEL",
    )

    scratch = get_or_build_candidate_predictions(
        experiment_name="scratch_physformer",
        result_name="scratch_result",
        model_name="SCRATCH_MODEL",
    )

    table = pd.concat(
        [frozen, full_transfer, balanced_transfer, scratch],
        ignore_index=True,
    )

    table["target_hr"] = table["target_hr_from_tensor"].astype(float)
    table["pred_hr"] = table["notebook_model_hr"].astype(float)
    table["signed_error"] = table["pred_hr"] - table["target_hr"]
    table["abs_error"] = table["signed_error"].abs()

    return table


def add_error_anatomy_bins(table: pd.DataFrame) -> pd.DataFrame:
    """
    Add physiologically and computationally useful subgroup columns.
    """
    enriched = table.copy()

    enriched["target_hr_bin"] = pd.cut(
        enriched["target_hr"],
        bins=[0, 65, 85, 100, 180, np.inf],
        labels=[
            "low_<65",
            "normal_low_65_85",
            "normal_high_85_100",
            "tachy_or_exercise_100_180",
            "above_180",
        ],
        right=False,
    ).astype(str)

    if "target_hr_range" in enriched.columns:
        target_range = pd.to_numeric(enriched["target_hr_range"], errors="coerce")

        enriched["target_hr_range_bin"] = pd.cut(
            target_range,
            bins=[-np.inf, 10, 25, 50, np.inf],
            labels=[
                "stable_lt_10_bpm",
                "moderate_10_25_bpm",
                "unstable_25_50_bpm",
                "highly_unstable_ge_50_bpm",
            ],
            right=False,
        ).astype(str)
    else:
        enriched["target_hr_range_bin"] = "not_available"

    if "source_fps" in enriched.columns:
        source_fps = pd.to_numeric(enriched["source_fps"], errors="coerce")

        enriched["source_fps_bin"] = pd.cut(
            source_fps,
            bins=[0, 24.5, 29.0, 31.0, 36.0, np.inf],
            labels=[
                "very_low_or_24fps",
                "mid_25_29fps",
                "near_30fps",
                "near_35fps",
                "above_36fps",
            ],
            right=False,
        ).astype(str)
    else:
        enriched["source_fps_bin"] = "not_available"

    if "actual_buffer_seconds" in enriched.columns:
        buffer_seconds = pd.to_numeric(enriched["actual_buffer_seconds"], errors="coerce")

        enriched["buffer_seconds_bin"] = pd.cut(
            buffer_seconds,
            bins=[0, 8.5, 11.5, 12.5, np.inf],
            labels=[
                "short_start_buffer",
                "partial_8p5_to_11p5s",
                "near_12s",
                "longer_than_12p5s",
            ],
            right=False,
        ).astype(str)
    else:
        enriched["buffer_seconds_bin"] = "not_available"

    if "label_stability_bucket" not in enriched.columns:
        enriched["label_stability_bucket"] = "not_available"

    return enriched


def summarize_error_groups(
    table: pd.DataFrame,
    group_columns: list[str],
) -> pd.DataFrame:
    """
    Summarize prediction error for each experiment and subgroup.
    """
    summary = (
        table.groupby(["experiment", *group_columns], dropna=False)
        .agg(
            rows=("abs_error", "size"),
            target_hr_mean=("target_hr", "mean"),
            pred_hr_mean=("pred_hr", "mean"),
            signed_error_mean=("signed_error", "mean"),
            signed_error_median=("signed_error", "median"),
            abs_error_mean=("abs_error", "mean"),
            abs_error_median=("abs_error", "median"),
            abs_error_p90=("abs_error", lambda s: float(s.quantile(0.90))),
            abs_error_p95=("abs_error", lambda s: float(s.quantile(0.95))),
        )
        .reset_index()
    )

    return summary


def compare_error_groups_to_frozen(
    summary: pd.DataFrame,
    group_columns: list[str],
) -> pd.DataFrame:
    """
    Compare each candidate subgroup against the frozen baseline subgroup.
    """
    frozen = summary[
        summary["experiment"] == "frozen_sourcefps_baseline"
    ].copy()

    candidates = summary[
        summary["experiment"] != "frozen_sourcefps_baseline"
    ].copy()

    frozen = frozen.rename(
        columns={
            "pred_hr_mean": "frozen_pred_hr_mean",
            "signed_error_mean": "frozen_signed_error_mean",
            "signed_error_median": "frozen_signed_error_median",
            "abs_error_mean": "frozen_abs_error_mean",
            "abs_error_median": "frozen_abs_error_median",
            "abs_error_p90": "frozen_abs_error_p90",
            "abs_error_p95": "frozen_abs_error_p95",
        }
    )

    keep_frozen_columns = [
        *group_columns,
        "rows",
        "target_hr_mean",
        "frozen_pred_hr_mean",
        "frozen_signed_error_mean",
        "frozen_signed_error_median",
        "frozen_abs_error_mean",
        "frozen_abs_error_median",
        "frozen_abs_error_p90",
        "frozen_abs_error_p95",
    ]

    comparison = candidates.merge(
        frozen[keep_frozen_columns],
        on=[*group_columns, "rows", "target_hr_mean"],
        how="inner",
    )

    comparison["delta_signed_error_mean"] = (
        comparison["signed_error_mean"] - comparison["frozen_signed_error_mean"]
    )
    comparison["delta_abs_error_mean"] = (
        comparison["abs_error_mean"] - comparison["frozen_abs_error_mean"]
    )
    comparison["delta_abs_error_median"] = (
        comparison["abs_error_median"] - comparison["frozen_abs_error_median"]
    )
    comparison["delta_abs_error_p90"] = (
        comparison["abs_error_p90"] - comparison["frozen_abs_error_p90"]
    )
    comparison["delta_abs_error_p95"] = (
        comparison["abs_error_p95"] - comparison["frozen_abs_error_p95"]
    )

    return comparison.sort_values(
        ["experiment", "delta_abs_error_mean"],
        ascending=[True, True],
    )


def build_error_anatomy_table(
    table: pd.DataFrame,
    group_columns: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build raw subgroup summary and frozen-comparison table.
    """
    summary = summarize_error_groups(table, group_columns)
    comparison = compare_error_groups_to_frozen(summary, group_columns)

    return summary, comparison


def display_error_anatomy_result(
    title: str,
    comparison: pd.DataFrame,
    sort_by: str = "delta_abs_error_mean",
) -> None:
    """
    Display one comparison table with useful columns first.
    """
    display_columns = [
        "experiment",
        *[
            column
            for column in comparison.columns
            if column
            not in {
                "experiment",
                "target_hr_mean",
                "pred_hr_mean",
                "signed_error_mean",
                "signed_error_median",
                "abs_error_mean",
                "abs_error_median",
                "abs_error_p90",
                "abs_error_p95",
                "frozen_pred_hr_mean",
                "frozen_signed_error_mean",
                "frozen_signed_error_median",
                "frozen_abs_error_mean",
                "frozen_abs_error_median",
                "frozen_abs_error_p90",
                "frozen_abs_error_p95",
                "delta_signed_error_mean",
                "delta_abs_error_mean",
                "delta_abs_error_median",
                "delta_abs_error_p90",
                "delta_abs_error_p95",
            }
        ],
        "target_hr_mean",
        "signed_error_mean",
        "frozen_signed_error_mean",
        "delta_signed_error_mean",
        "abs_error_mean",
        "frozen_abs_error_mean",
        "delta_abs_error_mean",
        "abs_error_p90",
        "frozen_abs_error_p90",
        "delta_abs_error_p90",
    ]

    display_columns = [
        column for column in display_columns if column in comparison.columns
    ]

    print(f"\n{title}")
    display(
        round_metric_table(
            comparison.sort_values(["experiment", sort_by])[display_columns]
        )
    )


def collect_top_error_changes(
    anatomy_tables: dict[str, dict[str, pd.DataFrame]],
    n: int = 12,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Collect the largest improvements and harms across all anatomy views.
    """
    rows = []

    for view_name, tables in anatomy_tables.items():
        comparison = tables["comparison"].copy()
        comparison["view"] = view_name

        group_columns = [
            column
            for column in comparison.columns
            if column
            not in {
                "experiment",
                "rows",
                "target_hr_mean",
                "pred_hr_mean",
                "signed_error_mean",
                "signed_error_median",
                "abs_error_mean",
                "abs_error_median",
                "abs_error_p90",
                "abs_error_p95",
                "frozen_pred_hr_mean",
                "frozen_signed_error_mean",
                "frozen_signed_error_median",
                "frozen_abs_error_mean",
                "frozen_abs_error_median",
                "frozen_abs_error_p90",
                "frozen_abs_error_p95",
                "delta_signed_error_mean",
                "delta_abs_error_mean",
                "delta_abs_error_median",
                "delta_abs_error_p90",
                "delta_abs_error_p95",
                "view",
            }
        ]

        comparison["subgroup"] = comparison[group_columns].astype(str).agg(" | ".join, axis=1)

        rows.append(
            comparison[
                [
                    "view",
                    "experiment",
                    "subgroup",
                    "rows",
                    "target_hr_mean",
                    "delta_signed_error_mean",
                    "delta_abs_error_mean",
                    "delta_abs_error_p90",
                ]
            ]
        )

    all_changes = pd.concat(rows, ignore_index=True)

    biggest_improvements = all_changes.sort_values(
        "delta_abs_error_mean",
        ascending=True,
    ).head(n)

    biggest_harms = all_changes.sort_values(
        "delta_abs_error_mean",
        ascending=False,
    ).head(n)

    return biggest_improvements, biggest_harms


def run_nb11_error_anatomy() -> dict:
    """
    Run the first NB11 follow-up analysis: error anatomy.
    """
    assert_error_anatomy_prerequisites()

    predictions_long = build_nb11_prediction_long_table()
    predictions_long = add_error_anatomy_bins(predictions_long)

    analysis_specs = {
        "dataset_role": ["dataset", "window_role"],
        "label_stability": ["label_stability_bucket"],
        "target_hr_bin": ["target_hr_bin"],
        "source_fps_bin": ["source_fps_bin"],
        "buffer_seconds_bin": ["buffer_seconds_bin"],
        "target_hr_range_bin": ["target_hr_range_bin"],
    }

    anatomy_tables = {}

    for view_name, group_columns in analysis_specs.items():
        summary, comparison = build_error_anatomy_table(
            table=predictions_long,
            group_columns=group_columns,
        )

        anatomy_tables[view_name] = {
            "summary": summary,
            "comparison": comparison,
        }

        summary.to_csv(
            ERROR_ANATOMY_DIR / f"{view_name}_summary.csv",
            index=False,
        )

        comparison.to_csv(
            ERROR_ANATOMY_DIR / f"{view_name}_vs_frozen.csv",
            index=False,
        )

    biggest_improvements, biggest_harms = collect_top_error_changes(
        anatomy_tables=anatomy_tables,
        n=12,
    )

    biggest_improvements.to_csv(
        ERROR_ANATOMY_DIR / "biggest_subgroup_improvements.csv",
        index=False,
    )

    biggest_harms.to_csv(
        ERROR_ANATOMY_DIR / "biggest_subgroup_harms.csv",
        index=False,
    )

    report = {
        "rows": int(len(predictions_long)),
        "experiments": sorted(predictions_long["experiment"].unique().tolist()),
        "unique_original_windows": int(len(predictions_long) / predictions_long["experiment"].nunique()),
        "analysis_views": list(analysis_specs.keys()),
        "artifact_dir": str(ERROR_ANATOMY_DIR),
    }

    return {
        "predictions_long": predictions_long,
        "anatomy_tables": anatomy_tables,
        "biggest_improvements": biggest_improvements,
        "biggest_harms": biggest_harms,
        "report": report,
    }


nb11_error_anatomy = run_nb11_error_anatomy()

NB11_ERROR_ANATOMY_TABLES = nb11_error_anatomy["anatomy_tables"]
NB11_ERROR_ANATOMY_REPORT = nb11_error_anatomy["report"]

print("NB11 error anatomy report:")
print(json.dumps(NB11_ERROR_ANATOMY_REPORT, indent=2))

display_error_anatomy_result(
    title="Error anatomy by dataset and role:",
    comparison=NB11_ERROR_ANATOMY_TABLES["dataset_role"]["comparison"],
)

display_error_anatomy_result(
    title="Error anatomy by label stability:",
    comparison=NB11_ERROR_ANATOMY_TABLES["label_stability"]["comparison"],
)

display_error_anatomy_result(
    title="Error anatomy by target HR bin:",
    comparison=NB11_ERROR_ANATOMY_TABLES["target_hr_bin"]["comparison"],
)

display_error_anatomy_result(
    title="Error anatomy by source FPS bin:",
    comparison=NB11_ERROR_ANATOMY_TABLES["source_fps_bin"]["comparison"],
)

display_error_anatomy_result(
    title="Error anatomy by live buffer duration bin:",
    comparison=NB11_ERROR_ANATOMY_TABLES["buffer_seconds_bin"]["comparison"],
)

display_error_anatomy_result(
    title="Error anatomy by target HR range bin:",
    comparison=NB11_ERROR_ANATOMY_TABLES["target_hr_range_bin"]["comparison"],
)

print("\nLargest subgroup improvements versus frozen:")
display(round_metric_table(nb11_error_anatomy["biggest_improvements"]))

print("\nLargest subgroup harms versus frozen:")
display(round_metric_table(nb11_error_anatomy["biggest_harms"]))

print("\nDONE: NB11 error anatomy analysis completed.")

NB11 error anatomy report:
{
  "rows": 76208,
  "experiments": [
    "balanced_full_model_transfer",
    "frozen_sourcefps_baseline",
    "full_model_transfer",
    "scratch_physformer"
  ],
  "unique_original_windows": 19052,
  "analysis_views": [
    "dataset_role",
    "label_stability",
    "target_hr_bin",
    "source_fps_bin",
    "buffer_seconds_bin",
    "target_hr_range_bin"
  ],
  "artifact_dir": "/kaggle/working/crvse_nb11_error_anatomy"
}

Error anatomy by dataset and role:


,experiment,dataset,window_role,rows,target_hr_mean,signed_error_mean,frozen_signed_error_mean,delta_signed_error_mean,abs_error_mean,frozen_abs_error_mean,delta_abs_error_mean,abs_error_p90,frozen_abs_error_p90,delta_abs_error_p90
8,balanced_full_model_transfer,ubfc_rppg,finetune_train,462,98.4492,0.3059,-1.0723,1.3782,4.0194,4.6548,-0.6354,8.5612,9.9189,-1.3577
7,balanced_full_model_transfer,ubfc_rppg,finetune_test,76,98.2693,1.2490,-0.4102,1.6592,3.1105,3.4085,-0.2979,6.3532,6.0835,0.2697
9,balanced_full_model_transfer,ubfc_rppg,finetune_val,110,98.2772,-0.7876,-2.5125,1.7249,4.3588,4.6533,-0.2945,9.7114,12.3774,-2.6660
3,balanced_full_model_transfer,mcd_rppg,finetune_val,1743,79.1086,0.5980,1.2656,-0.6676,4.8694,5.1325,-0.2631,10.8467,11.5368,-0.6902
2,balanced_full_model_transfer,mcd_rppg,finetune_train,7446,79.0648,0.7018,1.2645,-0.5627,4.4661,4.6876,-0.2216,9.9697,10.3722,-0.4025
1,balanced_full_model_transfer,mcd_rppg,finetune_test,2094,79.8533,0.5036,0.9242,-0.4206,4.8131,4.8847,-0.0716,10.7764,10.9700,-0.1936
6,balanced_full_model_transfer,ubfc_phys,finetune_val,986,80.0024,2.8177,2.2826,0.5352,7.7093,7.7629,-0.0536,18.3111,18.2725,0.0386
5,balanced_full_model_transfer,ubfc_phys,finetune_train,4595,80.3055,2.9429,2.3192,0.6237,7.8180,7.7819,0.0361,18.6172,18.5928,0.0244
4,balanced_full_model_transfer,ubfc_phys,finetune_test,658,77.4205,3.5080,3.1980,0.3100,8.9021,8.8584,0.0437,20.4269,20.2862,0.1407
0,balanced_full_model_transfer,ecg_fitness,ood_eval,882,109.2048,-9.3535,-7.9487,-1.4048,14.5699,14.0355,0.5344,39.3819,38.1498,1.2321



Error anatomy by label stability:


,experiment,label_stability_bucket,rows,target_hr_mean,signed_error_mean,frozen_signed_error_mean,delta_signed_error_mean,abs_error_mean,frozen_abs_error_mean,delta_abs_error_mean,abs_error_p90,frozen_abs_error_p90,delta_abs_error_p90
2,balanced_full_model_transfer,stable_lt_10_bpm,6508,82.9041,-0.0329,0.3207,-0.3536,5.1732,5.3579,-0.1847,10.9188,11.5533,-0.6345
1,balanced_full_model_transfer,moderate_10_25_bpm,5986,82.7221,-0.2219,0.1651,-0.3870,5.0048,5.1251,-0.1203,11.0655,11.3930,-0.3274
3,balanced_full_model_transfer,unstable_25_50_bpm,2234,78.4900,1.2479,1.1127,0.1351,6.2695,6.3772,-0.1076,14.6763,15.1192,-0.4430
0,balanced_full_model_transfer,highly_unstable_ge_50_bpm,4324,79.2471,3.8357,3.5137,0.3221,9.0182,8.9792,0.0390,20.2623,20.1165,0.1458
4,full_model_transfer,highly_unstable_ge_50_bpm,4324,79.2471,3.1706,3.5137,-0.3430,8.7976,8.9792,-0.1816,19.3274,20.1165,-0.7891
5,full_model_transfer,moderate_10_25_bpm,5986,82.7221,-0.0426,0.1651,-0.2078,4.9478,5.1251,-0.1773,10.7781,11.3930,-0.6148
6,full_model_transfer,stable_lt_10_bpm,6508,82.9041,-0.0947,0.3207,-0.4155,5.1972,5.3579,-0.1606,10.8570,11.5533,-0.6963
7,full_model_transfer,unstable_25_50_bpm,2234,78.4900,1.0106,1.1127,-0.1021,6.2245,6.3772,-0.1526,14.4557,15.1192,-0.6636
8,scratch_physformer,highly_unstable_ge_50_bpm,4324,79.2471,1.3375,3.5137,-2.1761,9.2692,8.9792,0.2901,19.8857,20.1165,-0.2309
11,scratch_physformer,unstable_25_50_bpm,2234,78.4900,-0.2373,1.1127,-1.3500,6.9098,6.3772,0.5326,15.9776,15.1192,0.8584



Error anatomy by target HR bin:


...


Largest subgroup improvements versus frozen:


,view,experiment,subgroup,rows,target_hr_mean,delta_signed_error_mean,delta_abs_error_mean,delta_abs_error_p90
0,dataset_role,balanced_full_model_transfer,ubfc_rppg | finetune_train,462,98.4492,1.3782,-0.6354,-1.3577
50,target_hr_bin,scratch_physformer,low_<65,2501,59.8242,-0.8299,-0.5232,-1.2761
46,target_hr_bin,full_model_transfer,low_<65,2501,59.8242,-0.4465,-0.4575,-1.1334
10,dataset_role,full_model_transfer,mcd_rppg | finetune_train,7446,79.0648,-0.2040,-0.3289,-0.7602
1,dataset_role,balanced_full_model_transfer,ubfc_rppg | finetune_test,76,98.2693,1.6592,-0.2979,0.2697
2,dataset_role,balanced_full_model_transfer,ubfc_rppg | finetune_val,110,98.2772,1.7249,-0.2945,-2.6660
3,dataset_role,balanced_full_model_transfer,mcd_rppg | finetune_val,1743,79.1086,-0.6676,-0.2631,-0.6902
11,dataset_role,full_model_transfer,mcd_rppg | finetune_val,1743,79.1086,-0.4441,-0.2549,-0.4848
47,target_hr_bin,full_model_transfer,normal_low_65_85,9793,75.7035,0.0016,-0.2482,-0.4797
4,dataset_role,balanced_full_model_transfer,mcd_rppg | finetune_train,7446,79.0648,-0.5627,-0.2216,-0.4025



Largest subgroup harms versus frozen:


,view,experiment,subgroup,rows,target_hr_mean,delta_signed_error_mean,delta_abs_error_mean,delta_abs_error_p90
29,dataset_role,scratch_physformer,ecg_fitness | ood_eval,882,109.2048,-19.2639,14.9040,20.5738
53,target_hr_bin,scratch_physformer,tachy_or_exercise_100_180,2125,111.5762,-9.0270,7.2458,20.3222
71,buffer_seconds_bin,scratch_physformer,short_start_buffer,364,86.8839,-4.6745,1.9966,6.4011
41,label_stability,scratch_physformer,stable_lt_10_bpm,6508,82.9041,-3.1849,1.8575,4.3377
65,source_fps_bin,scratch_physformer,near_30fps,8502,83.1547,-3.1297,1.7653,4.4080
28,dataset_role,scratch_physformer,ubfc_rppg | finetune_val,110,98.2772,0.3517,1.6780,1.2957
52,target_hr_bin,scratch_physformer,normal_high_85_100,4633,91.6558,-2.4632,1.3075,3.1934
64,source_fps_bin,scratch_physformer,mid_25_29fps,32,75.1992,-0.1544,1.2332,2.1981
19,dataset_role,full_model_transfer,ecg_fitness | ood_eval,882,109.2048,-2.3475,1.1031,2.0705
74,target_hr_range_bin,scratch_physformer,not_available,19052,81.4994,-2.3580,1.0316,1.8584



DONE: NB11 error anatomy analysis completed.


## NB11 Prediction Bias Analysis

### What is being tested

This section analyzes signed error and calibration behavior for the frozen baseline and NB11 candidate models.

### Why this matters

MAE shows how large the error is, but it does not show direction.

For heart-rate estimation, direction matters:

```text
positive signed error = model overestimates HR
negative signed error = model underestimates HR

In [9]:
BIAS_ANALYSIS_DIR = KAGGLE_WORKING / "crvse_nb11_bias_analysis"
BIAS_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)


def assert_bias_analysis_prerequisites() -> None:
    """
    Check that the NB11 error anatomy analysis has been run.
    """
    required_names = [
        "KAGGLE_WORKING",
        "round_metric_table",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run the NB11 setup and baseline cells before this analysis. "
            f"Missing names: {missing}"
        )

    if "nb11_error_anatomy" not in globals() and "build_nb11_prediction_long_table" not in globals():
        raise NameError(
            "Run the NB11 error anatomy cell first, or provide the prediction-long table."
        )


def get_prediction_long_table_for_bias() -> pd.DataFrame:
    """
    Get the long prediction table from error anatomy, or rebuild it if needed.
    """
    if "nb11_error_anatomy" in globals():
        result = globals()["nb11_error_anatomy"]

        if isinstance(result, dict) and "predictions_long" in result:
            table = result["predictions_long"].copy()
        else:
            table = None
    else:
        table = None

    if table is None:
        table = build_nb11_prediction_long_table()
        table = add_error_anatomy_bins(table)

    required_columns = [
        "experiment",
        "dataset",
        "window_role",
        "target_hr",
        "pred_hr",
        "signed_error",
        "abs_error",
    ]

    missing = [column for column in required_columns if column not in table.columns]

    if missing:
        raise ValueError(
            "Prediction-long table is missing required columns: "
            f"{missing}"
        )

    return table


def ensure_bias_bins(table: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure the prediction table has HR bins needed for bias analysis.
    """
    enriched = table.copy()

    if "target_hr_bin" not in enriched.columns:
        enriched["target_hr_bin"] = pd.cut(
            enriched["target_hr"],
            bins=[0, 65, 85, 100, 180, np.inf],
            labels=[
                "low_<65",
                "normal_low_65_85",
                "normal_high_85_100",
                "tachy_or_exercise_100_180",
                "above_180",
            ],
            right=False,
        ).astype(str)

    enriched["high_hr_flag"] = np.where(
        enriched["target_hr"] >= 100.0,
        "target_hr_ge_100",
        "target_hr_lt_100",
    )

    enriched["signed_error_direction"] = np.select(
        [
            enriched["signed_error"] <= -10.0,
            enriched["signed_error"] < 0.0,
            enriched["signed_error"] == 0.0,
            enriched["signed_error"] > 10.0,
            enriched["signed_error"] > 0.0,
        ],
        [
            "under_by_ge_10",
            "under_by_lt_10",
            "exact",
            "over_by_ge_10",
            "over_by_lt_10",
        ],
        default="unknown",
    )

    return enriched


def summarize_bias_groups(
    table: pd.DataFrame,
    group_columns: list[str],
) -> pd.DataFrame:
    """
    Summarize signed error, absolute error, and under/overestimation rates.
    """
    grouping = ["experiment", *group_columns]

    summary = (
        table.groupby(grouping, dropna=False)
        .agg(
            rows=("signed_error", "size"),
            target_hr_mean=("target_hr", "mean"),
            pred_hr_mean=("pred_hr", "mean"),
            signed_error_mean=("signed_error", "mean"),
            signed_error_median=("signed_error", "median"),
            signed_error_p10=("signed_error", lambda s: float(s.quantile(0.10))),
            signed_error_p90=("signed_error", lambda s: float(s.quantile(0.90))),
            abs_error_mean=("abs_error", "mean"),
            abs_error_median=("abs_error", "median"),
            abs_error_p90=("abs_error", lambda s: float(s.quantile(0.90))),
            underprediction_rate=("signed_error", lambda s: float((s < 0).mean())),
            severe_underprediction_rate=("signed_error", lambda s: float((s <= -10).mean())),
            severe_overprediction_rate=("signed_error", lambda s: float((s >= 10).mean())),
        )
        .reset_index()
    )

    return summary


def compare_bias_groups_to_frozen(
    summary: pd.DataFrame,
    group_columns: list[str],
) -> pd.DataFrame:
    """
    Compare candidate bias summaries against frozen baseline summaries.
    """
    frozen = summary[
        summary["experiment"] == "frozen_sourcefps_baseline"
    ].copy()

    candidates = summary[
        summary["experiment"] != "frozen_sourcefps_baseline"
    ].copy()

    frozen = frozen.rename(
        columns={
            "pred_hr_mean": "frozen_pred_hr_mean",
            "signed_error_mean": "frozen_signed_error_mean",
            "signed_error_median": "frozen_signed_error_median",
            "signed_error_p10": "frozen_signed_error_p10",
            "signed_error_p90": "frozen_signed_error_p90",
            "abs_error_mean": "frozen_abs_error_mean",
            "abs_error_median": "frozen_abs_error_median",
            "abs_error_p90": "frozen_abs_error_p90",
            "underprediction_rate": "frozen_underprediction_rate",
            "severe_underprediction_rate": "frozen_severe_underprediction_rate",
            "severe_overprediction_rate": "frozen_severe_overprediction_rate",
        }
    )

    keep_columns = [
        *group_columns,
        "rows",
        "target_hr_mean",
        "frozen_pred_hr_mean",
        "frozen_signed_error_mean",
        "frozen_signed_error_median",
        "frozen_signed_error_p10",
        "frozen_signed_error_p90",
        "frozen_abs_error_mean",
        "frozen_abs_error_median",
        "frozen_abs_error_p90",
        "frozen_underprediction_rate",
        "frozen_severe_underprediction_rate",
        "frozen_severe_overprediction_rate",
    ]

    comparison = candidates.merge(
        frozen[keep_columns],
        on=[*group_columns, "rows", "target_hr_mean"],
        how="inner",
    )

    comparison["delta_signed_error_mean"] = (
        comparison["signed_error_mean"] - comparison["frozen_signed_error_mean"]
    )
    comparison["delta_abs_error_mean"] = (
        comparison["abs_error_mean"] - comparison["frozen_abs_error_mean"]
    )
    comparison["delta_abs_error_p90"] = (
        comparison["abs_error_p90"] - comparison["frozen_abs_error_p90"]
    )
    comparison["delta_underprediction_rate"] = (
        comparison["underprediction_rate"] - comparison["frozen_underprediction_rate"]
    )
    comparison["delta_severe_underprediction_rate"] = (
        comparison["severe_underprediction_rate"]
        - comparison["frozen_severe_underprediction_rate"]
    )
    comparison["delta_severe_overprediction_rate"] = (
        comparison["severe_overprediction_rate"]
        - comparison["frozen_severe_overprediction_rate"]
    )

    return comparison


def compute_calibration_metrics(
    table: pd.DataFrame,
    group_columns: list[str],
) -> pd.DataFrame:
    """
    Compute simple linear calibration metrics.

    A perfect model would have:
    - pred_vs_target_slope close to 1
    - pred_vs_target_intercept close to 0
    - signed_error_vs_target_slope close to 0
    """
    rows = []
    grouping = ["experiment", *group_columns]

    for keys, group in table.groupby(grouping, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        key_values = dict(zip(grouping, keys))

        target = group["target_hr"].to_numpy(dtype=float)
        pred = group["pred_hr"].to_numpy(dtype=float)
        signed_error = pred - target

        if len(group) < 3 or np.std(target) < 1e-6 or np.std(pred) < 1e-6:
            pred_slope = np.nan
            pred_intercept = np.nan
            error_slope = np.nan
            corr = np.nan
        else:
            pred_slope, pred_intercept = np.polyfit(target, pred, 1)
            error_slope, _ = np.polyfit(target, signed_error, 1)
            corr = float(np.corrcoef(target, pred)[0, 1])

        rows.append(
            {
                **key_values,
                "rows": int(len(group)),
                "target_hr_mean": float(np.mean(target)),
                "pred_hr_mean": float(np.mean(pred)),
                "signed_error_mean": float(np.mean(signed_error)),
                "pred_vs_target_slope": float(pred_slope),
                "pred_vs_target_intercept": float(pred_intercept),
                "signed_error_vs_target_slope": float(error_slope),
                "corr_pred_target": float(corr),
            }
        )

    return pd.DataFrame(rows)


def display_bias_comparison(
    title: str,
    comparison: pd.DataFrame,
    group_columns: list[str],
) -> None:
    """
    Display a compact candidate-vs-frozen bias comparison.
    """
    columns = [
        "experiment",
        *group_columns,
        "rows",
        "target_hr_mean",
        "signed_error_mean",
        "frozen_signed_error_mean",
        "delta_signed_error_mean",
        "abs_error_mean",
        "frozen_abs_error_mean",
        "delta_abs_error_mean",
        "underprediction_rate",
        "frozen_underprediction_rate",
        "delta_underprediction_rate",
        "severe_underprediction_rate",
        "frozen_severe_underprediction_rate",
        "delta_severe_underprediction_rate",
    ]

    columns = [column for column in columns if column in comparison.columns]

    print(f"\n{title}")
    display(
        round_metric_table(
            comparison[columns].sort_values(
                ["experiment", "delta_signed_error_mean"]
            )
        )
    )


def run_nb11_bias_analysis() -> dict:
    """
    Run NB11 signed-error and calibration analysis.
    """
    assert_bias_analysis_prerequisites()

    predictions = get_prediction_long_table_for_bias()
    predictions = ensure_bias_bins(predictions)

    overall_summary = summarize_bias_groups(predictions, [])
    target_bin_summary = summarize_bias_groups(predictions, ["target_hr_bin"])
    high_hr_summary = summarize_bias_groups(predictions, ["high_hr_flag"])
    dataset_role_summary = summarize_bias_groups(predictions, ["dataset", "window_role"])

    target_bin_comparison = compare_bias_groups_to_frozen(
        target_bin_summary,
        ["target_hr_bin"],
    )
    high_hr_comparison = compare_bias_groups_to_frozen(
        high_hr_summary,
        ["high_hr_flag"],
    )
    dataset_role_comparison = compare_bias_groups_to_frozen(
        dataset_role_summary,
        ["dataset", "window_role"],
    )

    overall_calibration = compute_calibration_metrics(predictions, [])
    dataset_calibration = compute_calibration_metrics(predictions, ["dataset"])
    role_calibration = compute_calibration_metrics(predictions, ["window_role"])

    overall_summary.to_csv(BIAS_ANALYSIS_DIR / "overall_bias_summary.csv", index=False)
    target_bin_comparison.to_csv(BIAS_ANALYSIS_DIR / "target_hr_bin_bias_vs_frozen.csv", index=False)
    high_hr_comparison.to_csv(BIAS_ANALYSIS_DIR / "high_hr_bias_vs_frozen.csv", index=False)
    dataset_role_comparison.to_csv(BIAS_ANALYSIS_DIR / "dataset_role_bias_vs_frozen.csv", index=False)
    overall_calibration.to_csv(BIAS_ANALYSIS_DIR / "overall_calibration.csv", index=False)
    dataset_calibration.to_csv(BIAS_ANALYSIS_DIR / "dataset_calibration.csv", index=False)
    role_calibration.to_csv(BIAS_ANALYSIS_DIR / "role_calibration.csv", index=False)

    report = {
        "rows": int(len(predictions)),
        "experiments": sorted(predictions["experiment"].unique().tolist()),
        "artifact_dir": str(BIAS_ANALYSIS_DIR),
        "main_question": (
            "Does transfer improve aggregate MAE by increasing high-HR underprediction?"
        ),
    }

    return {
        "predictions": predictions,
        "overall_summary": overall_summary,
        "target_bin_comparison": target_bin_comparison,
        "high_hr_comparison": high_hr_comparison,
        "dataset_role_comparison": dataset_role_comparison,
        "overall_calibration": overall_calibration,
        "dataset_calibration": dataset_calibration,
        "role_calibration": role_calibration,
        "report": report,
    }


nb11_bias_analysis = run_nb11_bias_analysis()

NB11_BIAS_ANALYSIS = nb11_bias_analysis

print("NB11 bias analysis report:")
print(json.dumps(nb11_bias_analysis["report"], indent=2))

print("\nOverall signed-error summary:")
display(round_metric_table(nb11_bias_analysis["overall_summary"]))

print("\nOverall calibration metrics:")
display(round_metric_table(nb11_bias_analysis["overall_calibration"]))

display_bias_comparison(
    title="Bias comparison by target HR bin:",
    comparison=nb11_bias_analysis["target_bin_comparison"],
    group_columns=["target_hr_bin"],
)

display_bias_comparison(
    title="Bias comparison by high-HR flag:",
    comparison=nb11_bias_analysis["high_hr_comparison"],
    group_columns=["high_hr_flag"],
)

display_bias_comparison(
    title="Bias comparison by dataset and role:",
    comparison=nb11_bias_analysis["dataset_role_comparison"],
    group_columns=["dataset", "window_role"],
)

print("\nCalibration by dataset:")
display(round_metric_table(nb11_bias_analysis["dataset_calibration"]))

print("\nCalibration by role:")
display(round_metric_table(nb11_bias_analysis["role_calibration"]))

print("\nDONE: NB11 prediction bias analysis completed.")

NB11 bias analysis report:
{
  "rows": 76208,
  "experiments": [
    "balanced_full_model_transfer",
    "frozen_sourcefps_baseline",
    "full_model_transfer",
    "scratch_physformer"
  ],
  "artifact_dir": "/kaggle/working/crvse_nb11_bias_analysis",
  "main_question": "Does transfer improve aggregate MAE by increasing high-HR underprediction?"
}

Overall signed-error summary:


,experiment,rows,target_hr_mean,pred_hr_mean,signed_error_mean,signed_error_median,signed_error_p10,signed_error_p90,abs_error_mean,abs_error_median,abs_error_p90,underprediction_rate,severe_underprediction_rate,severe_overprediction_rate
0,balanced_full_model_transfer,19052,81.4994,82.4353,0.9359,0.8925,-7.8305,10.2960,6.1215,3.7307,14.3231,0.4255,0.0722,0.1042
1,frozen_sourcefps_baseline,19052,81.4994,82.5887,1.0894,0.8437,-7.7609,10.7918,6.2261,3.8553,14.5286,0.4343,0.0711,0.1111
2,full_model_transfer,19052,81.4994,82.2917,0.7923,0.8582,-7.8922,9.9594,6.0565,3.7019,14.1501,0.4316,0.0736,0.0995
3,scratch_physformer,19052,81.4994,80.2307,-1.2687,-0.2723,-12.1216,9.4550,7.2577,4.3592,16.3870,0.5192,0.1304,0.0910



Overall calibration metrics:


,experiment,rows,target_hr_mean,pred_hr_mean,signed_error_mean,pred_vs_target_slope,pred_vs_target_intercept,signed_error_vs_target_slope,corr_pred_target
0,balanced_full_model_transfer,19052,81.4994,82.4353,0.9359,0.6806,26.9683,-0.3194,0.7905
1,frozen_sourcefps_baseline,19052,81.4994,82.5887,1.0894,0.6931,26.1002,-0.3069,0.7914
2,full_model_transfer,19052,81.4994,82.2917,0.7923,0.6759,27.2101,-0.3241,0.7943
3,scratch_physformer,19052,81.4994,80.2307,-1.2687,0.5192,37.9131,-0.4808,0.6736



Bias comparison by target HR bin:


...


Calibration by dataset:


,experiment,dataset,rows,target_hr_mean,pred_hr_mean,signed_error_mean,pred_vs_target_slope,pred_vs_target_intercept,signed_error_vs_target_slope,corr_pred_target
0,balanced_full_model_transfer,ecg_fitness,882,109.2048,99.8513,-9.3535,0.6175,32.4156,-0.3825,0.6200
1,balanced_full_model_transfer,mcd_rppg,11283,79.2179,79.8669,0.6490,0.7774,18.2824,-0.2226,0.8646
2,balanced_full_model_transfer,ubfc_phys,6239,79.9533,82.9360,2.9827,0.4211,49.2693,-0.5789,0.5407
3,balanced_full_model_transfer,ubfc_rppg,648,98.3989,98.6298,0.2309,0.9604,4.1321,-0.0396,0.9485
4,frozen_sourcefps_baseline,ecg_fitness,882,109.2048,101.2561,-7.9487,0.6353,31.8742,-0.3647,0.6314
5,frozen_sourcefps_baseline,mcd_rppg,11283,79.2179,80.4195,1.2015,0.8054,16.6138,-0.1946,0.8581
6,frozen_sourcefps_baseline,ubfc_phys,6239,79.9533,82.3594,2.4061,0.4110,49.5005,-0.5890,0.5344
7,frozen_sourcefps_baseline,ubfc_rppg,648,98.3989,97.1598,-1.2391,0.9170,6.9251,-0.0830,0.9320
8,full_model_transfer,ecg_fitness,882,109.2048,98.9086,-10.2962,0.6097,32.3313,-0.3903,0.6154
9,full_model_transfer,mcd_rppg,11283,79.2179,80.1910,0.9731,0.7995,16.8573,-0.2005,0.8718



Calibration by role:


,experiment,window_role,rows,target_hr_mean,pred_hr_mean,signed_error_mean,pred_vs_target_slope,pred_vs_target_intercept,signed_error_vs_target_slope,corr_pred_target
0,balanced_full_model_transfer,finetune_test,2828,79.7822,81.0048,1.2227,0.7128,24.1364,-0.2872,0.7713
1,balanced_full_model_transfer,finetune_train,12503,80.2370,81.7479,1.5108,0.6977,25.7701,-0.3023,0.7972
2,balanced_full_model_transfer,finetune_val,2839,80.1617,81.4770,1.3152,0.6974,25.5747,-0.3026,0.8002
3,balanced_full_model_transfer,ood_eval,882,109.2048,99.8513,-9.3535,0.6175,32.4156,-0.3825,0.6200
4,frozen_sourcefps_baseline,finetune_test,2828,79.7822,81.1995,1.4174,0.7378,22.3394,-0.2622,0.7767
5,frozen_sourcefps_baseline,finetune_train,12503,80.2370,81.8028,1.5658,0.7002,25.6192,-0.2998,0.7922
6,frozen_sourcefps_baseline,finetune_val,2839,80.1617,81.6342,1.4724,0.6948,25.9399,-0.3052,0.7915
7,frozen_sourcefps_baseline,ood_eval,882,109.2048,101.2561,-7.9487,0.6353,31.8742,-0.3647,0.6314
8,full_model_transfer,finetune_test,2828,79.7822,81.0775,1.2953,0.7221,23.4650,-0.2779,0.7816
9,full_model_transfer,finetune_train,12503,80.2370,81.6073,1.3702,0.6997,25.4688,-0.3003,0.8076



DONE: NB11 prediction bias analysis completed.


## NB11 Transfer Candidate Decision Analysis

### What is being tested

This section compares full-model transfer and balanced full-model transfer as candidate app checkpoints.

### Why this matters

The previous analyses show a trade-off:

```text
Full transfer      -> stronger in-distribution gain, larger ECG-Fitness / high-HR penalty
Balanced transfer  -> weaker in-distribution gain, smaller ECG-Fitness / high-HR penalty
```
This section makes that trade-off explicit using validation, test, p90, OOD, high-HR bias, and severe-underprediction metrics.

In [10]:
TRANSFER_DECISION_DIR = KAGGLE_WORKING / "crvse_nb11_transfer_decision"
TRANSFER_DECISION_DIR.mkdir(parents=True, exist_ok=True)


def assert_transfer_decision_prerequisites() -> None:
    """
    Check that the transfer experiments and bias analysis are available.
    """
    required_names = [
        "FULL_TRANSFER_ROLE_COMPARISON",
        "BALANCED_FULL_TRANSFER_ROLE_COMPARISON",
        "FULL_TRANSFER_DATASET_ROLE_COMPARISON",
        "BALANCED_FULL_TRANSFER_DATASET_ROLE_COMPARISON",
        "FULL_TRANSFER_RESULT_SUMMARY",
        "BALANCED_FULL_TRANSFER_RESULT_SUMMARY",
        "NB11_BIAS_ANALYSIS",
        "get_role_delta",
        "round_metric_table",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run transfer experiments and NB11 bias analysis before this cell. "
            f"Missing names: {missing}"
        )


def extract_bias_metric(
    comparison: pd.DataFrame,
    experiment: str,
    group_column: str,
    group_value: str,
    metric_column: str,
) -> float:
    """
    Extract one metric from a bias comparison table.
    """
    matching = comparison.loc[
        (comparison["experiment"] == experiment)
        & (comparison[group_column] == group_value),
        metric_column,
    ]

    if len(matching) != 1:
        raise ValueError(
            f"Expected one row for {experiment=} {group_column=} {group_value=}, "
            f"found {len(matching)}."
        )

    return float(matching.iloc[0])


def extract_dataset_role_delta(
    comparison: pd.DataFrame,
    dataset: str,
    window_role: str,
    metric_column: str,
) -> float:
    """
    Extract one dataset-role delta metric.
    """
    matching = comparison.loc[
        (comparison["dataset"] == dataset)
        & (comparison["window_role"] == window_role),
        metric_column,
    ]

    if len(matching) != 1:
        raise ValueError(
            f"Expected one row for {dataset=} {window_role=}, found {len(matching)}."
        )

    return float(matching.iloc[0])


def build_transfer_decision_row(
    experiment: str,
    role_comparison: pd.DataFrame,
    dataset_role_comparison: pd.DataFrame,
    result_summary: dict,
) -> dict:
    """
    Build one decision-analysis row for a transfer candidate.
    """
    high_hr_bias = NB11_BIAS_ANALYSIS["high_hr_comparison"]
    target_bin_bias = NB11_BIAS_ANALYSIS["target_bin_comparison"]

    val_delta_mean = get_role_delta(role_comparison, "finetune_val", "delta_mae_mean")
    test_delta_mean = get_role_delta(role_comparison, "finetune_test", "delta_mae_mean")
    ood_delta_mean = get_role_delta(role_comparison, "ood_eval", "delta_mae_mean")

    val_delta_p90 = get_role_delta(role_comparison, "finetune_val", "delta_mae_p90")
    test_delta_p90 = get_role_delta(role_comparison, "finetune_test", "delta_mae_p90")
    ood_delta_p90 = get_role_delta(role_comparison, "ood_eval", "delta_mae_p90")

    high_hr_delta_abs = extract_bias_metric(
        comparison=high_hr_bias,
        experiment=experiment,
        group_column="high_hr_flag",
        group_value="target_hr_ge_100",
        metric_column="delta_abs_error_mean",
    )

    high_hr_delta_signed = extract_bias_metric(
        comparison=high_hr_bias,
        experiment=experiment,
        group_column="high_hr_flag",
        group_value="target_hr_ge_100",
        metric_column="delta_signed_error_mean",
    )

    high_hr_delta_severe_under = extract_bias_metric(
        comparison=high_hr_bias,
        experiment=experiment,
        group_column="high_hr_flag",
        group_value="target_hr_ge_100",
        metric_column="delta_severe_underprediction_rate",
    )

    non_high_hr_delta_abs = extract_bias_metric(
        comparison=high_hr_bias,
        experiment=experiment,
        group_column="high_hr_flag",
        group_value="target_hr_lt_100",
        metric_column="delta_abs_error_mean",
    )

    tachy_delta_abs = extract_bias_metric(
        comparison=target_bin_bias,
        experiment=experiment,
        group_column="target_hr_bin",
        group_value="tachy_or_exercise_100_180",
        metric_column="delta_abs_error_mean",
    )

    mcd_test_delta = extract_dataset_role_delta(
        comparison=dataset_role_comparison,
        dataset="mcd_rppg",
        window_role="finetune_test",
        metric_column="delta_mae_mean",
    )

    ubfc_phys_test_delta = extract_dataset_role_delta(
        comparison=dataset_role_comparison,
        dataset="ubfc_phys",
        window_role="finetune_test",
        metric_column="delta_mae_mean",
    )

    ubfc_rppg_test_delta = extract_dataset_role_delta(
        comparison=dataset_role_comparison,
        dataset="ubfc_rppg",
        window_role="finetune_test",
        metric_column="delta_mae_mean",
    )

    in_distribution_gain = -0.5 * (val_delta_mean + test_delta_mean)
    p90_gain = -0.5 * (val_delta_p90 + test_delta_p90)

    ood_penalty = max(0.0, ood_delta_mean)
    high_hr_penalty = max(0.0, high_hr_delta_abs)
    severe_under_penalty = max(0.0, high_hr_delta_severe_under) * 10.0

    risk_adjusted_score = (
        in_distribution_gain
        + 0.25 * p90_gain
        - 0.50 * ood_penalty
        - 0.50 * high_hr_penalty
        - severe_under_penalty
    )

    return {
        "experiment": experiment,
        "best_epoch": int(result_summary["best_epoch"]),
        "val_delta_mean": val_delta_mean,
        "test_delta_mean": test_delta_mean,
        "ood_delta_mean": ood_delta_mean,
        "val_delta_p90": val_delta_p90,
        "test_delta_p90": test_delta_p90,
        "ood_delta_p90": ood_delta_p90,
        "high_hr_delta_abs": high_hr_delta_abs,
        "high_hr_delta_signed": high_hr_delta_signed,
        "high_hr_delta_severe_under_rate": high_hr_delta_severe_under,
        "non_high_hr_delta_abs": non_high_hr_delta_abs,
        "tachy_delta_abs": tachy_delta_abs,
        "mcd_test_delta_mean": mcd_test_delta,
        "ubfc_phys_test_delta_mean": ubfc_phys_test_delta,
        "ubfc_rppg_test_delta_mean": ubfc_rppg_test_delta,
        "in_distribution_gain": in_distribution_gain,
        "p90_gain": p90_gain,
        "risk_adjusted_score": risk_adjusted_score,
    }


def assign_transfer_candidate_profile(row: pd.Series) -> str:
    """
    Assign a human-readable candidate profile.
    """
    useful_indistribution = (
        row["val_delta_mean"] < 0.0
        and row["test_delta_mean"] < 0.0
    )

    ood_acceptable_for_analysis = row["ood_delta_mean"] <= 0.75
    ood_problematic = row["ood_delta_mean"] > 1.0
    high_hr_problematic = row["high_hr_delta_abs"] > 0.50

    if useful_indistribution and ood_acceptable_for_analysis and not high_hr_problematic:
        return "balanced_analysis_candidate"

    if useful_indistribution and ood_problematic:
        return "performance_candidate_with_ood_penalty"

    if useful_indistribution:
        return "weak_analysis_candidate"

    return "reject"


def run_transfer_candidate_decision_analysis() -> dict:
    """
    Compare full-transfer and balanced-transfer candidates using decision metrics.
    """
    assert_transfer_decision_prerequisites()

    rows = [
        build_transfer_decision_row(
            experiment="full_model_transfer",
            role_comparison=FULL_TRANSFER_ROLE_COMPARISON,
            dataset_role_comparison=FULL_TRANSFER_DATASET_ROLE_COMPARISON,
            result_summary=FULL_TRANSFER_RESULT_SUMMARY,
        ),
        build_transfer_decision_row(
            experiment="balanced_full_model_transfer",
            role_comparison=BALANCED_FULL_TRANSFER_ROLE_COMPARISON,
            dataset_role_comparison=BALANCED_FULL_TRANSFER_DATASET_ROLE_COMPARISON,
            result_summary=BALANCED_FULL_TRANSFER_RESULT_SUMMARY,
        ),
    ]

    decision_table = pd.DataFrame(rows)
    decision_table["candidate_profile"] = decision_table.apply(
        assign_transfer_candidate_profile,
        axis=1,
    )

    decision_table = decision_table.sort_values(
        ["risk_adjusted_score", "in_distribution_gain"],
        ascending=[False, False],
    ).reset_index(drop=True)

    best_risk_adjusted = str(decision_table.iloc[0]["experiment"])

    if "balanced_full_model_transfer" in decision_table["experiment"].tolist():
        balanced_row = decision_table.loc[
            decision_table["experiment"] == "balanced_full_model_transfer"
        ].iloc[0]
    else:
        balanced_row = None

    if "full_model_transfer" in decision_table["experiment"].tolist():
        full_row = decision_table.loc[
            decision_table["experiment"] == "full_model_transfer"
        ].iloc[0]
    else:
        full_row = None

    interpretation = {
        "best_risk_adjusted_candidate": best_risk_adjusted,
        "adopt_checkpoint_now": False,
        "why_not_adopt": (
            "Both transfer candidates improve in-distribution error, but both worsen "
            "high-HR / ECG-Fitness behavior. This remains a research result rather "
            "than an app-checkpoint adoption result."
        ),
        "full_transfer_tradeoff": (
            "Full transfer gives stronger normal/app-like performance gain, but it "
            "has the larger OOD and high-HR underprediction penalty."
        ),
        "balanced_transfer_tradeoff": (
            "Balanced transfer gives weaker in-distribution gain, but it is the more "
            "conservative candidate because OOD degradation is smaller."
        ),
        "recommended_next_experiment": (
            "Run prediction blending first. It is cheaper and safer than checkpoint "
            "interpolation and directly tests whether a conservative mix can preserve "
            "transfer gains while reducing high-HR/OOD harm."
        ),
    }

    decision_table.to_csv(
        TRANSFER_DECISION_DIR / "transfer_candidate_decision_table.csv",
        index=False,
    )

    with (TRANSFER_DECISION_DIR / "transfer_candidate_decision_interpretation.json").open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(interpretation, file, indent=2)

    return {
        "decision_table": decision_table,
        "interpretation": interpretation,
        "artifact_dir": str(TRANSFER_DECISION_DIR),
    }


transfer_candidate_decision = run_transfer_candidate_decision_analysis()

NB11_TRANSFER_DECISION = transfer_candidate_decision

print("NB11 transfer candidate decision table:")
display(round_metric_table(transfer_candidate_decision["decision_table"]))

print("\nNB11 transfer candidate interpretation:")
print(json.dumps(transfer_candidate_decision["interpretation"], indent=2))

print("\nDONE: NB11 transfer candidate decision analysis completed.")

NB11 transfer candidate decision table:


,experiment,best_epoch,val_delta_mean,test_delta_mean,ood_delta_mean,val_delta_p90,test_delta_p90,ood_delta_p90,high_hr_delta_abs,high_hr_delta_signed,high_hr_delta_severe_under_rate,non_high_hr_delta_abs,tachy_delta_abs,mcd_test_delta_mean,ubfc_phys_test_delta_mean,ubfc_rppg_test_delta_mean,in_distribution_gain,p90_gain,risk_adjusted_score,candidate_profile
0,balanced_full_model_transfer,5,-0.1916,-0.0509,0.5344,-0.3699,-0.2118,1.2321,0.3710,-0.4720,0.0118,-0.1644,0.3710,-0.0716,0.0437,-0.2979,0.1212,0.2908,-0.3764,balanced_analysis_candidate
1,full_model_transfer,9,-0.1793,-0.1829,1.1031,-0.2900,-0.6465,2.0705,0.4769,-0.9796,0.0212,-0.2509,0.4769,-0.1930,-0.1692,-0.0211,0.1811,0.4683,-0.7036,performance_candidate_with_ood_penalty



NB11 transfer candidate interpretation:
{
  "best_risk_adjusted_candidate": "balanced_full_model_transfer",
  "adopt_checkpoint_now": false,
  "why_not_adopt": "Both transfer candidates improve in-distribution error, but both worsen high-HR / ECG-Fitness behavior. This remains a research result rather than an app-checkpoint adoption result.",
  "full_transfer_tradeoff": "Full transfer gives stronger normal/app-like performance gain, but it has the larger OOD and high-HR underprediction penalty.",
  "balanced_transfer_tradeoff": "Balanced transfer gives weaker in-distribution gain, but it is the more conservative candidate because OOD degradation is smaller.",
  "recommended_next_experiment": "Run prediction blending first. It is cheaper and safer than checkpoint interpolation and directly tests whether a conservative mix can preserve transfer gains while reducing high-HR/OOD harm."
}

DONE: NB11 transfer candidate decision analysis completed.


## NB11 Prediction Blending Analysis

### What is being tested

This section blends frozen baseline predictions with transfer-model predictions.

The blended prediction is:

```text
blended_hr = (1 - alpha) * frozen_hr + alpha * transfer_hr

```

where `alpha` controls how much of the transfer model is used.

### Why this matters

Transfer models improved live-compatible validation and test performance, but worsened high-HR / ECG-Fitness behavior.
Prediction blending tests whether conservative adaptation can preserve some in-distribution improvement while reducing OOD harm.

In [11]:
PREDICTION_BLEND_DIR = KAGGLE_WORKING / "crvse_nb11_prediction_blending"
PREDICTION_BLEND_DIR.mkdir(parents=True, exist_ok=True)

BLEND_ALPHAS = [0.10, 0.25, 0.40, 0.50, 0.60, 0.75, 0.90]


def assert_prediction_blending_prerequisites() -> None:
    """
    Check that row-level predictions and decision-analysis objects are available.
    """
    required_names = [
        "FROZEN_PREDICTIONS",
        "FULL_TRANSFER_MODEL",
        "BALANCED_FULL_TRANSFER_MODEL",
        "DATASETS",
        "evaluate_candidate_on_all_roles",
        "round_metric_table",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run frozen baseline and transfer experiment cells before blending. "
            f"Missing names: {missing}"
        )


def get_blending_prediction_source(
    experiment_name: str,
    result_name: str,
    model_name: str,
) -> pd.DataFrame:
    """
    Get row-level predictions for one transfer candidate.
    """
    result = globals().get(result_name)

    if isinstance(result, dict) and "candidate_predictions" in result:
        predictions = result["candidate_predictions"].copy()
    else:
        model = globals().get(model_name)

        if model is None:
            raise NameError(f"Could not find predictions or model for {experiment_name}.")

        predictions, _, _ = evaluate_candidate_on_all_roles(
            model=model,
            datasets=DATASETS,
        )

    predictions["source_experiment"] = experiment_name
    return predictions


def make_prediction_blend_table(
    candidate_predictions: pd.DataFrame,
    candidate_name: str,
    alphas: list[float],
) -> pd.DataFrame:
    """
    Build blended prediction rows for one candidate and alpha grid.
    """
    frozen = FROZEN_PREDICTIONS.copy().reset_index(drop=True)
    candidate = candidate_predictions.copy().reset_index(drop=True)

    if len(frozen) != len(candidate):
        raise ValueError(
            f"Frozen and candidate prediction row counts differ: "
            f"{len(frozen)} vs {len(candidate)}."
        )

    identity_columns = [
        "dataset",
        "subject_id",
        "recording_id",
        "window_role",
        "start_frame",
        "end_frame",
    ]

    for column in identity_columns:
        if column in frozen.columns and column in candidate.columns:
            frozen_values = frozen[column].astype(str).to_numpy()
            candidate_values = candidate[column].astype(str).to_numpy()

            if not np.array_equal(frozen_values, candidate_values):
                raise AssertionError(
                    f"Prediction rows are not aligned for column {column!r}. "
                    "Do not blend predictions until row order is verified."
                )

    rows = []

    for alpha in alphas:
        blend = frozen.copy()

        frozen_pred = frozen["notebook_model_hr"].astype(float).to_numpy()
        candidate_pred = candidate["notebook_model_hr"].astype(float).to_numpy()
        target = frozen["target_hr_from_tensor"].astype(float).to_numpy()

        blended_pred = (1.0 - alpha) * frozen_pred + alpha * candidate_pred

        blend["candidate_name"] = candidate_name
        blend["alpha"] = float(alpha)
        blend["frozen_hr"] = frozen_pred
        blend["candidate_hr"] = candidate_pred
        blend["blended_hr"] = blended_pred
        blend["target_hr"] = target
        blend["signed_error"] = blended_pred - target
        blend["abs_error"] = np.abs(blend["signed_error"])

        rows.append(blend)

    return pd.concat(rows, ignore_index=True)


def summarize_blended_predictions_by_role(blend_table: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize blended prediction performance by candidate, alpha, and role.
    """
    summary = (
        blend_table.groupby(["candidate_name", "alpha", "window_role"], dropna=False)
        .agg(
            rows=("abs_error", "size"),
            mae_mean=("abs_error", "mean"),
            mae_median=("abs_error", "median"),
            mae_p90=("abs_error", lambda s: float(s.quantile(0.90))),
            mae_p95=("abs_error", lambda s: float(s.quantile(0.95))),
            signed_error_mean=("signed_error", "mean"),
            severe_underprediction_rate=("signed_error", lambda s: float((s <= -10).mean())),
        )
        .reset_index()
        .rename(columns={"window_role": "role"})
    )

    return summary


def summarize_blended_predictions_by_high_hr(blend_table: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize blended predictions by high-HR flag.
    """
    table = blend_table.copy()
    table["high_hr_flag"] = np.where(
        table["target_hr"] >= 100.0,
        "target_hr_ge_100",
        "target_hr_lt_100",
    )

    summary = (
        table.groupby(["candidate_name", "alpha", "high_hr_flag"], dropna=False)
        .agg(
            rows=("abs_error", "size"),
            mae_mean=("abs_error", "mean"),
            mae_p90=("abs_error", lambda s: float(s.quantile(0.90))),
            signed_error_mean=("signed_error", "mean"),
            severe_underprediction_rate=("signed_error", lambda s: float((s <= -10).mean())),
        )
        .reset_index()
    )

    return summary


def extract_blend_role_metric(
    summary: pd.DataFrame,
    candidate_name: str,
    alpha: float,
    role: str,
    metric: str,
) -> float:
    """
    Extract one metric from blended role summary.
    """
    matching = summary.loc[
        (summary["candidate_name"] == candidate_name)
        & np.isclose(summary["alpha"], alpha)
        & (summary["role"] == role),
        metric,
    ]

    if len(matching) != 1:
        raise ValueError(
            f"Expected one row for {candidate_name=} {alpha=} {role=} {metric=}, "
            f"found {len(matching)}."
        )

    return float(matching.iloc[0])


def extract_blend_high_hr_metric(
    summary: pd.DataFrame,
    candidate_name: str,
    alpha: float,
    high_hr_flag: str,
    metric: str,
) -> float:
    """
    Extract one metric from blended high-HR summary.
    """
    matching = summary.loc[
        (summary["candidate_name"] == candidate_name)
        & np.isclose(summary["alpha"], alpha)
        & (summary["high_hr_flag"] == high_hr_flag),
        metric,
    ]

    if len(matching) != 1:
        raise ValueError(
            f"Expected one row for {candidate_name=} {alpha=} {high_hr_flag=} {metric=}, "
            f"found {len(matching)}."
        )

    return float(matching.iloc[0])


def build_prediction_blend_decision_table(
    role_summary: pd.DataFrame,
    high_hr_summary: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build decision metrics for all candidate/alpha blends.
    """
    frozen_role = FROZEN_ROLE_SUMMARY.set_index("role")

    rows = []

    for candidate_name in sorted(role_summary["candidate_name"].unique()):
        for alpha in sorted(role_summary.loc[role_summary["candidate_name"] == candidate_name, "alpha"].unique()):
            val_mae = extract_blend_role_metric(
                role_summary,
                candidate_name,
                alpha,
                "finetune_val",
                "mae_mean",
            )
            test_mae = extract_blend_role_metric(
                role_summary,
                candidate_name,
                alpha,
                "finetune_test",
                "mae_mean",
            )
            ood_mae = extract_blend_role_metric(
                role_summary,
                candidate_name,
                alpha,
                "ood_eval",
                "mae_mean",
            )

            val_p90 = extract_blend_role_metric(
                role_summary,
                candidate_name,
                alpha,
                "finetune_val",
                "mae_p90",
            )
            test_p90 = extract_blend_role_metric(
                role_summary,
                candidate_name,
                alpha,
                "finetune_test",
                "mae_p90",
            )
            ood_p90 = extract_blend_role_metric(
                role_summary,
                candidate_name,
                alpha,
                "ood_eval",
                "mae_p90",
            )

            high_hr_mae = extract_blend_high_hr_metric(
                high_hr_summary,
                candidate_name,
                alpha,
                "target_hr_ge_100",
                "mae_mean",
            )

            high_hr_signed_error = extract_blend_high_hr_metric(
                high_hr_summary,
                candidate_name,
                alpha,
                "target_hr_ge_100",
                "signed_error_mean",
            )

            high_hr_severe_under = extract_blend_high_hr_metric(
                high_hr_summary,
                candidate_name,
                alpha,
                "target_hr_ge_100",
                "severe_underprediction_rate",
            )

            val_delta = val_mae - float(frozen_role.loc["finetune_val", "mae_mean"])
            test_delta = test_mae - float(frozen_role.loc["finetune_test", "mae_mean"])
            ood_delta = ood_mae - float(frozen_role.loc["ood_eval", "mae_mean"])

            val_p90_delta = val_p90 - float(frozen_role.loc["finetune_val", "mae_p90"])
            test_p90_delta = test_p90 - float(frozen_role.loc["finetune_test", "mae_p90"])
            ood_p90_delta = ood_p90 - float(frozen_role.loc["ood_eval", "mae_p90"])

            in_distribution_gain = -0.5 * (val_delta + test_delta)
            p90_gain = -0.5 * (val_p90_delta + test_p90_delta)

            ood_penalty = max(0.0, ood_delta)
            high_hr_penalty = max(0.0, high_hr_mae - 10.6511)
            severe_under_penalty = max(0.0, high_hr_severe_under - 0.3026) * 10.0

            risk_adjusted_score = (
                in_distribution_gain
                + 0.25 * p90_gain
                - 0.50 * ood_penalty
                - 0.50 * high_hr_penalty
                - severe_under_penalty
            )

            rows.append(
                {
                    "candidate_name": candidate_name,
                    "alpha": float(alpha),
                    "val_delta_mean": val_delta,
                    "test_delta_mean": test_delta,
                    "ood_delta_mean": ood_delta,
                    "val_delta_p90": val_p90_delta,
                    "test_delta_p90": test_p90_delta,
                    "ood_delta_p90": ood_p90_delta,
                    "high_hr_mae": high_hr_mae,
                    "high_hr_signed_error": high_hr_signed_error,
                    "high_hr_severe_under_rate": high_hr_severe_under,
                    "in_distribution_gain": in_distribution_gain,
                    "p90_gain": p90_gain,
                    "risk_adjusted_score": risk_adjusted_score,
                }
            )

    decision = pd.DataFrame(rows)

    decision["blend_profile"] = np.select(
        [
            (decision["val_delta_mean"] < 0)
            & (decision["test_delta_mean"] < 0)
            & (decision["ood_delta_mean"] <= 0.25),
            (decision["val_delta_mean"] < 0)
            & (decision["test_delta_mean"] < 0)
            & (decision["ood_delta_mean"] <= 0.50),
            (decision["val_delta_mean"] < 0)
            & (decision["test_delta_mean"] < 0),
        ],
        [
            "strong_conservative_blend",
            "moderate_conservative_blend",
            "in_distribution_only_blend",
        ],
        default="not_useful",
    )

    return decision.sort_values(
        ["risk_adjusted_score", "in_distribution_gain"],
        ascending=[False, False],
    ).reset_index(drop=True)


def run_prediction_blending_analysis() -> dict:
    """
    Run prediction blending for full-transfer and balanced-transfer candidates.
    """
    assert_prediction_blending_prerequisites()

    full_predictions = get_blending_prediction_source(
        experiment_name="full_model_transfer",
        result_name="full_transfer_result",
        model_name="FULL_TRANSFER_MODEL",
    )

    balanced_predictions = get_blending_prediction_source(
        experiment_name="balanced_full_model_transfer",
        result_name="balanced_full_transfer_result",
        model_name="BALANCED_FULL_TRANSFER_MODEL",
    )

    full_blends = make_prediction_blend_table(
        candidate_predictions=full_predictions,
        candidate_name="full_model_transfer",
        alphas=BLEND_ALPHAS,
    )

    balanced_blends = make_prediction_blend_table(
        candidate_predictions=balanced_predictions,
        candidate_name="balanced_full_model_transfer",
        alphas=BLEND_ALPHAS,
    )

    blend_table = pd.concat([full_blends, balanced_blends], ignore_index=True)

    role_summary = summarize_blended_predictions_by_role(blend_table)
    high_hr_summary = summarize_blended_predictions_by_high_hr(blend_table)

    decision_table = build_prediction_blend_decision_table(
        role_summary=role_summary,
        high_hr_summary=high_hr_summary,
    )

    blend_table.to_csv(PREDICTION_BLEND_DIR / "prediction_blend_rows.csv", index=False)
    role_summary.to_csv(PREDICTION_BLEND_DIR / "prediction_blend_role_summary.csv", index=False)
    high_hr_summary.to_csv(PREDICTION_BLEND_DIR / "prediction_blend_high_hr_summary.csv", index=False)
    decision_table.to_csv(PREDICTION_BLEND_DIR / "prediction_blend_decision_table.csv", index=False)

    report = {
        "alphas": BLEND_ALPHAS,
        "candidate_names": sorted(decision_table["candidate_name"].unique().tolist()),
        "best_risk_adjusted_blend": {
            "candidate_name": str(decision_table.iloc[0]["candidate_name"]),
            "alpha": float(decision_table.iloc[0]["alpha"]),
            "blend_profile": str(decision_table.iloc[0]["blend_profile"]),
        },
        "artifact_dir": str(PREDICTION_BLEND_DIR),
    }

    return {
        "blend_table": blend_table,
        "role_summary": role_summary,
        "high_hr_summary": high_hr_summary,
        "decision_table": decision_table,
        "report": report,
    }


prediction_blending_result = run_prediction_blending_analysis()

NB11_PREDICTION_BLENDING = prediction_blending_result

print("NB11 prediction blending report:")
print(json.dumps(prediction_blending_result["report"], indent=2))

print("\nPrediction blend decision table:")
display(round_metric_table(prediction_blending_result["decision_table"]))

print("\nPrediction blend role summary:")
display(round_metric_table(prediction_blending_result["role_summary"]))

print("\nPrediction blend high-HR summary:")
display(round_metric_table(prediction_blending_result["high_hr_summary"]))

print("\nDONE: NB11 prediction blending analysis completed.")

NB11 prediction blending report:
{
  "alphas": [
    0.1,
    0.25,
    0.4,
    0.5,
    0.6,
    0.75,
    0.9
  ],
  "candidate_names": [
    "balanced_full_model_transfer",
    "full_model_transfer"
  ],
  "best_risk_adjusted_blend": {
    "candidate_name": "full_model_transfer",
    "alpha": 0.1,
    "blend_profile": "strong_conservative_blend"
  },
  "artifact_dir": "/kaggle/working/crvse_nb11_prediction_blending"
}

Prediction blend decision table:


,candidate_name,alpha,val_delta_mean,test_delta_mean,ood_delta_mean,val_delta_p90,test_delta_p90,ood_delta_p90,high_hr_mae,high_hr_signed_error,high_hr_severe_under_rate,in_distribution_gain,p90_gain,risk_adjusted_score,blend_profile
0,full_model_transfer,0.10,-0.0243,-0.0256,0.0552,-0.0449,-0.0577,0.1938,10.6751,-7.2661,0.3031,0.0250,0.0513,-0.0064,strong_conservative_blend
1,balanced_full_model_transfer,0.10,-0.0286,-0.0146,0.0290,-0.0265,0.0443,0.0713,10.6634,-7.2154,0.3035,0.0216,-0.0089,-0.0105,strong_conservative_blend
2,balanced_full_model_transfer,0.25,-0.0681,-0.0329,0.0826,-0.0344,-0.0614,0.1959,10.6916,-7.2862,0.3040,0.0505,0.0479,-0.0131,strong_conservative_blend
3,balanced_full_model_transfer,0.40,-0.1034,-0.0462,0.1576,-0.0657,-0.2058,0.2954,10.7353,-7.3570,0.3068,0.0748,0.1358,-0.0544,strong_conservative_blend
4,full_model_transfer,0.25,-0.0586,-0.0621,0.1758,-0.0757,-0.3417,0.3422,10.7275,-7.4131,0.3087,0.0604,0.2087,-0.0746,strong_conservative_blend
5,balanced_full_model_transfer,0.50,-0.1240,-0.0514,0.2136,-0.1252,-0.2198,0.6773,10.7710,-7.4042,0.3082,0.0877,0.1725,-0.0923,strong_conservative_blend
6,balanced_full_model_transfer,0.60,-0.1428,-0.0551,0.2711,-0.1102,-0.3102,1.0311,10.8103,-7.4514,0.3096,0.0990,0.2102,-0.1341,moderate_conservative_blend
7,full_model_transfer,0.40,-0.0897,-0.0954,0.3255,-0.0741,-0.5459,0.8688,10.7918,-7.5600,0.3106,0.0926,0.3100,-0.1429,moderate_conservative_blend
8,full_model_transfer,0.50,-0.1086,-0.1152,0.4344,-0.0342,-0.5420,0.7729,10.8389,-7.6580,0.3111,0.1119,0.2881,-0.2118,moderate_conservative_blend
9,balanced_full_model_transfer,0.75,-0.1656,-0.0578,0.3632,-0.2381,-0.3762,1.0393,10.8804,-7.5222,0.3134,0.1117,0.3071,-0.2159,moderate_conservative_blend



Prediction blend role summary:


,candidate_name,alpha,role,rows,mae_mean,mae_median,mae_p90,mae_p95,signed_error_mean,severe_underprediction_rate
0,balanced_full_model_transfer,0.10,finetune_test,2828,5.7550,3.6031,13.7426,19.0183,1.3979,0.0619
1,balanced_full_model_transfer,0.10,finetune_train,12503,5.7988,3.7135,13.6720,18.7286,1.5603,0.0565
2,balanced_full_model_transfer,0.10,finetune_val,2839,5.9989,3.8329,14.0436,19.2585,1.4567,0.0687
3,balanced_full_model_transfer,0.10,ood_eval,882,14.0645,7.6260,38.2211,49.9363,-8.0892,0.3095
4,balanced_full_model_transfer,0.25,finetune_test,2828,5.7367,3.5778,13.6370,18.8404,1.3687,0.0615
5,balanced_full_model_transfer,0.25,finetune_train,12503,5.7660,3.7046,13.5512,18.7358,1.5520,0.0565
6,balanced_full_model_transfer,0.25,finetune_val,2839,5.9594,3.8041,14.0358,19.0679,1.4331,0.0676
7,balanced_full_model_transfer,0.25,ood_eval,882,14.1181,7.6665,38.3457,49.8560,-8.2999,0.3118
8,balanced_full_model_transfer,0.40,finetune_test,2828,5.7234,3.5610,13.4926,19.0099,1.3395,0.0622
9,balanced_full_model_transfer,0.40,finetune_train,12503,5.7383,3.6766,13.4936,18.6411,1.5438,0.0567



Prediction blend high-HR summary:


,candidate_name,alpha,high_hr_flag,rows,mae_mean,mae_p90,signed_error_mean,severe_underprediction_rate
0,balanced_full_model_transfer,0.10,target_hr_ge_100,2125,10.6634,25.1621,-7.2154,0.3035
1,balanced_full_model_transfer,0.10,target_hr_lt_100,16927,5.6451,13.0941,2.1147,0.0416
2,balanced_full_model_transfer,0.25,target_hr_ge_100,2125,10.6916,25.2237,-7.2862,0.3040
3,balanced_full_model_transfer,0.25,target_hr_lt_100,16927,5.6104,13.0406,2.0977,0.0414
4,balanced_full_model_transfer,0.40,target_hr_ge_100,2125,10.7353,25.5092,-7.3570,0.3068
5,balanced_full_model_transfer,0.40,target_hr_lt_100,16927,5.5802,12.9894,2.0806,0.0414
6,balanced_full_model_transfer,0.50,target_hr_ge_100,2125,10.7710,25.6382,-7.4042,0.3082
7,balanced_full_model_transfer,0.50,target_hr_lt_100,16927,5.5627,12.9224,2.0693,0.0411
8,balanced_full_model_transfer,0.60,target_hr_ge_100,2125,10.8103,26.0864,-7.4514,0.3096
9,balanced_full_model_transfer,0.60,target_hr_lt_100,16927,5.5470,12.8969,2.0580,0.0410



DONE: NB11 prediction blending analysis completed.


## NB11 Checkpoint Interpolation Analysis

### What is being tested

This section interpolates model weights between the frozen source-FPS baseline checkpoint and transfer-learned checkpoints.

The interpolated checkpoint is:

```text
interpolated_weight = (1 - alpha) * frozen_weight + alpha * transfer_weight
```

### Why this matters

Prediction blending showed that conservative mixing can reduce out-of-domain harm, but prediction blending is not a single deployable model checkpoint.
Checkpoint interpolation tests whether a similar conservative adaptation can be represented as one PhysFormer model.


In [12]:
CHECKPOINT_INTERPOLATION_DIR = KAGGLE_WORKING / "crvse_nb11_checkpoint_interpolation"
CHECKPOINT_INTERPOLATION_DIR.mkdir(parents=True, exist_ok=True)

INTERPOLATION_ALPHAS = [0.10, 0.25, 0.40, 0.50, 0.60, 0.75, 0.90]


def assert_checkpoint_interpolation_prerequisites() -> None:
    """
    Check that frozen and transfer models are available.
    """
    required_names = [
        "FROZEN_MODEL",
        "FULL_TRANSFER_MODEL",
        "BALANCED_FULL_TRANSFER_MODEL",
        "FROZEN_MODEL_CONFIG",
        "DATASETS",
        "evaluate_candidate_on_all_roles",
        "compare_candidate_to_frozen_by_role",
        "compare_candidate_to_frozen_by_dataset_role",
        "round_metric_table",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run frozen, full-transfer, and balanced-transfer cells before interpolation. "
            f"Missing names: {missing}"
        )


def interpolate_state_dicts(
    frozen_state: dict[str, torch.Tensor],
    candidate_state: dict[str, torch.Tensor],
    alpha: float,
) -> dict[str, torch.Tensor]:
    """
    Interpolate two compatible model state dicts.

    Floating tensors are linearly interpolated. Non-floating tensors are copied
    from the candidate state only when alpha is 1.0; otherwise they are copied
    from frozen state.
    """
    if frozen_state.keys() != candidate_state.keys():
        raise ValueError("State dict keys do not match. Cannot interpolate checkpoints.")

    interpolated = {}

    for key in frozen_state:
        frozen_value = frozen_state[key].detach().cpu()
        candidate_value = candidate_state[key].detach().cpu()

        if frozen_value.shape != candidate_value.shape:
            raise ValueError(
                f"Shape mismatch for {key}: {tuple(frozen_value.shape)} vs {tuple(candidate_value.shape)}."
            )

        if torch.is_floating_point(frozen_value):
            interpolated[key] = (
                (1.0 - alpha) * frozen_value
                + alpha * candidate_value
            )
        else:
            interpolated[key] = candidate_value.clone() if alpha >= 1.0 else frozen_value.clone()

    return interpolated


def build_interpolated_model(
    candidate_model: nn.Module,
    alpha: float,
) -> CRVSEPhysFormer:
    """
    Build one interpolated CRVSE PhysFormer model.
    """
    frozen_state = FROZEN_MODEL.state_dict()
    candidate_state = candidate_model.state_dict()

    interpolated_state = interpolate_state_dicts(
        frozen_state=frozen_state,
        candidate_state=candidate_state,
        alpha=alpha,
    )

    model = CRVSEPhysFormer(**FROZEN_MODEL_CONFIG)
    model.load_state_dict(interpolated_state, strict=True)
    model.to(DEVICE)
    model.eval()

    for parameter in model.parameters():
        parameter.requires_grad = False

    return model


def summarize_interpolated_high_hr(predictions: pd.DataFrame) -> dict:
    """
    Summarize high-HR behavior for one interpolated model.
    """
    table = predictions.copy()
    table["target_hr"] = table["target_hr_from_tensor"].astype(float)
    table["pred_hr"] = table["notebook_model_hr"].astype(float)
    table["signed_error"] = table["pred_hr"] - table["target_hr"]
    table["abs_error"] = table["signed_error"].abs()

    high_hr = table[table["target_hr"] >= 100.0].copy()
    non_high_hr = table[table["target_hr"] < 100.0].copy()

    return {
        "high_hr_rows": int(len(high_hr)),
        "high_hr_mae": float(high_hr["abs_error"].mean()),
        "high_hr_p90": float(high_hr["abs_error"].quantile(0.90)),
        "high_hr_signed_error_mean": float(high_hr["signed_error"].mean()),
        "high_hr_severe_under_rate": float((high_hr["signed_error"] <= -10).mean()),
        "non_high_hr_mae": float(non_high_hr["abs_error"].mean()),
    }


def extract_frozen_high_hr_reference() -> dict:
    """
    Compute frozen high-HR reference metrics.
    """
    table = FROZEN_PREDICTIONS.copy()
    table["target_hr"] = table["target_hr_from_tensor"].astype(float)
    table["pred_hr"] = table["notebook_model_hr"].astype(float)
    table["signed_error"] = table["pred_hr"] - table["target_hr"]
    table["abs_error"] = table["signed_error"].abs()

    high_hr = table[table["target_hr"] >= 100.0].copy()

    return {
        "frozen_high_hr_mae": float(high_hr["abs_error"].mean()),
        "frozen_high_hr_p90": float(high_hr["abs_error"].quantile(0.90)),
        "frozen_high_hr_signed_error_mean": float(high_hr["signed_error"].mean()),
        "frozen_high_hr_severe_under_rate": float((high_hr["signed_error"] <= -10).mean()),
    }


def evaluate_interpolated_candidate_grid(
    candidate_name: str,
    candidate_model: nn.Module,
    alphas: list[float],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Evaluate interpolated checkpoints for one transfer candidate.
    """
    role_rows = []
    dataset_role_frames = []
    prediction_frames = []

    frozen_high_hr_reference = extract_frozen_high_hr_reference()

    for alpha in alphas:
        interpolated_model = build_interpolated_model(
            candidate_model=candidate_model,
            alpha=alpha,
        )

        predictions, role_summary, dataset_role_summary = evaluate_candidate_on_all_roles(
            model=interpolated_model,
            datasets=DATASETS,
        )

        role_comparison = compare_candidate_to_frozen_by_role(role_summary)
        high_hr_metrics = summarize_interpolated_high_hr(predictions)

        for _, row in role_comparison.iterrows():
            role_rows.append(
                {
                    "candidate_name": candidate_name,
                    "alpha": float(alpha),
                    **row.to_dict(),
                    **high_hr_metrics,
                    **frozen_high_hr_reference,
                }
            )

        dataset_role_summary = dataset_role_summary.copy()
        dataset_role_summary["candidate_name"] = candidate_name
        dataset_role_summary["alpha"] = float(alpha)
        dataset_role_frames.append(dataset_role_summary)

        predictions = predictions.copy()
        predictions["candidate_name"] = candidate_name
        predictions["alpha"] = float(alpha)
        prediction_frames.append(predictions)

    role_table = pd.DataFrame(role_rows)
    dataset_role_table = pd.concat(dataset_role_frames, ignore_index=True)
    prediction_table = pd.concat(prediction_frames, ignore_index=True)

    return role_table, dataset_role_table, prediction_table


def build_interpolation_decision_table(role_table: pd.DataFrame) -> pd.DataFrame:
    """
    Build one decision row per candidate/alpha from interpolated role metrics.
    """
    rows = []

    for (candidate_name, alpha), group in role_table.groupby(["candidate_name", "alpha"]):
        by_role = group.set_index("role")

        val_delta = float(by_role.loc["finetune_val", "delta_mae_mean"])
        test_delta = float(by_role.loc["finetune_test", "delta_mae_mean"])
        ood_delta = float(by_role.loc["ood_eval", "delta_mae_mean"])

        val_p90_delta = float(by_role.loc["finetune_val", "delta_mae_p90"])
        test_p90_delta = float(by_role.loc["finetune_test", "delta_mae_p90"])
        ood_p90_delta = float(by_role.loc["ood_eval", "delta_mae_p90"])

        high_hr_mae = float(group["high_hr_mae"].iloc[0])
        high_hr_p90 = float(group["high_hr_p90"].iloc[0])
        high_hr_signed_error = float(group["high_hr_signed_error_mean"].iloc[0])
        high_hr_severe_under = float(group["high_hr_severe_under_rate"].iloc[0])

        frozen_high_hr_mae = float(group["frozen_high_hr_mae"].iloc[0])
        frozen_high_hr_p90 = float(group["frozen_high_hr_p90"].iloc[0])
        frozen_high_hr_severe_under = float(group["frozen_high_hr_severe_under_rate"].iloc[0])

        in_distribution_gain = -0.5 * (val_delta + test_delta)
        p90_gain = -0.5 * (val_p90_delta + test_p90_delta)

        high_hr_delta_mae = high_hr_mae - frozen_high_hr_mae
        high_hr_delta_p90 = high_hr_p90 - frozen_high_hr_p90
        high_hr_delta_severe_under = high_hr_severe_under - frozen_high_hr_severe_under

        risk_adjusted_score = (
            in_distribution_gain
            + 0.25 * p90_gain
            - 0.50 * max(0.0, ood_delta)
            - 0.50 * max(0.0, high_hr_delta_mae)
            - 10.0 * max(0.0, high_hr_delta_severe_under)
        )

        rows.append(
            {
                "candidate_name": candidate_name,
                "alpha": float(alpha),
                "val_delta_mean": val_delta,
                "test_delta_mean": test_delta,
                "ood_delta_mean": ood_delta,
                "val_delta_p90": val_p90_delta,
                "test_delta_p90": test_p90_delta,
                "ood_delta_p90": ood_p90_delta,
                "high_hr_delta_mae": high_hr_delta_mae,
                "high_hr_delta_p90": high_hr_delta_p90,
                "high_hr_signed_error": high_hr_signed_error,
                "high_hr_delta_severe_under_rate": high_hr_delta_severe_under,
                "in_distribution_gain": in_distribution_gain,
                "p90_gain": p90_gain,
                "risk_adjusted_score": risk_adjusted_score,
            }
        )

    decision = pd.DataFrame(rows)

    decision["interpolation_profile"] = np.select(
        [
            (decision["val_delta_mean"] < 0)
            & (decision["test_delta_mean"] < 0)
            & (decision["ood_delta_mean"] <= 0.25),
            (decision["val_delta_mean"] < 0)
            & (decision["test_delta_mean"] < 0)
            & (decision["ood_delta_mean"] <= 0.50),
            (decision["val_delta_mean"] < 0)
            & (decision["test_delta_mean"] < 0),
        ],
        [
            "strong_conservative_interpolation",
            "moderate_conservative_interpolation",
            "in_distribution_only_interpolation",
        ],
        default="not_useful",
    )

    return decision.sort_values(
        ["risk_adjusted_score", "in_distribution_gain"],
        ascending=[False, False],
    ).reset_index(drop=True)


def run_checkpoint_interpolation_analysis() -> dict:
    """
    Run checkpoint interpolation for full and balanced transfer candidates.
    """
    assert_checkpoint_interpolation_prerequisites()

    full_role, full_dataset_role, full_predictions = evaluate_interpolated_candidate_grid(
        candidate_name="full_model_transfer",
        candidate_model=FULL_TRANSFER_MODEL,
        alphas=INTERPOLATION_ALPHAS,
    )

    balanced_role, balanced_dataset_role, balanced_predictions = evaluate_interpolated_candidate_grid(
        candidate_name="balanced_full_model_transfer",
        candidate_model=BALANCED_FULL_TRANSFER_MODEL,
        alphas=INTERPOLATION_ALPHAS,
    )

    role_table = pd.concat([full_role, balanced_role], ignore_index=True)
    dataset_role_table = pd.concat([full_dataset_role, balanced_dataset_role], ignore_index=True)
    prediction_table = pd.concat([full_predictions, balanced_predictions], ignore_index=True)

    decision_table = build_interpolation_decision_table(role_table)

    role_table.to_csv(CHECKPOINT_INTERPOLATION_DIR / "checkpoint_interpolation_role_table.csv", index=False)
    dataset_role_table.to_csv(CHECKPOINT_INTERPOLATION_DIR / "checkpoint_interpolation_dataset_role_table.csv", index=False)
    prediction_table.to_csv(CHECKPOINT_INTERPOLATION_DIR / "checkpoint_interpolation_prediction_rows.csv", index=False)
    decision_table.to_csv(CHECKPOINT_INTERPOLATION_DIR / "checkpoint_interpolation_decision_table.csv", index=False)

    report = {
        "alphas": INTERPOLATION_ALPHAS,
        "candidate_names": sorted(decision_table["candidate_name"].unique().tolist()),
        "best_risk_adjusted_interpolation": {
            "candidate_name": str(decision_table.iloc[0]["candidate_name"]),
            "alpha": float(decision_table.iloc[0]["alpha"]),
            "profile": str(decision_table.iloc[0]["interpolation_profile"]),
        },
        "artifact_dir": str(CHECKPOINT_INTERPOLATION_DIR),
    }

    return {
        "role_table": role_table,
        "dataset_role_table": dataset_role_table,
        "prediction_table": prediction_table,
        "decision_table": decision_table,
        "report": report,
    }


checkpoint_interpolation_result = run_checkpoint_interpolation_analysis()

NB11_CHECKPOINT_INTERPOLATION = checkpoint_interpolation_result

print("NB11 checkpoint interpolation report:")
print(json.dumps(checkpoint_interpolation_result["report"], indent=2))

print("\nCheckpoint interpolation decision table:")
display(round_metric_table(checkpoint_interpolation_result["decision_table"]))

print("\nCheckpoint interpolation role table:")
display(round_metric_table(checkpoint_interpolation_result["role_table"]))

print("\nDONE: NB11 checkpoint interpolation analysis completed.")

NB11 checkpoint interpolation report:
{
  "alphas": [
    0.1,
    0.25,
    0.4,
    0.5,
    0.6,
    0.75,
    0.9
  ],
  "candidate_names": [
    "balanced_full_model_transfer",
    "full_model_transfer"
  ],
  "best_risk_adjusted_interpolation": {
    "candidate_name": "balanced_full_model_transfer",
    "alpha": 0.25,
    "profile": "strong_conservative_interpolation"
  },
  "artifact_dir": "/kaggle/working/crvse_nb11_checkpoint_interpolation"
}

Checkpoint interpolation decision table:


,candidate_name,alpha,val_delta_mean,test_delta_mean,ood_delta_mean,val_delta_p90,test_delta_p90,ood_delta_p90,high_hr_delta_mae,high_hr_delta_p90,high_hr_signed_error,high_hr_delta_severe_under_rate,in_distribution_gain,p90_gain,risk_adjusted_score,interpolation_profile
0,balanced_full_model_transfer,0.25,-0.0719,-0.0339,0.0927,-0.0501,-0.0324,0.2552,0.0405,0.0432,-7.2656,0.0005,0.0529,0.0412,-0.0081,strong_conservative_interpolation
1,balanced_full_model_transfer,0.10,-0.0305,-0.0152,0.0343,-0.0125,0.0286,0.1025,0.0120,-0.0375,-7.2054,0.0009,0.0228,-0.0081,-0.0118,strong_conservative_interpolation
2,full_model_transfer,0.10,-0.0270,-0.0275,0.0771,-0.0476,-0.1610,0.1919,0.0373,0.1922,-7.2716,0.0009,0.0273,0.1043,-0.0133,strong_conservative_interpolation
3,balanced_full_model_transfer,0.40,-0.1076,-0.0477,0.1696,-0.0722,-0.3244,0.5725,0.0846,0.3352,-7.3304,0.0042,0.0776,0.1983,-0.0422,strong_conservative_interpolation
4,balanced_full_model_transfer,0.50,-0.1279,-0.0532,0.2262,-0.1625,-0.2982,1.0762,0.1201,0.4195,-7.3760,0.0038,0.0906,0.2304,-0.0626,strong_conservative_interpolation
5,full_model_transfer,0.25,-0.0637,-0.0659,0.2213,-0.0463,-0.4035,0.8844,0.1029,0.3257,-7.4245,0.0047,0.0648,0.2249,-0.0881,strong_conservative_interpolation
6,balanced_full_model_transfer,0.60,-0.1460,-0.0568,0.2831,-0.2395,-0.2810,1.1152,0.1597,0.8371,-7.4239,0.0047,0.1014,0.2602,-0.1020,moderate_conservative_interpolation
7,full_model_transfer,0.40,-0.0960,-0.0996,0.3800,-0.0588,-0.5266,0.9276,0.1721,0.7553,-7.5738,0.0085,0.0978,0.2927,-0.1898,moderate_conservative_interpolation
8,balanced_full_model_transfer,0.75,-0.1676,-0.0591,0.3728,-0.2892,-0.3662,1.0607,0.2298,1.0594,-7.5003,0.0099,0.1133,0.3277,-0.2049,moderate_conservative_interpolation
9,full_model_transfer,0.50,-0.1154,-0.1194,0.4890,-0.1018,-0.5673,0.9626,0.2191,0.8032,-7.6714,0.0071,0.1174,0.3346,-0.2236,moderate_conservative_interpolation



Checkpoint interpolation role table:


,candidate_name,alpha,role,rows,frozen_mae_mean,frozen_mae_median,frozen_mae_p90,frozen_mae_p95,frozen_mae_max,candidate_mae_mean,...,high_hr_rows,high_hr_mae,high_hr_p90,high_hr_signed_error_mean,high_hr_severe_under_rate,non_high_hr_mae,frozen_high_hr_mae,frozen_high_hr_p90,frozen_high_hr_signed_error_mean,frozen_high_hr_severe_under_rate
0,full_model_transfer,0.10,finetune_train,12503,5.8236,3.7437,13.7192,18.9426,54.9838,5.7862,...,2125,10.6884,25.3825,-7.2716,0.3035,5.6332,10.6511,25.1903,-7.1682,0.3026
1,full_model_transfer,0.10,finetune_val,2839,6.0275,3.8533,14.0701,19.2319,50.2965,6.0005,...,2125,10.6884,25.3825,-7.2716,0.3035,5.6332,10.6511,25.1903,-7.1682,0.3026
2,full_model_transfer,0.10,finetune_test,2828,5.7696,3.6274,13.6984,19.0461,72.2401,5.7421,...,2125,10.6884,25.3825,-7.2716,0.3035,5.6332,10.6511,25.1903,-7.1682,0.3026
3,full_model_transfer,0.10,ood_eval,882,14.0355,7.6838,38.1498,49.9899,89.9549,14.1126,...,2125,10.6884,25.3825,-7.2716,0.3035,5.6332,10.6511,25.1903,-7.1682,0.3026
4,full_model_transfer,0.25,finetune_train,12503,5.8236,3.7437,13.7192,18.9426,54.9838,5.7355,...,2125,10.7540,25.5160,-7.4245,0.3073,5.5825,10.6511,25.1903,-7.1682,0.3026
5,full_model_transfer,0.25,finetune_val,2839,6.0275,3.8533,14.0701,19.2319,50.2965,5.9638,...,2125,10.7540,25.5160,-7.4245,0.3073,5.5825,10.6511,25.1903,-7.1682,0.3026
6,full_model_transfer,0.25,finetune_test,2828,5.7696,3.6274,13.6984,19.0461,72.2401,5.7037,...,2125,10.7540,25.5160,-7.4245,0.3073,5.5825,10.6511,25.1903,-7.1682,0.3026
7,full_model_transfer,0.25,ood_eval,882,14.0355,7.6838,38.1498,49.9899,89.9549,14.2568,...,2125,10.7540,25.5160,-7.4245,0.3073,5.5825,10.6511,25.1903,-7.1682,0.3026
8,full_model_transfer,0.40,finetune_train,12503,5.8236,3.7437,13.7192,18.9426,54.9838,5.6911,...,2125,10.8232,25.9456,-7.5738,0.3111,5.5382,10.6511,25.1903,-7.1682,0.3026
9,full_model_transfer,0.40,finetune_val,2839,6.0275,3.8533,14.0701,19.2319,50.2965,5.9315,...,2125,10.8232,25.9456,-7.5738,0.3111,5.5382,10.6511,25.1903,-7.1682,0.3026



DONE: NB11 checkpoint interpolation analysis completed.


## NB11 BatchNorm-Frozen Full-Model Transfer

### What is being tested

This section repeats full-model transfer learning while freezing BatchNorm layers.

### Why this matters

The previous transfer experiments improved live-compatible validation and test performance, but worsened ECG-Fitness / high-HR behavior.

BatchNorm layers store running activation statistics. If transfer learning shifts those statistics toward the dominant training distribution, the model may become less compatible with exercise/high-HR windows.

This experiment tests:

```text
Can the model adapt its weights while preserving original BatchNorm behavior?

In [13]:
BN_FROZEN_TRANSFER_BATCH_SIZE = 256
BN_FROZEN_TRANSFER_MAX_EPOCHS = 25
BN_FROZEN_TRANSFER_PATIENCE = 6
BN_FROZEN_TRANSFER_MIN_DELTA = 1e-3
BN_FROZEN_TRANSFER_LR = 3e-5
BN_FROZEN_TRANSFER_WEIGHT_DECAY = 1e-4
BN_FROZEN_TRANSFER_HUBER_DELTA = 6.5
BN_FROZEN_TRANSFER_GRAD_CLIP = 5.0

BN_FROZEN_TRANSFER_DIR = KAGGLE_WORKING / "crvse_bn_frozen_full_model_transfer_sourcefps"
BN_FROZEN_TRANSFER_DIR.mkdir(parents=True, exist_ok=True)


def assert_bn_frozen_transfer_prerequisites() -> None:
    """
    Check that baseline and transfer helper functions are available.
    """
    required_names = [
        "DATASETS",
        "FROZEN_ROLE_SUMMARY",
        "FROZEN_DATASET_ROLE_SUMMARY",
        "ORIGINAL_CHECKPOINT_PATH",
        "build_full_transfer_model",
        "count_parameters_by_trainability",
        "make_eval_loader",
        "evaluate_hr_model",
        "evaluate_candidate_on_all_roles",
        "compare_candidate_to_frozen_by_role",
        "compare_candidate_to_frozen_by_dataset_role",
        "get_role_delta",
        "round_metric_table",
    ]

    missing = [name for name in required_names if name not in globals()]

    if missing:
        raise NameError(
            "Run the frozen baseline and full-transfer cells before this ablation. "
            f"Missing names: {missing}"
        )


def freeze_batchnorm_layers(model: nn.Module) -> list[str]:
    """
    Freeze BatchNorm modules and keep their stored running statistics fixed.

    Returns
    -------
    list[str]
        Names of BatchNorm modules that were frozen.
    """
    frozen_module_names = []

    for module_name, module in model.named_modules():
        if isinstance(module, nn.BatchNorm1d):
            module.eval()
            frozen_module_names.append(module_name)

            for parameter in module.parameters():
                parameter.requires_grad = False

    return frozen_module_names


def keep_batchnorm_layers_in_eval_mode(model: nn.Module) -> None:
    """
    Force BatchNorm modules back to eval mode after model.train() is called.
    """
    for module in model.modules():
        if isinstance(module, nn.BatchNorm1d):
            module.eval()


def build_bn_frozen_transfer_model(checkpoint_path: Path) -> tuple[CRVSEPhysFormer, list[str]]:
    """
    Build checkpoint-initialized PhysFormer with BatchNorm layers frozen.
    """
    model = build_full_transfer_model(checkpoint_path)
    frozen_bn_modules = freeze_batchnorm_layers(model)
    return model, frozen_bn_modules


def make_bn_frozen_training_loader(dataset: MaterializedTensorDataset) -> DataLoader:
    """
    Create the shuffled training DataLoader for BatchNorm-frozen transfer.
    """
    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=BN_FROZEN_TRANSFER_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        generator=generator,
    )


def train_one_bn_frozen_transfer_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
) -> dict:
    """
    Train one epoch while keeping BatchNorm layers frozen.
    """
    model.train()
    keep_batchnorm_layers_in_eval_mode(model)

    total_loss = 0.0
    total_mae = 0.0
    total_rows = 0
    grad_norms = []

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        pred = model(batch_x)
        loss = loss_fn(pred, batch_y)
        mae = torch.mean(torch.abs(pred - batch_y))

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            [parameter for parameter in model.parameters() if parameter.requires_grad],
            max_norm=BN_FROZEN_TRANSFER_GRAD_CLIP,
        )

        optimizer.step()

        batch_size = int(batch_x.shape[0])
        total_loss += float(loss.item()) * batch_size
        total_mae += float(mae.item()) * batch_size
        total_rows += batch_size
        grad_norms.append(float(grad_norm.item()))

    return {
        "train_loss": total_loss / total_rows,
        "train_mae": total_mae / total_rows,
        "grad_norm_mean": float(np.mean(grad_norms)),
        "grad_norm_max": float(np.max(grad_norms)),
    }


def summarize_bn_layer_status(model: nn.Module) -> pd.DataFrame:
    """
    Summarize BatchNorm module training mode and parameter trainability.
    """
    rows = []

    for module_name, module in model.named_modules():
        if isinstance(module, nn.BatchNorm1d):
            trainable_parameters = sum(
                parameter.numel()
                for parameter in module.parameters()
                if parameter.requires_grad
            )

            total_parameters = sum(
                parameter.numel()
                for parameter in module.parameters()
            )

            rows.append(
                {
                    "module": module_name,
                    "training_mode": bool(module.training),
                    "trainable_parameters": int(trainable_parameters),
                    "total_parameters": int(total_parameters),
                    "tracks_running_stats": bool(module.track_running_stats),
                }
            )

    return pd.DataFrame(rows)


def run_bn_frozen_full_model_transfer_experiment() -> dict:
    """
    Run full-model transfer while freezing BatchNorm layers and statistics.
    """
    assert_bn_frozen_transfer_prerequisites()

    model, frozen_bn_modules = build_bn_frozen_transfer_model(ORIGINAL_CHECKPOINT_PATH)

    parameter_summary = count_parameters_by_trainability(model)
    bn_status = summarize_bn_layer_status(model)

    train_loader = make_bn_frozen_training_loader(DATASETS["finetune_train"])
    val_loader = make_eval_loader(DATASETS["finetune_val"])

    trainable_parameters = [
        parameter for parameter in model.parameters() if parameter.requires_grad
    ]

    optimizer = AdamW(
        trainable_parameters,
        lr=BN_FROZEN_TRANSFER_LR,
        weight_decay=BN_FROZEN_TRANSFER_WEIGHT_DECAY,
    )

    loss_fn = nn.HuberLoss(delta=BN_FROZEN_TRANSFER_HUBER_DELTA)

    frozen_val_mae = float(
        FROZEN_ROLE_SUMMARY.loc[
            FROZEN_ROLE_SUMMARY["role"] == "finetune_val",
            "mae_mean",
        ].iloc[0]
    )

    best_val_mae = frozen_val_mae
    best_epoch = 0
    best_state_dict = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    history_rows = []

    print(f"Frozen validation MAE baseline: {frozen_val_mae:.4f} BPM")
    print(f"Frozen BatchNorm modules: {frozen_bn_modules}")

    for epoch in range(1, BN_FROZEN_TRANSFER_MAX_EPOCHS + 1):
        train_metrics = train_one_bn_frozen_transfer_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
        )

        keep_batchnorm_layers_in_eval_mode(model)

        _, val_metrics = evaluate_hr_model(
            model=model,
            dataset=DATASETS["finetune_val"],
            loader=val_loader,
        )

        improved = val_metrics["mae_mean"] < (
            best_val_mae - BN_FROZEN_TRANSFER_MIN_DELTA
        )

        if improved:
            best_val_mae = float(val_metrics["mae_mean"])
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history_rows.append(
            {
                "epoch": epoch,
                **train_metrics,
                "val_mae": float(val_metrics["mae_mean"]),
                "val_median": float(val_metrics["mae_median"]),
                "val_p90": float(val_metrics["mae_p90"]),
                "improved": bool(improved),
                "best_epoch_so_far": int(best_epoch),
                "best_val_mae_so_far": float(best_val_mae),
            }
        )

        print(
            f"Epoch {epoch:02d} | "
            f"train MAE {train_metrics['train_mae']:.4f} | "
            f"val MAE {val_metrics['mae_mean']:.4f} | "
            f"best {best_val_mae:.4f} @ epoch {best_epoch}"
        )

        if epochs_without_improvement >= BN_FROZEN_TRANSFER_PATIENCE:
            print(f"Early stopping after {epoch} epochs.")
            break

    model.load_state_dict(best_state_dict)
    keep_batchnorm_layers_in_eval_mode(model)

    candidate_predictions, candidate_role_summary, candidate_dataset_role_summary = (
        evaluate_candidate_on_all_roles(model=model, datasets=DATASETS)
    )

    role_comparison = compare_candidate_to_frozen_by_role(candidate_role_summary)
    dataset_role_comparison = compare_candidate_to_frozen_by_dataset_role(
        candidate_dataset_role_summary
    )

    history = pd.DataFrame(history_rows)

    val_delta = get_role_delta(role_comparison, "finetune_val", "delta_mae_mean")
    test_delta = get_role_delta(role_comparison, "finetune_test", "delta_mae_mean")
    ood_delta = get_role_delta(role_comparison, "ood_eval", "delta_mae_mean")

    decision = (
        "candidate_for_analysis"
        if val_delta <= -0.20 and test_delta <= -0.10 and ood_delta <= 0.50
        else "research_result_only"
    )

    history_path = BN_FROZEN_TRANSFER_DIR / "bn_frozen_full_model_transfer_history.csv"
    comparison_path = BN_FROZEN_TRANSFER_DIR / "bn_frozen_full_model_transfer_role_comparison.csv"
    dataset_role_path = BN_FROZEN_TRANSFER_DIR / "bn_frozen_full_model_transfer_dataset_role_comparison.csv"
    checkpoint_path = BN_FROZEN_TRANSFER_DIR / "bn_frozen_full_model_transfer_best.pt"
    bn_status_path = BN_FROZEN_TRANSFER_DIR / "bn_frozen_layer_status.csv"

    history.to_csv(history_path, index=False)
    role_comparison.to_csv(comparison_path, index=False)
    dataset_role_comparison.to_csv(dataset_role_path, index=False)
    bn_status.to_csv(bn_status_path, index=False)

    torch.save(
        {
            "experiment": "NB_P2_11_bn_frozen_full_model_transfer_sourcefps",
            "model_state": model.state_dict(),
            "model_config": FROZEN_MODEL_CONFIG,
            "frozen_bn_modules": frozen_bn_modules,
            "best_epoch": best_epoch,
            "frozen_val_mae": float(frozen_val_mae),
            "best_val_mae": float(best_val_mae),
            "role_comparison": role_comparison.to_dict("records"),
            "dataset_role_comparison": dataset_role_comparison.to_dict("records"),
        },
        checkpoint_path,
    )

    result_summary = {
        "best_epoch": int(best_epoch),
        "frozen_val_mae": float(frozen_val_mae),
        "best_val_mae": float(best_val_mae),
        "val_delta_vs_frozen": float(val_delta),
        "test_delta_vs_frozen": float(test_delta),
        "ood_delta_vs_frozen": float(ood_delta),
        "decision": decision,
        "frozen_bn_modules": frozen_bn_modules,
        "history_path": str(history_path),
        "comparison_path": str(comparison_path),
        "dataset_role_path": str(dataset_role_path),
        "bn_status_path": str(bn_status_path),
        "checkpoint_path": str(checkpoint_path),
    }

    return {
        "model": model,
        "parameter_summary": parameter_summary,
        "bn_status": bn_status,
        "history": history,
        "candidate_predictions": candidate_predictions,
        "candidate_role_summary": candidate_role_summary,
        "candidate_dataset_role_summary": candidate_dataset_role_summary,
        "role_comparison": role_comparison,
        "dataset_role_comparison": dataset_role_comparison,
        "result_summary": result_summary,
    }


bn_frozen_transfer_result = run_bn_frozen_full_model_transfer_experiment()

BN_FROZEN_TRANSFER_MODEL = bn_frozen_transfer_result["model"]
BN_FROZEN_TRANSFER_HISTORY = bn_frozen_transfer_result["history"]
BN_FROZEN_TRANSFER_ROLE_COMPARISON = bn_frozen_transfer_result["role_comparison"]
BN_FROZEN_TRANSFER_DATASET_ROLE_COMPARISON = bn_frozen_transfer_result["dataset_role_comparison"]
BN_FROZEN_TRANSFER_RESULT_SUMMARY = bn_frozen_transfer_result["result_summary"]

print("\nBatchNorm-frozen transfer parameter summary:")
display(bn_frozen_transfer_result["parameter_summary"])

print("\nBatchNorm layer status:")
display(bn_frozen_transfer_result["bn_status"])

print("\nBatchNorm-frozen transfer history:")
display(round_metric_table(BN_FROZEN_TRANSFER_HISTORY))

print("\nBatchNorm-frozen transfer vs frozen role comparison:")
display(round_metric_table(BN_FROZEN_TRANSFER_ROLE_COMPARISON))

print("\nBatchNorm-frozen transfer vs frozen dataset-role comparison:")
display(round_metric_table(BN_FROZEN_TRANSFER_DATASET_ROLE_COMPARISON))

print("\nBatchNorm-frozen transfer result summary:")
print(json.dumps(BN_FROZEN_TRANSFER_RESULT_SUMMARY, indent=2))

print("\nDONE: BatchNorm-frozen full-model transfer experiment completed.")

Frozen validation MAE baseline: 6.0275 BPM
Frozen BatchNorm modules: ['encoder.1', 'encoder.5', 'encoder.9']
Epoch 01 | train MAE 8.5493 | val MAE 5.9277 | best 5.9277 @ epoch 1
Epoch 02 | train MAE 8.5109 | val MAE 5.9213 | best 5.9213 @ epoch 2
Epoch 03 | train MAE 8.4718 | val MAE 5.9335 | best 5.9213 @ epoch 2
Epoch 04 | train MAE 8.5757 | val MAE 5.9069 | best 5.9069 @ epoch 4
Epoch 05 | train MAE 8.5641 | val MAE 5.8785 | best 5.8785 @ epoch 5
Epoch 06 | train MAE 8.4461 | val MAE 5.9012 | best 5.8785 @ epoch 5
Epoch 07 | train MAE 8.4833 | val MAE 5.9051 | best 5.8785 @ epoch 5
Epoch 08 | train MAE 8.5123 | val MAE 5.9358 | best 5.8785 @ epoch 5
Epoch 09 | train MAE 8.5447 | val MAE 5.9028 | best 5.8785 @ epoch 5
Epoch 10 | train MAE 8.4924 | val MAE 5.8794 | best 5.8785 @ epoch 5
Epoch 11 | train MAE 8.4142 | val MAE 5.9378 | best 5.8785 @ epoch 5
Early stopping after 11 epochs.

BatchNorm-frozen transfer parameter summary:


,module,trainable,parameters
0,encoder,False,80
1,encoder,True,1616
2,freq_proj,True,47680
3,head,True,2625
4,transformer_layers,True,270144



BatchNorm layer status:


,module,training_mode,trainable_parameters,total_parameters,tracks_running_stats
0,encoder.1,False,0,16,True
1,encoder.5,False,0,32,True
2,encoder.9,False,0,32,True



BatchNorm-frozen transfer history:


...


BatchNorm-frozen transfer vs frozen role comparison:


,role,rows,frozen_mae_mean,frozen_mae_median,frozen_mae_p90,frozen_mae_p95,frozen_mae_max,candidate_mae_mean,candidate_mae_median,candidate_mae_p90,candidate_mae_p95,candidate_mae_max,delta_mae_mean,delta_mae_median,delta_mae_p90,delta_mae_p95,delta_mae_max
0,finetune_train,12503,5.8236,3.7437,13.7192,18.9426,54.9838,5.5678,3.5714,13.0628,18.1209,60.8453,-0.2558,-0.1723,-0.6564,-0.8217,5.8615
1,finetune_val,2839,6.0275,3.8533,14.0701,19.2319,50.2965,5.8785,3.7573,14.0832,18.6085,48.0507,-0.1490,-0.0960,0.0131,-0.6234,-2.2457
2,finetune_test,2828,5.7696,3.6274,13.6984,19.0461,72.2401,5.6222,3.4309,13.0477,18.6518,71.4777,-0.1475,-0.1965,-0.6507,-0.3942,-0.7624
3,ood_eval,882,14.0355,7.6838,38.1498,49.9899,89.9549,15.1237,8.2547,41.0126,52.6467,89.5298,1.0882,0.5709,2.8628,2.6568,-0.4251



BatchNorm-frozen transfer vs frozen dataset-role comparison:


,dataset,window_role,rows,frozen_mae_mean,frozen_mae_median,frozen_mae_p90,target_hr_mean,candidate_mae_mean,candidate_mae_median,candidate_mae_p90,delta_mae_mean,delta_mae_median,delta_mae_p90
0,mcd_rppg,finetune_test,2094,4.8847,3.1502,10.9700,79.853302,4.7458,2.9579,10.7569,-0.1390,-0.1923,-0.2131
1,ubfc_phys,finetune_test,658,8.8584,5.6735,20.2862,77.420601,8.6755,5.6171,19.7777,-0.1829,-0.0564,-0.5085
2,ubfc_rppg,finetune_test,76,3.4085,3.1453,6.0835,98.269302,3.3340,3.0181,6.0540,-0.0745,-0.1272,-0.0295
3,mcd_rppg,finetune_train,7446,4.6876,3.1639,10.3722,79.064796,4.3770,2.9751,9.6502,-0.3107,-0.1888,-0.7219
4,ubfc_phys,finetune_train,4595,7.7819,5.4248,18.5928,80.305496,7.5966,5.3355,18.2028,-0.1854,-0.0893,-0.3901
5,ubfc_rppg,finetune_train,462,4.6548,3.1447,9.9189,98.449203,4.5836,3.3066,10.0556,-0.0712,0.1620,0.1366
6,mcd_rppg,finetune_val,1743,5.1325,3.3746,11.5368,79.108597,4.9172,3.2424,11.2134,-0.2153,-0.1322,-0.3235
7,ubfc_phys,finetune_val,986,7.7629,4.8461,18.2725,80.002403,7.7039,5.0407,18.1200,-0.0590,0.1946,-0.1525
8,ubfc_rppg,finetune_val,110,4.6533,2.4438,12.3774,98.277199,4.7486,2.5612,12.8361,0.0953,0.1174,0.4588
9,ecg_fitness,ood_eval,882,14.0355,7.6838,38.1498,109.204803,15.1237,8.2547,41.0126,1.0882,0.5709,2.8628



BatchNorm-frozen transfer result summary:
{
  "best_epoch": 5,
  "frozen_val_mae": 6.027502059936523,
  "best_val_mae": 5.878497123718262,
  "val_delta_vs_frozen": -0.14900493621826172,
  "test_delta_vs_frozen": -0.14745759963989258,
  "ood_delta_vs_frozen": 1.0881567001342773,
  "decision": "research_result_only",
  "frozen_bn_modules": [
    "encoder.1",
    "encoder.5",
    "encoder.9"
  ],
  "history_path": "/kaggle/working/crvse_bn_frozen_full_model_transfer_sourcefps/bn_frozen_full_model_transfer_history.csv",
  "comparison_path": "/kaggle/working/crvse_bn_frozen_full_model_transfer_sourcefps/bn_frozen_full_model_transfer_role_comparison.csv",
  "dataset_role_path": "/kaggle/working/crvse_bn_frozen_full_model_transfer_sourcefps/bn_frozen_full_model_transfer_dataset_role_comparison.csv",
  "bn_status_path": "/kaggle/working/crvse_bn_frozen_full_model_transfer_sourcefps/bn_frozen_layer_status.csv",
  "checkpoint_path": "/kaggle/working/crvse_bn_frozen_full_model_transfer_sourcefps

In [14]:
NB11_SUMMARY_DIR = KAGGLE_WORKING / "crvse_nb11_summary"
NB11_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)


def get_optional_summary(name: str) -> dict | None:
    """
    Return a result summary dictionary if it exists.
    """
    value = globals().get(name)

    if isinstance(value, dict):
        return value

    return None


def get_optional_table(name: str) -> pd.DataFrame | None:
    """
    Return a DataFrame if it exists.
    """
    value = globals().get(name)

    if isinstance(value, pd.DataFrame):
        return value.copy()

    return None


def build_nb11_final_scoreboard() -> pd.DataFrame:
    """
    Build a compact final scoreboard across NB11 experiments.
    """
    rows = [
        {
            "experiment": "frozen_sourcefps_baseline",
            "type": "baseline",
            "best_epoch": 0,
            "val_delta_mean": 0.0,
            "test_delta_mean": 0.0,
            "ood_delta_mean": 0.0,
            "val_delta_p90": 0.0,
            "test_delta_p90": 0.0,
            "ood_delta_p90": 0.0,
            "interpretation": "current_app_checkpoint_reference",
        }
    ]

    experiment_specs = [
        (
            "full_model_transfer",
            "transfer",
            get_optional_summary("FULL_TRANSFER_RESULT_SUMMARY"),
            get_optional_table("FULL_TRANSFER_ROLE_COMPARISON"),
        ),
        (
            "balanced_full_model_transfer",
            "transfer",
            get_optional_summary("BALANCED_FULL_TRANSFER_RESULT_SUMMARY"),
            get_optional_table("BALANCED_FULL_TRANSFER_ROLE_COMPARISON"),
        ),
        (
            "scratch_physformer",
            "scratch",
            get_optional_summary("SCRATCH_RESULT_SUMMARY"),
            get_optional_table("SCRATCH_ROLE_COMPARISON"),
        ),
        (
            "bn_frozen_full_model_transfer",
            "transfer_ablation",
            get_optional_summary("BN_FROZEN_TRANSFER_RESULT_SUMMARY"),
            get_optional_table("BN_FROZEN_TRANSFER_ROLE_COMPARISON"),
        ),
    ]

    for experiment_name, experiment_type, summary, comparison in experiment_specs:
        if summary is None or comparison is None:
            continue

        rows.append(
            {
                "experiment": experiment_name,
                "type": experiment_type,
                "best_epoch": int(summary["best_epoch"]),
                "val_delta_mean": float(summary["val_delta_vs_frozen"]),
                "test_delta_mean": float(summary["test_delta_vs_frozen"]),
                "ood_delta_mean": float(summary["ood_delta_vs_frozen"]),
                "val_delta_p90": get_role_delta(comparison, "finetune_val", "delta_mae_p90"),
                "test_delta_p90": get_role_delta(comparison, "finetune_test", "delta_mae_p90"),
                "ood_delta_p90": get_role_delta(comparison, "ood_eval", "delta_mae_p90"),
                "interpretation": str(summary.get("decision", "research_result_only")),
            }
        )

    scoreboard = pd.DataFrame(rows)

    scoreboard["in_distribution_mean_gain"] = -(
        scoreboard["val_delta_mean"] + scoreboard["test_delta_mean"]
    ) / 2.0

    scoreboard["ood_penalty"] = scoreboard["ood_delta_mean"].clip(lower=0.0)

    scoreboard["adoption_candidate"] = (
        (scoreboard["val_delta_mean"] <= -0.20)
        & (scoreboard["test_delta_mean"] <= -0.10)
        & (scoreboard["ood_delta_mean"] <= 0.50)
    )

    return scoreboard.sort_values(
        ["adoption_candidate", "in_distribution_mean_gain"],
        ascending=[False, False],
    ).reset_index(drop=True)


def summarize_best_prediction_blend() -> dict | None:
    """
    Summarize the best prediction blending result if available.
    """
    result = globals().get("NB11_PREDICTION_BLENDING")

    if not isinstance(result, dict):
        return None

    decision_table = result.get("decision_table")

    if not isinstance(decision_table, pd.DataFrame) or len(decision_table) == 0:
        return None

    best = decision_table.iloc[0]

    return {
        "candidate_name": str(best["candidate_name"]),
        "alpha": float(best["alpha"]),
        "blend_profile": str(best["blend_profile"]),
        "val_delta_mean": float(best["val_delta_mean"]),
        "test_delta_mean": float(best["test_delta_mean"]),
        "ood_delta_mean": float(best["ood_delta_mean"]),
        "risk_adjusted_score": float(best["risk_adjusted_score"]),
    }


def summarize_best_checkpoint_interpolation() -> dict | None:
    """
    Summarize the best checkpoint interpolation result if available.
    """
    result = globals().get("NB11_CHECKPOINT_INTERPOLATION")

    if not isinstance(result, dict):
        return None

    decision_table = result.get("decision_table")

    if not isinstance(decision_table, pd.DataFrame) or len(decision_table) == 0:
        return None

    best = decision_table.iloc[0]

    return {
        "candidate_name": str(best["candidate_name"]),
        "alpha": float(best["alpha"]),
        "interpolation_profile": str(best["interpolation_profile"]),
        "val_delta_mean": float(best["val_delta_mean"]),
        "test_delta_mean": float(best["test_delta_mean"]),
        "ood_delta_mean": float(best["ood_delta_mean"]),
        "high_hr_delta_mae": float(best["high_hr_delta_mae"]),
        "risk_adjusted_score": float(best["risk_adjusted_score"]),
    }


def build_nb11_final_conclusion(scoreboard: pd.DataFrame) -> dict:
    """
    Build the final NB11 scientific conclusion.
    """
    best_in_distribution = scoreboard.loc[
        scoreboard["experiment"] != "frozen_sourcefps_baseline"
    ].sort_values(
        "in_distribution_mean_gain",
        ascending=False,
    ).iloc[0]

    best_checkpoint_interpolation = summarize_best_checkpoint_interpolation()
    best_prediction_blend = summarize_best_prediction_blend()

    conclusion = {
        "notebook": "NB_P2_11_Live-Compatible_PhysFormer_Training",
        "adopt_checkpoint_now": False,
        "current_app_checkpoint_remains": "frozen_sourcefps_baseline",
        "best_direct_in_distribution_experiment": str(best_in_distribution["experiment"]),
        "best_direct_in_distribution_gain_bpm": float(best_in_distribution["in_distribution_mean_gain"]),
        "best_direct_ood_penalty_bpm": float(best_in_distribution["ood_penalty"]),
        "best_prediction_blend": best_prediction_blend,
        "best_checkpoint_interpolation": best_checkpoint_interpolation,
        "main_findings": [
            "Full-model transfer learning can improve live-compatible validation and held-out test performance.",
            "Balanced full-model transfer reduces OOD/high-HR penalty compared with natural full transfer, but still does not eliminate it.",
            "Scratch PhysFormer training from random initialization is not competitive in this notebook.",
            "Prediction blending and checkpoint interpolation show a conservative trade-off curve, but practical gains are too small for app adoption.",
            "BatchNorm-frozen transfer does not solve ECG-Fitness/high-HR degradation, suggesting the issue is broader domain shift rather than only BatchNorm statistic drift.",
        ],
        "scientific_interpretation": (
            "NB11 supports the idea that the existing checkpoint contains useful learned "
            "rPPG structure and can be adapted toward live-compatible source-FPS tensors. "
            "However, adaptation improves app-like in-distribution windows at the cost of "
            "worse high-HR/exercise behavior, especially ECG-Fitness. No NB11 checkpoint "
            "should replace the current app checkpoint."
        ),
        "recommended_next_notebook": (
            "NB12 should run a proper Optuna search on live-compatible tensors, with explicit "
            "branches for ECG-Fitness excluded as OOD versus ECG-Fitness included with a "
            "subject-level split for exercise/high-HR generalization."
        ),
    }

    return conclusion


nb11_final_scoreboard = build_nb11_final_scoreboard()
nb11_final_conclusion = build_nb11_final_conclusion(nb11_final_scoreboard)

scoreboard_path = NB11_SUMMARY_DIR / "nb11_final_scoreboard.csv"
conclusion_json_path = NB11_SUMMARY_DIR / "nb11_final_conclusion.json"
conclusion_markdown_path = NB11_SUMMARY_DIR / "nb11_final_conclusion.md"

nb11_final_scoreboard.to_csv(scoreboard_path, index=False)

with conclusion_json_path.open("w", encoding="utf-8") as file:
    json.dump(nb11_final_conclusion, file, indent=2)

markdown_lines = [
    "# NB_P2_11 Live-Compatible PhysFormer Training Conclusion",
    "",
    "## Adoption Decision",
    "",
    f"Adopt checkpoint now: **{nb11_final_conclusion['adopt_checkpoint_now']}**",
    "",
    f"Current app checkpoint remains: `{nb11_final_conclusion['current_app_checkpoint_remains']}`",
    "",
    "## Best Direct Experiment",
    "",
    f"`{nb11_final_conclusion['best_direct_in_distribution_experiment']}`",
    "",
    f"In-distribution mean gain: `{nb11_final_conclusion['best_direct_in_distribution_gain_bpm']:.4f}` BPM",
    "",
    f"OOD penalty: `{nb11_final_conclusion['best_direct_ood_penalty_bpm']:.4f}` BPM",
    "",
    "## Main Findings",
    "",
]

for finding in nb11_final_conclusion["main_findings"]:
    markdown_lines.append(f"- {finding}")

markdown_lines.extend(
    [
        "",
        "## Scientific Interpretation",
        "",
        nb11_final_conclusion["scientific_interpretation"],
        "",
        "## Recommended Next Notebook",
        "",
        nb11_final_conclusion["recommended_next_notebook"],
        "",
    ]
)

conclusion_markdown_path.write_text("\n".join(markdown_lines), encoding="utf-8")

print("NB11 final scoreboard:")
display(round_metric_table(nb11_final_scoreboard))

print("\nNB11 final conclusion:")
print(json.dumps(nb11_final_conclusion, indent=2))

print("\nSaved NB11 conclusion artifacts:")
display(
    pd.DataFrame(
        [
            {"artifact": "scoreboard_csv", "path": str(scoreboard_path)},
            {"artifact": "conclusion_json", "path": str(conclusion_json_path)},
            {"artifact": "conclusion_markdown", "path": str(conclusion_markdown_path)},
        ]
    )
)

print("\nDONE: NB11 final conclusion artifacts saved.")

NB11 final scoreboard:


,experiment,type,best_epoch,val_delta_mean,test_delta_mean,ood_delta_mean,val_delta_p90,test_delta_p90,ood_delta_p90,interpretation,in_distribution_mean_gain,ood_penalty,adoption_candidate
0,full_model_transfer,transfer,9,-0.1793,-0.1829,1.1031,-0.2900,-0.6465,2.0705,research_result_only,0.1811,1.1031,False
1,bn_frozen_full_model_transfer,transfer_ablation,5,-0.1490,-0.1475,1.0882,0.0131,-0.6507,2.8628,research_result_only,0.1482,1.0882,False
2,balanced_full_model_transfer,transfer,5,-0.1916,-0.0509,0.5344,-0.3699,-0.2118,1.2321,research_result_only,0.1212,0.5344,False
3,frozen_sourcefps_baseline,baseline,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,current_app_checkpoint_reference,-0.0000,0.0000,False
4,scratch_physformer,scratch,35,0.6489,0.3383,14.9040,1.6083,-0.0625,20.5738,research_result_only,-0.4936,14.9040,False



NB11 final conclusion:
{
  "notebook": "NB_P2_11_Live-Compatible_PhysFormer_Training",
  "adopt_checkpoint_now": false,
  "current_app_checkpoint_remains": "frozen_sourcefps_baseline",
  "best_direct_in_distribution_experiment": "full_model_transfer",
  "best_direct_in_distribution_gain_bpm": 0.18106675148010254,
  "best_direct_ood_penalty_bpm": 1.1030645370483398,
  "best_prediction_blend": {
    "candidate_name": "full_model_transfer",
    "alpha": 0.1,
    "blend_profile": "strong_conservative_blend",
    "val_delta_mean": -0.02434272322699993,
    "test_delta_mean": -0.025562381474603768,
    "ood_delta_mean": 0.05521907114387936,
    "risk_adjusted_score": -0.006436562193002371
  },
  "best_checkpoint_interpolation": {
    "candidate_name": "balanced_full_model_transfer",
    "alpha": 0.25,
    "interpolation_profile": "strong_conservative_interpolation",
    "val_delta_mean": -0.07187032699584961,
    "test_delta_mean": -0.03389310836791992,
    "ood_delta_mean": 0.0927410125732

,artifact,path
0,scoreboard_csv,/kaggle/working/crvse_nb11_summary/nb11_final_...
1,conclusion_json,/kaggle/working/crvse_nb11_summary/nb11_final_...
2,conclusion_markdown,/kaggle/working/crvse_nb11_summary/nb11_final_...



DONE: NB11 final conclusion artifacts saved.
